In [8]:

import pandas as pd
import re

ecg = pd.read_excel("2024_Ecg1.xlsx")
echo = pd.read_excel("2024echo_reports.xlsx")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")

ecg = ecg[ecg["name_key"] != ""]
echo = echo[echo["name_key"] != ""]

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "name_key"]],
    echo[["EF_Teich", "name_key"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "EF_Teich": "ef_teich_%"
})

result = result[["name", "age", "sex", "file_name", "ef_teich_%"]]

result.to_excel("matched_ecg_echo.xlsx", index=False)

print("Matched patients:", len(result))
print("Saved as matched_ecg_echo.xlsx")

display(result)


PermissionError: [Errno 13] Permission denied: 'matched_ecg_echo.xlsx'

In [ ]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""

    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    name = " ".join(name.split())

    name = re.sub(r"^SALIVA\s+", "", name)

    return name

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"

    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()

    match = re.fullmatch(r"\d+(?:\.0+)?", age)

    if match:
        return int(float(age))

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        echo_name = h["name_key"]
        echo_compact = h["compact_key"]

        score = max(
            fuzz.ratio(ecg_name, echo_name),
            fuzz.ratio(ecg_compact, echo_compact),
            fuzz.token_sort_ratio(ecg_name, echo_name),
            fuzz.partial_ratio(ecg_name, echo_name),
            fuzz.partial_ratio(ecg_compact, echo_compact)
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "age": e[ecg_age_col],
                "gender": e[ecg_sex_col],
                "file_name": e[file_col],
                "EF (Teich) %": h[ef_col],
                "gender_key": ecg_sex,
                "match_score": round(score, 2)
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG gender": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = row["gender_key"]

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"

    if ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "age",
            "gender",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched ECG-Echo records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)
```

In [9]:

import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("2024_Ecg1.xlsx")
echo = pd.read_excel("2024echo_reports.xlsx")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")

echo["EF_Teich"] = echo["EF_Teich"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "name_key"]],
    echo[["name_key", "EF_Teich"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "EF_Teich": "ef_teich_%"
})

result = result[
    ["name", "age", "sex", "file_name", "ef_teich_%"]
]

filename = "matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Total unique ECG patients:", len(ecg))
print("Total unique Echo patients:", len(echo))
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)


Total unique ECG patients: 3663
Total unique Echo patients: 1556
Matched unique patients: 687
Saved file: matched_ecg_echo_20261023_150615.xlsx


,name,age,sex,file_name,ef_teich_%
0,JANNU CHIRANJEEVI,41 Years,Male,EK__[20240101134404]_JANNU CHIRANJEEVI____[]_...,77.7
1,S UDAY KISHORE,38 Years,Male,EK__[20240101120955]_S UDAY KISHORE____[]_.pdf,76.1
2,GUNDA VIJAYALAKSHMI,55 Years,Female,EK__[20240103192248]_GUNDA VIJAYALAKSHMI____[...,85.2
3,MATURI RAJAMMA,63 Years,Female,EK__[20240103190607]_MATURI RAJAMMA____[]_.pdf,66.7
4,MOGASANI MADHAVI,38 Years,Female,EK__[20240103144802]_MOGASANI MADHAVI____[]_.pdf,74.3
...,...,...,...,...,...
682,MADURI PADMAJA,57 Years,Female,EK__[20241214122623]_MADURI PADMAJA____[]_.pdf,74.0
683,CHEBROL PADMAVATHI,56 Years,Female,EK__[20241215101658]_CHEBROL PADMAVATHI____[]_...,73.7
684,ABBOJU SHANKARAIAH,57 Years,Male,EK__[20241215110240]_ABBOJU SHANKARAIAH____[]_...,42.9
685,CHINNALA YADAGIRI,75 Years,Male,EK__[20241224121631]_CHINNALA YADAGIRI____[]_.pdf,28.7


In [ ]:

import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("2022_ECG_EXCEL.xlsx")
echo = pd.read_excel("2022echo_reports.xlsx")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")

echo["EF_Teich"] = echo["EF_Teich"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "name_key"]],
    echo[["name_key", "EF_Teich"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "EF_Teich": "ef_teich_%"
})

result = result[
    ["name", "age", "sex", "file_name", "ef_teich_%"]
]

filename = "2022_matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Total unique ECG patients:", len(ecg))
print("Total unique Echo patients:", len(echo))
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)


KeyError: "['File Name'] not in index"

In [13]:

import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("ECG_2022.xlsx")
echo = pd.read_excel("2022echo_reports.xlsx", sheet_name="Sheet1")

def normalize_col(col):
    return re.sub(r"[^a-z0-9]", "", str(col).lower())

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

ecg_map = {normalize_col(c): c for c in ecg.columns}
echo_map = {normalize_col(c): c for c in echo.columns}

name_ecg = ecg_map["name"]
age_ecg = ecg_map["age"]
sex_ecg = ecg_map["sex"]
file_ecg = ecg_map["filename"]

name_echo = echo_map["name"]
ef_echo = echo_map["efteich"]

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg[name_ecg].apply(clean_name)
echo["name_key"] = echo[name_echo].apply(clean_name)

echo[ef_echo] = pd.to_numeric(echo[ef_echo], errors="coerce")
echo[ef_echo] = echo[ef_echo].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[[name_ecg, age_ecg, sex_ecg, file_ecg, "name_key"]],
    echo[["name_key", ef_echo]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    name_ecg: "name",
    age_ecg: "age",
    sex_ecg: "sex",
    file_ecg: "file_name",
    ef_echo: "ef_teich_%"
})

result = result[
    ["name", "age", "sex", "file_name", "ef_teich_%"]
]

filename = "2022_matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Unique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)


Unique ECG patients: 4297
Unique Echo patients: 943
Matched unique patients: 422
Saved file: 2022_matched_ecg_echo_20261023_154027.xlsx


,name,age,sex,file_name,ef_teich_%
0,BANDI SHANTHA,48 Years,Female,EK__[20220101113048]_BANDI SHANTHA____[]_.pdf,66.1
1,RAVULA MADHUKAR,41 Years,Male,EK__[20220106112548]_RAVULA MADHUKAR____[]_.pdf,72.1
2,KOMMULA BABU,64 Years,Male,EK__[20220119122734]_KOMMULA BABU____[]_.pdf,70.3
3,ALLOJU HARITHA,33 Years,Female,EK__[20220120113820]_ALLOJU HARITHA____[]_.pdf,37.7
4,THURPATI SRINIVAS,38 Years,Male,EK__[20220120181129]_THURPATI SRINIVAS____[]_.pdf,58.6
...,...,...,...,...,...
417,THANDAMALLA VENKATESHWARLU,53 Years,Male,EK__[20220830101016]_THANDAMALLA VENKATESHWARL...,49.6
418,BANDA SAROJANA,52 Years,Female,EK__[20220830110109]_BANDA SAROJANA____[]_.pdf,71.2
419,NANNEBOINA POOLA,35 Years,Female,EK__[20220830120955]_NANNEBOINA POOLA____[]_.pdf,67.7
420,YADA CHANDRAIAH,75 Years,Male,EK__[20220830162348]_YADA CHANDRAIAH____[]_.pdf,69.0


In [15]:

import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("ECG_2022.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2022echo_reports.xlsx", sheet_name="Sheet1")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")

echo["EF_Teich"] = echo["EF_Teich"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "name_key"]],
    echo[["name_key", "EF_Teich"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "EF_Teich": "ef_teich_%"
})

result = result[
    ["name", "age", "sex", "file_name", "ef_teich_%"]
]

filename = "2022_matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Unique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)


Unique ECG patients: 4297
Unique Echo patients: 943
Matched unique patients: 422
Saved file: 2022_matched_ecg_echo_20261023_154536.xlsx


,name,age,sex,file_name,ef_teich_%
0,BANDI SHANTHA,48 Years,Female,EK__[20220101113048]_BANDI SHANTHA____[]_.pdf,66.1
1,RAVULA MADHUKAR,41 Years,Male,EK__[20220106112548]_RAVULA MADHUKAR____[]_.pdf,72.1
2,KOMMULA BABU,64 Years,Male,EK__[20220119122734]_KOMMULA BABU____[]_.pdf,70.3
3,ALLOJU HARITHA,33 Years,Female,EK__[20220120113820]_ALLOJU HARITHA____[]_.pdf,37.7
4,THURPATI SRINIVAS,38 Years,Male,EK__[20220120181129]_THURPATI SRINIVAS____[]_.pdf,58.6
...,...,...,...,...,...
417,THANDAMALLA VENKATESHWARLU,53 Years,Male,EK__[20220830101016]_THANDAMALLA VENKATESHWARL...,49.6
418,BANDA SAROJANA,52 Years,Female,EK__[20220830110109]_BANDA SAROJANA____[]_.pdf,71.2
419,NANNEBOINA POOLA,35 Years,Female,EK__[20220830120955]_NANNEBOINA POOLA____[]_.pdf,67.7
420,YADA CHANDRAIAH,75 Years,Male,EK__[20220830162348]_YADA CHANDRAIAH____[]_.pdf,69.0


In [18]:
import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("2024_Ecg1.xlsx")
echo = pd.read_excel("2024echo_reports.xlsx")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")

echo["EF_Teich"] = echo["EF_Teich"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "name_key"]],
    echo[["name_key", "EF_Teich"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "EF_Teich": "ef_teich_%"
})

def classify_lv(row):
    ef = row["ef_teich_%"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        elif 41 <= ef <= 52:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        elif 41 <= ef < 54:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

result["LV"] = result.apply(classify_lv, axis=1)

result = result[
    ["name", "age", "sex", "file_name", "ef_teich_%", "LV"]
]

filename = "matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Unique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)

Unique ECG patients: 3663
Unique Echo patients: 1556
Matched unique patients: 687
Saved file: matched_ecg_echo_20261023_160109.xlsx


,name,age,sex,file_name,ef_teich_%,LV
0,JANNU CHIRANJEEVI,41 Years,Male,EK__[20240101134404]_JANNU CHIRANJEEVI____[]_...,77.7,Normal
1,S UDAY KISHORE,38 Years,Male,EK__[20240101120955]_S UDAY KISHORE____[]_.pdf,76.1,Normal
2,GUNDA VIJAYALAKSHMI,55 Years,Female,EK__[20240103192248]_GUNDA VIJAYALAKSHMI____[...,85.2,Normal
3,MATURI RAJAMMA,63 Years,Female,EK__[20240103190607]_MATURI RAJAMMA____[]_.pdf,66.7,Normal
4,MOGASANI MADHAVI,38 Years,Female,EK__[20240103144802]_MOGASANI MADHAVI____[]_.pdf,74.3,Normal
...,...,...,...,...,...,...
682,MADURI PADMAJA,57 Years,Female,EK__[20241214122623]_MADURI PADMAJA____[]_.pdf,74.0,Normal
683,CHEBROL PADMAVATHI,56 Years,Female,EK__[20241215101658]_CHEBROL PADMAVATHI____[]_...,73.7,Normal
684,ABBOJU SHANKARAIAH,57 Years,Male,EK__[20241215110240]_ABBOJU SHANKARAIAH____[]_...,42.9,Mild
685,CHINNALA YADAGIRI,75 Years,Male,EK__[20241224121631]_CHINNALA YADAGIRI____[]_.pdf,28.7,Severe


In [19]:
import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("Ecg_2022.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2022echo_reports.xlsx", sheet_name="Sheet1")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")

echo["EF_Teich"] = echo["EF_Teich"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "name_key"]],
    echo[["name_key", "EF_Teich"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "EF_Teich": "ef_teich_%"
})

def classify_lv(row):
    ef = row["ef_teich_%"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        elif 41 <= ef <= 52:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        elif 41 <= ef < 54:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

result["LV"] = result.apply(classify_lv, axis=1)

result = result[
    ["name", "age", "sex", "file_name", "ef_teich_%", "LV"]
]

filename = "matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Unique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)

Unique ECG patients: 4297
Unique Echo patients: 943
Matched unique patients: 422
Saved file: matched_ecg_echo_20261023_160408.xlsx


,name,age,sex,file_name,ef_teich_%,LV
0,BANDI SHANTHA,48 Years,Female,EK__[20220101113048]_BANDI SHANTHA____[]_.pdf,66.1,Normal
1,RAVULA MADHUKAR,41 Years,Male,EK__[20220106112548]_RAVULA MADHUKAR____[]_.pdf,72.1,Normal
2,KOMMULA BABU,64 Years,Male,EK__[20220119122734]_KOMMULA BABU____[]_.pdf,70.3,Normal
3,ALLOJU HARITHA,33 Years,Female,EK__[20220120113820]_ALLOJU HARITHA____[]_.pdf,37.7,Moderate
4,THURPATI SRINIVAS,38 Years,Male,EK__[20220120181129]_THURPATI SRINIVAS____[]_.pdf,58.6,Normal
...,...,...,...,...,...,...
417,THANDAMALLA VENKATESHWARLU,53 Years,Male,EK__[20220830101016]_THANDAMALLA VENKATESHWARL...,49.6,Mild
418,BANDA SAROJANA,52 Years,Female,EK__[20220830110109]_BANDA SAROJANA____[]_.pdf,71.2,Normal
419,NANNEBOINA POOLA,35 Years,Female,EK__[20220830120955]_NANNEBOINA POOLA____[]_.pdf,67.7,Normal
420,YADA CHANDRAIAH,75 Years,Male,EK__[20220830162348]_YADA CHANDRAIAH____[]_.pdf,69.0,Normal


In [22]:
import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021_data.xlsx", sheet_name="DEC")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)
echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)
age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)
sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)
file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)
ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns), None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": age_col,
    "ECG sex/gender": sex_col,
    "ECG file name": file_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")
echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

ecg_part = ecg[
    [ecg_name_col, age_col, sex_col, file_col, "name_key"]
].copy()

echo_part = echo[["name_key", ef_col]].copy()

result = pd.merge(
    ecg_part,
    echo_part,
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    ecg_name_col: "name",
    age_col: "age",
    sex_col: "sex",
    file_col: "file_name",
    ef_col: "EF (Teich) %"
})

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

result["LV"] = result.apply(classify_lv, axis=1)

result = result[
    ["name", "age", "sex", "file_name", "EF (Teich) %", "LV"]
]

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

result.to_excel(filename, index=False)

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'GENDER', 'DATE', 'INTERPRETATION COMMENTS', 'EF (TEICH)']

Unique ECG patients: 4897
Unique Echo patients: 420
Matched unique patients: 133
Saved file: matched_ecg_echo_20261023_163211.xlsx


,name,age,sex,file_name,EF (Teich) %,LV
0,MUTHYALA RAMESH,51 Years,Male,EK__[20210107103953]_MUTHYALA RAMESH____[]_.pdf,42.6,Mild
1,VADIKA MALLAIAH,65 Years,Male,EK__[20210129100858]_VADIKA MALLAIAH____[]_.pdf,57.2,Normal
2,MANUKA SAMPATH,45 Years,Male,EK__[20210130172205]_MANUKA SAMPATH____[]_.pdf,72.6,Normal
3,CHINNALA RANGAIAH,74 Years,Male,EK__[20210218182737]_CHINNALA RANGAIAH____[]_.pdf,79.8,Normal
4,MARRI MOHAN,42 Years,Male,EK__[20210315144503]_MARRI MOHAN____[]_.pdf,66.6,Normal
...,...,...,...,...,...,...
128,VALLURI LASYAPRIYA,16 Years,Female,EK__[20211224175558]_VALLURI LASYAPRIYA____[]_...,73.5,Normal
129,DODDIPATI LALITHA,49 Years,Female,EK__[20211225112053]_DODDIPATI LALITHA____[]_.pdf,72.4,Normal
130,PALAKURTHY PADMA,39 Years,Female,EK__[20211225121005]_PALAKURTHY PADMA____[]_.pdf,74.7,Normal
131,SAMA SUSHEELA,60 Years,Female,EK__[20211227132616]_SAMA SUSHEELA____[]_.pdf,65.8,Normal


In [1]:
import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)
echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)
age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)
sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)
file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)
ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns), None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": age_col,
    "ECG sex/gender": sex_col,
    "ECG file name": file_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")
echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

ecg_part = ecg[
    [ecg_name_col, age_col, sex_col, file_col, "name_key"]
].copy()

echo_part = echo[["name_key", ef_col]].copy()

result = pd.merge(
    ecg_part,
    echo_part,
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    ecg_name_col: "name",
    age_col: "age",
    sex_col: "sex",
    file_col: "file_name",
    ef_col: "EF (Teich) %"
})

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

result["LV"] = result.apply(classify_lv, axis=1)

result = result[
    ["name", "age", "sex", "file_name", "EF (Teich) %", "LV"]
]

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

result.to_excel(filename, index=False)

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']

Unique ECG patients: 4897
Unique Echo patients: 2047
Matched unique patients: 735
Saved file: matched_ecg_echo_20261028_153633.xlsx


,name,age,sex,file_name,EF (Teich) %,LV
0,EDULA SRINIVAS,60 Years,Male,EK__[20210102105955]_EDULA SRINIVAS____[]_.pdf,74.9,Normal
1,KYATHARAJU ASHOK,35 Years,Male,EK__[20210102114455]_KYATHARAJU ASHOK____[]_.pdf,75.5,Normal
2,SEETHA VIMALA,50 Years,Female,EK__[20210102123552]_SEETHA VIMALA____[]_.pdf,78.4,Normal
3,POTHARAJU SARANGAM,65 Years,Male,EK__[20210102210914]_POTHARAJU SARANGAM____[]_...,71.1,Normal
4,MALOTH PRAMEELA,46 Years,Female,EK__[20210104114532]_MALOTH PRAMEELA____[]_.pdf,76.9,Normal
...,...,...,...,...,...,...
730,DIDDI RAJKUMAR,51 Years,Male,EK__[20211229145120]_DIDDI RAJKUMAR____[]_.pdf,53.6,Normal
731,PARSA KISTAIAH,54 Years,Male,EK__[20211230095030]_PARSA KISTAIAH____[]_.pdf,77.0,Normal
732,DIDDI VAJRAMMA,85 Years,Female,EK__[20211230163917]_DIDDI VAJRAMMA____[]_.pdf,52.6,Mild
733,BODA SUNITHA,27 Years,Male,EK__[20211230182735]_BODA SUNITHA____[]_.pdf,79.4,Normal


In [24]:
import pandas as pd
import re
from datetime import datetime

ecg_file = "ECG.xlsx"
echo_file = "2021echo_reports.xlsx"

ecg = pd.read_excel(ecg_file, sheet_name="ECG Data")
ecg.columns = ecg.columns.astype(str).str.strip().str.upper()

def standardize_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

xls = pd.ExcelFile(echo_file)
echo = None

for sheet in xls.sheet_names:
    for header_row in range(5):
        temp = pd.read_excel(
            echo_file,
            sheet_name=sheet,
            header=header_row
        )

        temp.columns = [
            re.sub(r"\s+", " ", str(c)).strip().upper()
            for c in temp.columns
        ]

        name_found = "NAME" in temp.columns
        ef_found = any(
            c.replace("\\", "").replace(" ", "") in
            ["EF_TEICH_%", "EF_TEICH_%"]
            for c in temp.columns
        )

        if name_found and ef_found:
            echo = temp
            print("Echo sheet found:", sheet)
            print("Header row:", header_row + 1)
            break

    if echo is not None:
        break

if echo is None:
    print("Available sheets:", xls.sheet_names)
    for sheet in xls.sheet_names:
        print("\n", sheet)
        print(pd.read_excel(
            echo_file,
            sheet_name=sheet,
            header=None,
            nrows=5
        ).to_string(index=False, header=False))

    raise ValueError(
        "Could not automatically find the Echo Name and EF_Teich_% columns. "
        "Check the sheet/header preview above."
    )

def find_col(df, options):
    for col in options:
        if col in df.columns:
            return col
    return None

ecg_name = find_col(ecg, ["NAME"])
ecg_age = find_col(ecg, ["AGE"])
ecg_sex = find_col(ecg, ["SEX", "GENDER"])
ecg_file_col = find_col(ecg, ["FILE NAME", "FILENAME"])

echo_name = find_col(echo, ["NAME"])
echo_ef = find_col(
    echo,
    ["EF_TEICH\\_%", "EF_TEICH_%", "EF_TEICH_%"]
)

if any(x is None for x in [
    ecg_name, ecg_age, ecg_sex, ecg_file_col, echo_name, echo_ef
]):
    print("ECG columns:", ecg.columns.tolist())
    print("Echo columns:", echo.columns.tolist())
    raise ValueError("One or more required columns were not found.")

ecg["name_key"] = ecg[ecg_name].apply(standardize_name)
echo["name_key"] = echo[echo_name].apply(standardize_name)

echo[echo_ef] = pd.to_numeric(echo[echo_ef], errors="coerce")

echo[echo_ef] = echo[echo_ef].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[[ecg_name, ecg_age, ecg_sex, ecg_file_col, "name_key"]],
    echo[["name_key", echo_ef]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    ecg_name: "name",
    ecg_age: "age",
    ecg_sex: "sex",
    ecg_file_col: "file_name",
    echo_ef: "EF (Teich) %"
})

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        if ef >= 41:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        if ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    if ef < 31:
        return "Severe"

    return "Unknown"

result["LV"] = result.apply(classify_lv, axis=1)

result = result[
    ["name", "age", "sex", "file_name", "EF (Teich) %", "LV"]
]

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

result.to_excel(filename, index=False)

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved Excel file:", filename)

display(result)

Echo sheet found: Sheet1
Header row: 1

Unique ECG patients: 4897
Unique Echo patients: 2047
Matched unique patients: 735
Saved Excel file: matched_ecg_echo_20261023_164549.xlsx


,name,age,sex,file_name,EF (Teich) %,LV
0,EDULA SRINIVAS,60 Years,Male,EK__[20210102105955]_EDULA SRINIVAS____[]_.pdf,74.9,Normal
1,KYATHARAJU ASHOK,35 Years,Male,EK__[20210102114455]_KYATHARAJU ASHOK____[]_.pdf,75.5,Normal
2,SEETHA VIMALA,50 Years,Female,EK__[20210102123552]_SEETHA VIMALA____[]_.pdf,78.4,Normal
3,POTHARAJU SARANGAM,65 Years,Male,EK__[20210102210914]_POTHARAJU SARANGAM____[]_...,71.1,Normal
4,MALOTH PRAMEELA,46 Years,Female,EK__[20210104114532]_MALOTH PRAMEELA____[]_.pdf,76.9,Normal
...,...,...,...,...,...,...
730,DIDDI RAJKUMAR,51 Years,Male,EK__[20211229145120]_DIDDI RAJKUMAR____[]_.pdf,53.6,Normal
731,PARSA KISTAIAH,54 Years,Male,EK__[20211230095030]_PARSA KISTAIAH____[]_.pdf,77.0,Normal
732,DIDDI VAJRAMMA,85 Years,Female,EK__[20211230163917]_DIDDI VAJRAMMA____[]_.pdf,52.6,Mild
733,BODA SUNITHA,27 Years,Male,EK__[20211230182735]_BODA SUNITHA____[]_.pdf,79.4,Normal


In [ ]:
import pandas as pd
import re
from datetime import datetime

ecg_file = "ECG.xlsx"
echo_file = "2021echo_reports.xlsx"

ecg = pd.read_excel(ecg_file, sheet_name="ECG Data")
ecg.columns = ecg.columns.astype(str).str.strip().str.upper()

def standardize_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

xls = pd.ExcelFile(echo_file)
echo = None

for sheet in xls.sheet_names:
    for header_row in range(5):
        temp = pd.read_excel(
            echo_file,
            sheet_name=sheet,
            header=header_row
        )

        temp.columns = [
            re.sub(r"\s+", " ", str(c)).strip().upper()
            for c in temp.columns
        ]

        name_found = "NAME" in temp.columns
        ef_found = any(
            c.replace("\\", "").replace(" ", "") in
            ["EF_TEICH_%", "EF_TEICH_%"]
            for c in temp.columns
        )

        if name_found and ef_found:
            echo = temp
            print("Echo sheet found:", sheet)
            print("Header row:", header_row + 1)
            break

    if echo is not None:
        break

if echo is None:
    print("Available sheets:", xls.sheet_names)
    for sheet in xls.sheet_names:
        print("\n", sheet)
        print(pd.read_excel(
            echo_file,
            sheet_name=sheet,
            header=None,
            nrows=5
        ).to_string(index=False, header=False))

    raise ValueError(
        "Could not automatically find the Echo Name and EF_Teich_% columns. "
        "Check the sheet/header preview above."
    )

def find_col(df, options):
    for col in options:
        if col in df.columns:
            return col
    return None

ecg_name = find_col(ecg, ["NAME"])
ecg_age = find_col(ecg, ["AGE"])
ecg_sex = find_col(ecg, ["SEX", "GENDER"])
ecg_file_col = find_col(ecg, ["FILE NAME", "FILENAME"])

echo_name = find_col(echo, ["NAME"])
echo_ef = find_col(
    echo,
    ["EF_TEICH\\_%", "EF_TEICH_%", "EF_TEICH_%"]
)

if any(x is None for x in [
    ecg_name, ecg_age, ecg_sex, ecg_file_col, echo_name, echo_ef
]):
    print("ECG columns:", ecg.columns.tolist())
    print("Echo columns:", echo.columns.tolist())
    raise ValueError("One or more required columns were not found.")

ecg["name_key"] = ecg[ecg_name].apply(standardize_name)
echo["name_key"] = echo[echo_name].apply(standardize_name)

echo[echo_ef] = pd.to_numeric(echo[echo_ef], errors="coerce")

echo[echo_ef] = echo[echo_ef].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[[ecg_name, ecg_age, ecg_sex, ecg_file_col, "name_key"]],
    echo[["name_key", echo_ef]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    ecg_name: "name",
    ecg_age: "age",
    ecg_sex: "sex",
    ecg_file_col: "file_name",
    echo_ef: "EF (Teich) %"
})

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        if ef >= 41:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        if ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    if ef < 31:
        return "Severe"

    return "Unknown"

result["LV"] = result.apply(classify_lv, axis=1)

result = result[
    ["name", "age", "sex", "file_name", "EF (Teich) %", "LV"]
]

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

result.to_excel(filename, index=False)

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))
print("Saved Excel file:", filename)

display(result)

In [16]:

import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("2024_Ecg1.xlsx")
echo = pd.read_excel("2024echo_reports.xlsx")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")
echo["EF_Teich"] = echo["EF_Teich"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "ECG_COMMENT", "name_key"]],
    echo[["name_key", "EF_Teich"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "ECG_COMMENT": "ecg_comment",
    "EF_Teich": "ef_teich_%"
})

result = result[
    ["name", "age", "sex", "file_name", "ecg_comment", "ef_teich_%"]
]

counts = (
    result["ecg_comment"]
    .fillna("MISSING")
    .astype(str)
    .str.strip()
    .replace("", "MISSING")
    .value_counts()
    .rename_axis("ECG_COMMENT")
    .reset_index(name="Count")
)

filename = "matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Unique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))

print("\nECG comment counts:")
display(counts)

print("Saved Excel file:", filename)
display(result)


Unique ECG patients: 3663
Unique Echo patients: 1556
Matched unique patients: 687

ECG comment counts:


,ECG_COMMENT,Count
0,ABNORMAL ECG,307
1,BORDERLINE ECG,166
2,NORMAL ECG,110
3,OTHERWISE NORMAL ECG,104


Saved Excel file: matched_ecg_echo_20261023_155424.xlsx


,name,age,sex,file_name,ecg_comment,ef_teich_%
0,JANNU CHIRANJEEVI,41 Years,Male,EK__[20240101134404]_JANNU CHIRANJEEVI____[]_...,ABNORMAL ECG,77.7
1,S UDAY KISHORE,38 Years,Male,EK__[20240101120955]_S UDAY KISHORE____[]_.pdf,NORMAL ECG,76.1
2,GUNDA VIJAYALAKSHMI,55 Years,Female,EK__[20240103192248]_GUNDA VIJAYALAKSHMI____[...,ABNORMAL ECG,85.2
3,MATURI RAJAMMA,63 Years,Female,EK__[20240103190607]_MATURI RAJAMMA____[]_.pdf,NORMAL ECG,66.7
4,MOGASANI MADHAVI,38 Years,Female,EK__[20240103144802]_MOGASANI MADHAVI____[]_.pdf,OTHERWISE NORMAL ECG,74.3
...,...,...,...,...,...,...
682,MADURI PADMAJA,57 Years,Female,EK__[20241214122623]_MADURI PADMAJA____[]_.pdf,ABNORMAL ECG,74.0
683,CHEBROL PADMAVATHI,56 Years,Female,EK__[20241215101658]_CHEBROL PADMAVATHI____[]_...,ABNORMAL ECG,73.7
684,ABBOJU SHANKARAIAH,57 Years,Male,EK__[20241215110240]_ABBOJU SHANKARAIAH____[]_...,ABNORMAL ECG,42.9
685,CHINNALA YADAGIRI,75 Years,Male,EK__[20241224121631]_CHINNALA YADAGIRI____[]_.pdf,ABNORMAL ECG,28.7


In [17]:

import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("ECG_2022.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2022echo_reports.xlsx", sheet_name="Sheet1")

ecg.columns = ecg.columns.str.strip()
echo.columns = echo.columns.str.strip()

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg["name_key"] = ecg["Name"].apply(clean_name)
echo["name_key"] = echo["Name"].apply(clean_name)

echo["EF_Teich"] = pd.to_numeric(echo["EF_Teich"], errors="coerce")
echo["EF_Teich"] = echo["EF_Teich"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")
echo = echo.drop_duplicates(subset="name_key", keep="first")

result = pd.merge(
    ecg[["Name", "Age", "Sex", "File Name", "ECG_COMMENT", "name_key"]],
    echo[["name_key", "EF_Teich"]],
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    "Name": "name",
    "Age": "age",
    "Sex": "sex",
    "File Name": "file_name",
    "ECG_COMMENT": "ecg_comment",
    "EF_Teich": "ef_teich_%"
})

result = result[
    ["name", "age", "sex", "file_name", "ecg_comment", "ef_teich_%"]
]

counts = (
    result["ecg_comment"]
    .fillna("MISSING")
    .astype(str)
    .str.strip()
    .replace("", "MISSING")
    .value_counts()
    .rename_axis("ECG_COMMENT")
    .reset_index(name="Count")
)

filename = "matched_ecg_echo_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".xlsx"

result.to_excel(filename, index=False)

print("Unique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched unique patients:", len(result))

print("\nECG comment counts:")
display(counts)

print("Saved Excel file:", filename)
display(result)


Unique ECG patients: 4297
Unique Echo patients: 943
Matched unique patients: 422

ECG comment counts:


,ECG_COMMENT,Count
0,ABNORMAL ECG,212
1,BORDERLINE ECG,81
2,OTHERWISE NORMAL ECG,76
3,NORMAL ECG,53


Saved Excel file: matched_ecg_echo_20261023_155531.xlsx


,name,age,sex,file_name,ecg_comment,ef_teich_%
0,BANDI SHANTHA,48 Years,Female,EK__[20220101113048]_BANDI SHANTHA____[]_.pdf,ABNORMAL ECG,66.1
1,RAVULA MADHUKAR,41 Years,Male,EK__[20220106112548]_RAVULA MADHUKAR____[]_.pdf,OTHERWISE NORMAL ECG,72.1
2,KOMMULA BABU,64 Years,Male,EK__[20220119122734]_KOMMULA BABU____[]_.pdf,OTHERWISE NORMAL ECG,70.3
3,ALLOJU HARITHA,33 Years,Female,EK__[20220120113820]_ALLOJU HARITHA____[]_.pdf,ABNORMAL ECG,37.7
4,THURPATI SRINIVAS,38 Years,Male,EK__[20220120181129]_THURPATI SRINIVAS____[]_.pdf,NORMAL ECG,58.6
...,...,...,...,...,...,...
417,THANDAMALLA VENKATESHWARLU,53 Years,Male,EK__[20220830101016]_THANDAMALLA VENKATESHWARL...,ABNORMAL ECG,49.6
418,BANDA SAROJANA,52 Years,Female,EK__[20220830110109]_BANDA SAROJANA____[]_.pdf,BORDERLINE ECG,71.2
419,NANNEBOINA POOLA,35 Years,Female,EK__[20220830120955]_NANNEBOINA POOLA____[]_.pdf,NORMAL ECG,67.7
420,YADA CHANDRAIAH,75 Years,Male,EK__[20220830162348]_YADA CHANDRAIAH____[]_.pdf,OTHERWISE NORMAL ECG,69.0


In [2]:
import pandas as pd
import re
from datetime import datetime

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9\s]", "", str(name).lower())
    return " ".join(sorted(name.split()))

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": age_col,
    "ECG sex/gender": sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

ecg = ecg.drop_duplicates(subset="name_key", keep="first")

echo = echo.drop_duplicates(
    subset=["name_key", "echo_study_date"],
    keep="first"
)

ecg_part = ecg[
    [ecg_name_col, age_col, sex_col, file_col, "name_key"]
].copy()

echo_part = echo[
    ["name_key", "echo_study_date", ef_col]
].copy()

result = pd.merge(
    ecg_part,
    echo_part,
    on="name_key",
    how="inner"
)

result = result.rename(columns={
    ecg_name_col: "name",
    age_col: "age",
    sex_col: "sex",
    file_col: "file_name",
    ef_col: "EF (Teich) %"
})

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

result["LV"] = result.apply(classify_lv, axis=1)

result = result[
    [
        "name",
        "age",
        "sex",
        "file_name",
        "echo_study_date",
        "EF (Teich) %",
        "LV"
    ]
]

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

result.to_excel(filename, index=False)

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched ECG-Echo records:", len(result))
print("Saved file:", filename)

display(result)



ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']


C:\Users\Admin\AppData\Local\Temp\ipykernel_5360\1525279323.py:78: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(



Unique ECG patients: 4897
Unique Echo patients: 2047
Matched ECG-Echo records: 764
Saved file: matched_ecg_echo_20261028_155158.xlsx


,name,age,sex,file_name,echo_study_date,EF (Teich) %,LV
0,EDULA SRINIVAS,60 Years,Male,EK__[20210102105955]_EDULA SRINIVAS____[]_.pdf,2021-01-02,74.9,Normal
1,KYATHARAJU ASHOK,35 Years,Male,EK__[20210102114455]_KYATHARAJU ASHOK____[]_.pdf,2021-01-02,75.5,Normal
2,SEETHA VIMALA,50 Years,Female,EK__[20210102123552]_SEETHA VIMALA____[]_.pdf,2021-01-02,78.4,Normal
3,POTHARAJU SARANGAM,65 Years,Male,EK__[20210102210914]_POTHARAJU SARANGAM____[]_...,2021-01-03,71.1,Normal
4,POTHARAJU SARANGAM,65 Years,Male,EK__[20210102210914]_POTHARAJU SARANGAM____[]_...,2021-09-30,65.7,Normal
...,...,...,...,...,...,...,...
759,DIDDI RAJKUMAR,51 Years,Male,EK__[20211229145120]_DIDDI RAJKUMAR____[]_.pdf,2021-12-29,53.6,Normal
760,PARSA KISTAIAH,54 Years,Male,EK__[20211230095030]_PARSA KISTAIAH____[]_.pdf,2021-12-30,77.0,Normal
761,DIDDI VAJRAMMA,85 Years,Female,EK__[20211230163917]_DIDDI VAJRAMMA____[]_.pdf,2021-12-30,52.6,Mild
762,BODA SUNITHA,27 Years,Male,EK__[20211230182735]_BODA SUNITHA____[]_.pdf,2021-12-16,79.4,Normal


In [3]:
from rapidfuzz import fuzz
import re
import pandas as pd

def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"[^a-zA-Z0-9]", "", str(name).lower())
    return name

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")
echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

matches = []

for _, e in ecg.iterrows():
    for _, h in echo.iterrows():
        score = fuzz.ratio(e["name_key"], h["name_key"])

        if score >= 80:
            matches.append({
                "name": e[ecg_name_col],
                "age": e[age_col],
                "sex": e[sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": score
            })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name", "age", "sex", "file_name",
            "echo_study_date", "EF (Teich) %",
            "LV", "match_score"
        ]
    ]

result.to_excel("matched_ecg_echo_fuzzy.xlsx", index=False)

display(result)

ModuleNotFoundError: No module named 'rapidfuzz'

In [4]:
%pip install rapidfuzz

  Using cached rapidfuzz-3.14.6-cp313-cp313-win_amd64.whl.metadata (12 kB)
Using cached rapidfuzz-3.14.6-cp313-cp313-win_amd64.whl (1.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
from rapidfuzz import fuzz

print(fuzz.ratio("CHADRAGIRI NARESHKUMAR", "NARESH KUMAR CHANDRAGIRI"))

47.82608695652174


In [4]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": age_col,
    "ECG sex/gender": sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")
matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]

    best_record = None
    best_score = 0

    for h in echo_records:
        echo_name = h["name_key"]
        echo_compact = h["compact_key"]

        score1 = fuzz.ratio(ecg_name, echo_name)
        score2 = fuzz.ratio(ecg_compact, echo_compact)

        score3 = fuzz.token_sort_ratio(ecg_name, echo_name)

        score = max(score1, score2, score3)

        if score > best_score:
            best_score = score
            best_record = h

    if best_record is not None and best_score >= 85:
        matched_echo = [
            h for h in echo_records
            if h["name_key"] == best_record["name_key"]
        ]

        if best_score < 95:
            matched_echo = [
                h for h in echo_records
                if max(
                    fuzz.ratio(ecg_name, h["name_key"]),
                    fuzz.ratio(ecg_compact, h["compact_key"]),
                    fuzz.token_sort_ratio(ecg_name, h["name_key"])
                ) >= 85
            ]

        for h in matched_echo:
            matches.append({
                "name": e[ecg_name_col],
                "echo_name": h[echo_name_col],
                "age": e[age_col],
                "sex": e[sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": max(
                    fuzz.ratio(ecg_name, h["name_key"]),
                    fuzz.ratio(ecg_compact, h["compact_key"]),
                    fuzz.token_sort_ratio(ecg_name, h["name_key"])
                )
            })
    else:
        unmatched.append({
            "ECG name": e[ecg_name_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2)
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = str(row["sex"]).strip().lower()

    if pd.isna(ef):
        return "Unknown"

    if sex in ["male", "m"]:
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex in ["female", "f"]:
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "echo_name",
            "age",
            "sex",
            "file_name",
            "echo_study_date",
            "EF (Teich) %",
            "LV",
            "match_score"
        ]
    ]

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_df.to_excel(writer, sheet_name="Unmatched ECG", index=False)

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched ECG-Echo records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)
display(unmatched_df)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\3952621777.py:86: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(



Unique ECG patients: 4898
Unique Echo patients: 2064
Matched ECG-Echo records: 2076
Unmatched ECG records: 3427
Saved file: matched_ecg_echo_20261028_161232.xlsx


,name,echo_name,age,sex,file_name,echo_study_date,EF (Teich) %,LV,match_score
0,KALLURI VARALAXMI,KALLURI VARALAKSHMI,70 Years,Female,EK__[20210101110036]_KALLURI VARALAXMI____[]_.pdf,2021-10-02,68.7,Normal,88.888889
1,THALAPELLI VINOD,THALLAPALLI VINOD,28 Years,Male,EK__[20210101153258]_THALAPELLI VINOD____[]_.pdf,2021-01-01,71.8,Normal,90.909091
2,MD JAMEELAHMED,JAMEEL AHMED,52 Years,Male,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,2021-08-02,36.2,Moderate,91.666667
3,MD JAMEELAHMED,JAMEEL AHMED,52 Years,Male,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,2021-01-02,37.4,Moderate,91.666667
4,EDULA SRINIVAS,SRINIVAS EDULA,60 Years,Male,EK__[20210102105955]_EDULA SRINIVAS____[]_.pdf,2021-01-02,74.9,Normal,100.000000
...,...,...,...,...,...,...,...,...,...
2071,KHAYURUNNISA,KHAIRUNNISA,93 Years,Female,EK__[20211231124322]_KHAYURUNNISA____[]_.pdf,2021-12-31,70.6,Normal,86.956522
2072,NEERUGONGA CHANDRAKALA,CHANDRAKALA NEEERUBONDA,38 Years,Female,EK__[20211231131117]_NEERUGONGA CHANDRAKALA___...,2021-12-31,75.0,Normal,88.888889
2073,NEERUGONDA CHANDRAKALA,CHANDRAKALA NEEERUBONDA,38 Years,Female,EK__[20211231142128]_NEERUGONDA CHANDRAKALA___...,2021-12-31,75.0,Normal,93.333333
2074,BODA NARSIMHA,BODA NARASIMHA,45 Years,Male,EK__[20211231143841]_BODA NARSIMHA____[]_.pdf,2021-03-25,71.3,Normal,96.296296


,ECG name,Best Echo match,Match score
0,BUKYA VEERANNA,BHUKYA ANJALI,66.67
1,CHINTHA THIRUPATHI,POTHU TIRUPATHI,72.73
2,AKUNURI BHADRAIAH,AKUNURI VENKATAIAH,74.29
3,BOINAPELLI ANJANIDEVI,BONALA UMA DEVI,66.67
4,YERRAMREDDY VIJAYA,BOJJAM RAM REDDY,64.71
...,...,...,...
3422,BODA TEJESHKUMAR,POLU SATISH KUMAR,66.67
3423,VARANASI PADMAJA DEVI,VEMURI PADMAJA DEVI,80.00
3424,CHEVULA ANDALAMMA,CHAGANTI ILAMMA,66.67
3425,CHEVULA ANDALAMMA,CHAGANTI ILAMMA,66.67


In [6]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def normalize_sex(value):
    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value in ["M", "MALE"]:
        return "M"

    if value in ["F", "FEMALE"]:
        return "F"

    return value if value else None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = pd.to_numeric(ecg[ecg_age_col], errors="coerce")
echo["age_key"] = pd.to_numeric(echo[echo_age_col], errors="coerce")

ecg["sex_key"] = ecg[ecg_sex_col].apply(normalize_sex)
echo["sex_key"] = echo[echo_sex_col].apply(normalize_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "echo_study_date", "age_key", "sex_key"],
    keep="first"
)

matches = []
unmatched = []

for _, e in ecg.iterrows():
    candidates = []

    for _, h in echo.iterrows():
        name_score = max(
            fuzz.ratio(e["name_key"], h["name_key"]),
            fuzz.ratio(e["compact_key"], h["compact_key"]),
            fuzz.token_sort_ratio(e["name_key"], h["name_key"])
        )

        if name_score < 85:
            continue

        age_match = (
            pd.notna(e["age_key"])
            and pd.notna(h["age_key"])
            and e["age_key"] == h["age_key"]
        )

        sex_match = (
            e["sex_key"] is not None
            and h["sex_key"] is not None
            and e["sex_key"] == h["sex_key"]
        )

        if age_match and sex_match:
            candidates.append((h, name_score))

    if candidates:
        best_score = max(score for _, score in candidates)

        best_names = [
            h["name_key"]
            for h, score in candidates
            if score == best_score
        ]

        for h, score in candidates:
            if h["name_key"] in best_names:
                matches.append({
                    "ECG Name": e[ecg_name_col],
                    "Echo Name": h[echo_name_col],
                    "ECG Age": e[ecg_age_col],
                    "Echo Age": h[echo_age_col],
                    "ECG Sex": e[ecg_sex_col],
                    "Echo Sex": h[echo_sex_col],
                    "File Name": e[file_col],
                    "Echo Study Date": h["echo_study_date"],
                    "EF (Teich) %": h[ef_col],
                    "Name Match Score": score,
                    "Age Match": "Yes",
                    "Sex Match": "Yes"
                })
    else:
        unmatched.append({
            "ECG Name": e[ecg_name_col],
            "ECG Age": e[ecg_age_col],
            "ECG Sex": e[ecg_sex_col],
            "Reason": "No record matched name, age, and sex"
        })

result = pd.DataFrame(matches)
unmatched_df = pd.DataFrame(unmatched)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = normalize_sex(row["ECG Sex"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_df.to_excel(writer, sheet_name="Unmatched ECG", index=False)

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)
display(unmatched_df)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\4262108105.py:116: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(


KeyboardInterrupt: 

In [ ]:
from rapidfuzz import process, fuzz
from collections import defaultdict

matches = []
unmatched = []

echo_groups = defaultdict(list)

for _, h in echo.iterrows():
    echo_groups[h["name_key"]].append(h)

echo_names = list(echo_groups.keys())

compact_groups = defaultdict(list)

for name in echo_names:
    compact = re.sub(r"[^A-Z0-9]", "", name)
    compact_groups[compact].append(name)

def find_candidates(name):
    if name in echo_groups:
        return [(name, 100)]

    compact = re.sub(r"[^A-Z0-9]", "", name)

    if compact in compact_groups:
        return [(n, 100) for n in compact_groups[compact]]

    return process.extract(
        name,
        echo_names,
        scorer=fuzz.WRatio,
        score_cutoff=85,
        limit=10
    ) and [
        (n, score)
        for n, score, _ in process.extract(
            name,
            echo_names,
            scorer=fuzz.WRatio,
            score_cutoff=85,
            limit=10
        )
    ]

for _, e in ecg.iterrows():
    candidates = find_candidates(e["name_key"])
    found = False

    for echo_name, score in candidates:
        for h in echo_groups[echo_name]:
            age_match = (
                pd.notna(e["age_key"])
                and pd.notna(h["age_key"])
                and e["age_key"] == h["age_key"]
            )

            sex_match = (
                pd.notna(e["sex_key"])
                and pd.notna(h["sex_key"])
                and e["sex_key"] == h["sex_key"]
            )

            if age_match and sex_match:
                matches.append({
                    "ECG Name": e[ecg_name_col],
                    "Echo Name": h[echo_name_col],
                    "ECG Age": e[ecg_age_col],
                    "Echo Age": h[echo_age_col],
                    "ECG Sex": e[ecg_sex_col],
                    "Echo Sex": h[echo_sex_col],
                    "File Name": e[file_col],
                    "Echo Study Date": h["echo_study_date"],
                    "EF (Teich) %": h[ef_col],
                    "Name Match Score": score,
                    "Age Match": "Yes",
                    "Sex Match": "Yes"
                })
                found = True

    if not found:
        unmatched.append({
            "ECG Name": e[ecg_name_col],
            "ECG Age": e[ecg_age_col],
            "ECG Sex": e[ecg_sex_col],
            "Reason": "No matching name, age, and sex"
        })

result = pd.DataFrame(matches)
unmatched_df = pd.DataFrame(unmatched)

In [8]:
import pandas as pd
import re
from collections import defaultdict
from rapidfuzz import process, fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [re.sub(r"\s+", " ", str(c)).strip().upper() for c in df.columns]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

def find_col(df, options):
    for c in options:
        if c in df.columns:
            return c
    raise ValueError(f"Column not found. Tried {options}. Available columns: {list(df.columns)}")

ecg_name_col = find_col(ecg, ["NAME", "PATIENT NAME", "PATIENT_NAME"])
echo_name_col = find_col(echo, ["NAME", "PATIENT NAME", "PATIENT_NAME"])

ecg_age_col = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age_col = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex_col = find_col(ecg, ["SEX", "GENDER"])
echo_sex_col = find_col(echo, ["SEX", "GENDER"])

date_col = find_col(echo, ["ECHO STUDY DATE", "STUDY DATE", "DATE OF STUDY", "EXAM DATE"])

ef_col = find_col(echo, ["EF (TEICH) %", "EF (TEICH)", "EF", "EJECTION FRACTION"])

def clean_name(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = re.sub(r"[^A-Z0-9]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x

def name_key(x):
    return "".join(sorted(clean_name(x).split()))

def compact_key(x):
    return re.sub(r"[^A-Z0-9]", "", clean_name(x))

def clean_sex(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    if x in ["M", "MALE"]:
        return "M"
    if x in ["F", "FEMALE"]:
        return "F"
    return x

def clean_age(x):
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except (ValueError, TypeError):
        return None

ecg["name_key"] = ecg[ecg_name_col].apply(name_key)
echo["name_key"] = echo[echo_name_col].apply(name_key)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_key)
echo["compact_key"] = echo[echo_name_col].apply(compact_key)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo[date_col] = pd.to_datetime(echo[date_col], errors="coerce")

echo_groups = defaultdict(list)
compact_groups = defaultdict(list)

for idx, row in echo.iterrows():
    if row["name_key"]:
        echo_groups[row["name_key"]].append(idx)
        compact_groups[row["compact_key"]].append(idx)

echo_names = list(echo_groups.keys())

def get_candidates(key, compact):
    if not key:
        return []

    if key in echo_groups:
        return [(key, 100)]

    if compact in compact_groups:
        return [(echo.loc[i, "name_key"], 100) for i in compact_groups[compact]]

    results = process.extract(
        key,
        echo_names,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=85,
        limit=10
    )

    return [(n, score) for n, score, _ in results]

matches = []
unmatched = []

for _, e in ecg.iterrows():
    candidates = get_candidates(e["name_key"], e["compact_key"])
    found = False

    for echo_name, score in candidates:
        for i in echo_groups[echo_name]:
            h = echo.loc[i]

            age_match = (
                e["age_key"] is not None
                and h["age_key"] is not None
                and e["age_key"] == h["age_key"]
            )

            sex_match = (
                bool(e["sex_key"])
                and bool(h["sex_key"])
                and e["sex_key"] == h["sex_key"]
            )

            if age_match and sex_match:
                ef = pd.to_numeric(h[ef_col], errors="coerce")

                if pd.isna(ef):
                    lv = "Unknown"
                elif e["sex_key"] == "M":
                    if ef > 52:
                        lv = "Normal"
                    elif ef >= 41:
                        lv = "Mild"
                    elif ef >= 31:
                        lv = "Moderate"
                    else:
                        lv = "Severe"
                elif e["sex_key"] == "F":
                    if ef >= 54:
                        lv = "Normal"
                    elif ef >= 41:
                        lv = "Mild"
                    elif ef >= 31:
                        lv = "Moderate"
                    else:
                        lv = "Severe"
                else:
                    lv = "Unknown"

                matches.append({
                    "ECG Name": e[ecg_name_col],
                    "Echo Name": h[echo_name_col],
                    "ECG Age": e[ecg_age_col],
                    "Echo Age": h[echo_age_col],
                    "ECG Sex": e[ecg_sex_col],
                    "Echo Sex": h[echo_sex_col],
                    "Echo Study Date": h[date_col],
                    "EF (Teich) %": h[ef_col],
                    "LV Category": lv,
                    "Name Match Score": score,
                    "Age Match": age_match,
                    "Sex Match": sex_match
                })
                found = True

    if not found:
        unmatched.append({
            "ECG Name": e[ecg_name_col],
            "ECG Age": e[ecg_age_col],
            "ECG Sex": e[ecg_sex_col]
        })

result = pd.DataFrame(matches)
unmatched_df = pd.DataFrame(unmatched)

if not result.empty:
    result = result.sort_values(
        ["ECG Name", "Echo Study Date"],
        na_position="last"
    )

with pd.ExcelWriter("ECG_Echo_Matched_Results.xlsx", engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_df.to_excel(writer, sheet_name="Unmatched ECG", index=False)

print("Matching completed.")
print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved to ECG_Echo_Matched_Results.xlsx")

ValueError: Column not found. Tried ['EF (TEICH) %', 'EF (TEICH)', 'EF', 'EJECTION FRACTION']. Available columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']

In [9]:
import pandas as pd
import re
from collections import defaultdict
from rapidfuzz import process, fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

def clean_name(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = re.sub(r"[^A-Z0-9]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

def name_key(x):
    return " ".join(sorted(clean_name(x).split()))

def compact_key(x):
    return re.sub(r"[^A-Z0-9]", "", clean_name(x))

def clean_age(x):
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except (ValueError, TypeError):
        return None

def clean_sex(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    if x in ["M", "MALE"]:
        return "M"
    if x in ["F", "FEMALE"]:
        return "F"
    return x

ecg["NAME_KEY"] = ecg["NAME"].apply(name_key)
ecg["COMPACT_KEY"] = ecg["NAME"].apply(compact_key)
ecg["AGE_KEY"] = ecg["AGE"].apply(clean_age)
ecg["SEX_KEY"] = ecg["GENDER"].apply(clean_sex)

echo["NAME_KEY"] = echo["NAME"].apply(name_key)
echo["COMPACT_KEY"] = echo["NAME"].apply(compact_key)
echo["AGE_KEY"] = echo["AGE"].apply(clean_age)
echo["SEX_KEY"] = echo["GENDER"].apply(clean_sex)

echo["STUDY DATE"] = pd.to_datetime(echo["STUDY DATE"], errors="coerce")

ef_col = "EF_TEICH_%"
if ef_col not in echo.columns:
    ef_col = "EF_CUBED_%"

if ef_col not in echo.columns:
    raise ValueError("Neither EF_TEICH_% nor EF_CUBED_% exists in the Echo file.")

echo_groups = defaultdict(list)
compact_groups = defaultdict(list)

for i, row in echo.iterrows():
    if row["NAME_KEY"]:
        echo_groups[row["NAME_KEY"]].append(i)
        compact_groups[row["COMPACT_KEY"]].append(i)

echo_names = list(echo_groups.keys())

def get_candidates(key, compact):
    if not key:
        return []

    if key in echo_groups:
        return [(key, 100)]

    if compact in compact_groups:
        return list({
            (echo.loc[i, "NAME_KEY"], 100)
            for i in compact_groups[compact]
        })

    results = process.extract(
        key,
        echo_names,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=85,
        limit=10
    )

    return [(name, score) for name, score, _ in results]

matches = []
unmatched = []

for _, e in ecg.iterrows():
    candidates = get_candidates(e["NAME_KEY"], e["COMPACT_KEY"])
    found = False

    for echo_name, score in candidates:
        for i in echo_groups[echo_name]:
            h = echo.loc[i]

            age_match = (
                e["AGE_KEY"] is not None
                and h["AGE_KEY"] is not None
                and e["AGE_KEY"] == h["AGE_KEY"]
            )

            sex_match = (
                bool(e["SEX_KEY"])
                and bool(h["SEX_KEY"])
                and e["SEX_KEY"] == h["SEX_KEY"]
            )

            if not (age_match and sex_match):
                continue

            ef = pd.to_numeric(h[ef_col], errors="coerce")

            if pd.isna(ef):
                lv = "Unknown"
            elif e["SEX_KEY"] == "M":
                if ef > 52:
                    lv = "Normal"
                elif ef >= 41:
                    lv = "Mild"
                elif ef >= 31:
                    lv = "Moderate"
                else:
                    lv = "Severe"
            elif e["SEX_KEY"] == "F":
                if ef >= 54:
                    lv = "Normal"
                elif ef >= 41:
                    lv = "Mild"
                elif ef >= 31:
                    lv = "Moderate"
                else:
                    lv = "Severe"
            else:
                lv = "Unknown"

            matches.append({
                "ECG Name": e["NAME"],
                "Echo Name": h["NAME"],
                "ECG Date": e["DATE"],
                "ECG Time": e["TIME"],
                "ECG Age": e["AGE"],
                "Echo Age": h["AGE"],
                "ECG Gender": e["GENDER"],
                "Echo Gender": h["GENDER"],
                "Echo MRN": h["MRN"],
                "Echo Study Date": h["STUDY DATE"],
                "EF Column Used": ef_col,
                "EF (%)": ef,
                "LV Category": lv,
                "Name Match Score": score,
                "Age Match": age_match,
                "Gender Match": sex_match,
                "Echo File Name": h["FILE_NAME"],
                "Echo File Path": h["FILE_PATH"]
            })

            found = True

    if not found:
        unmatched.append({
            "ECG Name": e["NAME"],
            "ECG Date": e["DATE"],
            "ECG Age": e["AGE"],
            "ECG Gender": e["GENDER"]
        })

result = pd.DataFrame(matches)
unmatched_df = pd.DataFrame(unmatched)

if not result.empty:
    result = result.sort_values(
        ["ECG Name", "Echo Study Date"],
        na_position="last"
    )

with pd.ExcelWriter("ECG_Echo_Matched_Results.xlsx", engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_df.to_excel(writer, sheet_name="Unmatched ECG", index=False)

print("Matching completed.")
print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Output file: ECG_Echo_Matched_Results.xlsx")

C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\147603405.py:60: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["STUDY DATE"] = pd.to_datetime(echo["STUDY DATE"], errors="coerce")


Matching completed.
Matched records: 0
Unmatched ECG records: 5405
Output file: ECG_Echo_Matched_Results.xlsx


In [11]:

import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"
    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()
    match = re.search(r"\d+", age)

    if match:
        return int(match.group())

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "echo_name": h[echo_name_col],
                "ECG age": e[ecg_age_col],
                "Echo age": h[echo_age_col],
                "ECG sex": e[ecg_sex_col],
                "Echo gender": h[echo_sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": round(score, 2),
                "age_match": True,
                "sex_match": True
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG sex": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["ECG sex"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "echo_name",
            "ECG age",
            "Echo age",
            "ECG sex",
            "Echo gender",
            "file_name",
            "echo_study_date",
            "EF (Teich) %",
            "LV",
            "match_score",
            "age_match",
            "sex_match"
        ]
    ]

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )
    unmatched_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched ECG-Echo records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)
display(unmatched_df)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\2232287829.py:126: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(



Unique ECG patients: 4898
Unique Echo patients: 2064
Matched ECG-Echo records: 1447
Unmatched ECG records: 4004
Saved file: matched_ecg_echo_20261028_163727.xlsx


,name,echo_name,ECG age,Echo age,ECG sex,Echo gender,file_name,echo_study_date,EF (Teich) %,LV,match_score,age_match,sex_match
0,THALAPELLI VINOD,THALLAPALLI VINOD,28 Years,28.0,Male,Male,EK__[20210101153258]_THALAPELLI VINOD____[]_.pdf,2021-01-01,71.8,Normal,90.91,True,True
1,MD JAMEELAHMED,JAMEEL AHMED,52 Years,52.0,Male,Male,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,2021-08-02,36.2,Moderate,91.67,True,True
2,EDULA SRINIVAS,SRINIVAS EDULA,60 Years,60.0,Male,Male,EK__[20210102105955]_EDULA SRINIVAS____[]_.pdf,2021-01-02,74.9,Normal,100.00,True,True
3,YERRABELLI BAPURAO,YERRABELLI BAPU RAO,64 Years,64.0,Male,Male,EK__[20210102111059]_YERRABELLI BAPURAO____[]_...,2021-01-03,76.9,Normal,100.00,True,True
4,KYATHARAJU ASHOK,KYATHARAJU ASHOK,35 Years,35.0,Male,Male,EK__[20210102114455]_KYATHARAJU ASHOK____[]_.pdf,2021-01-02,75.5,Normal,100.00,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1442,DIDDI VAJRAMMA,DIDDI VAJRAMMA,85 Years,85.0,Female,Female,EK__[20211230163917]_DIDDI VAJRAMMA____[]_.pdf,2021-12-30,52.6,Mild,100.00,True,True
1443,ARRA RAMCHANDAR,ARRA RAMCHANDER,63 Years,63.0,Male,Male,EK__[20211231114421]_ARRA RAMCHANDAR____[]_.pdf,2021-12-31,56.9,Normal,93.33,True,True
1444,NEERUGONGA CHANDRAKALA,CHANDRAKALA NEEERUBONDA,38 Years,38.0,Female,Female,EK__[20211231131117]_NEERUGONGA CHANDRAKALA___...,2021-12-31,75.0,Normal,88.89,True,True
1445,NEERUGONDA CHANDRAKALA,CHANDRAKALA NEEERUBONDA,38 Years,38.0,Female,Female,EK__[20211231142128]_NEERUGONDA CHANDRAKALA___...,2021-12-31,75.0,Normal,93.33,True,True


,ECG name,ECG age,ECG sex,Best Echo match,Echo age,Echo gender,Match score,Reason
0,KALLURI VARALAXMI,70 Years,Female,KALLURI VARALAKSHMI,75.0,Female,88.89,Age mismatch
1,BUKYA VEERANNA,28 Years,Male,BHUKYA ANJALI,26.0,Female,66.67,"Name mismatch, Age mismatch, Sex/Gender mismatch"
2,CHINTHA THIRUPATHI,38 Years,Male,POTHU TIRUPATHI,33.0,Male,72.73,"Name mismatch, Age mismatch"
3,AKUNURI BHADRAIAH,43 Years,Male,AKUNURI VENKATAIAH,60.0,Male,74.29,"Name mismatch, Age mismatch"
4,BOINAPELLI ANJANIDEVI,60 Years,Female,BONALA UMA DEVI,43.0,Female,66.67,"Name mismatch, Age mismatch"
...,...,...,...,...,...,...,...,...
3999,VARANASI PADMAJA DEVI,56 Years,Male,VEMURI PADMAJA DEVI,69.0,Female,80.00,"Name mismatch, Age mismatch, Sex/Gender mismatch"
4000,CHEVULA ANDALAMMA,66 Years,Female,CHAGANTI ILAMMA,45.0,Female,66.67,"Name mismatch, Age mismatch"
4001,CHEVULA ANDALAMMA,66 Years,Female,CHAGANTI ILAMMA,45.0,Female,66.67,"Name mismatch, Age mismatch"
4002,CHEVULA LAXMINARASAIAH,75 Years,Male,VELPULA NARSAIAH,55.0,Male,68.42,"Name mismatch, Age mismatch"


In [16]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"
    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()
    match = re.search(r"\d+", age)

    if match:
        return int(match.group())

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "echo_name": h[echo_name_col],
                "ECG age": e[ecg_age_col],
                "Echo age": h[echo_age_col],
                "ECG sex": e[ecg_sex_col],
                "Echo gender": h[echo_sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": round(score, 2),
                "age_match": True,
                "sex_match": True
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG sex": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["ECG sex"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "age",
            "gender",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("\nMatched ECG-Echo records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\98564987.py:126: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(


KeyError: "['age', 'gender'] not in index"

In [18]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"
    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()
    match = re.search(r"\d+", age)

    if match:
        return int(match.group())

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "echo_name": h[echo_name_col],
                "ECG age": e[ecg_age_col],
                "Echo age": h[echo_age_col],
                "ECG sex": e[ecg_sex_col],
                "Echo gender": h[echo_sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": round(score, 2),
                "age_match": True,
                "sex_match": True
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG sex": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["ECG sex"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "ECG age",
            "ECG sex",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

    result = result.rename(columns={
        "ECG age": "age",
        "ECG sex": "gender"
    })

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

print("Matched records:", len(result))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\3895710819.py:126: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(


Matched records: 1447
Saved file: matched_ecg_echo_20261028_165504.xlsx


,name,age,gender,file_name,EF (Teich) %,LV
0,THALAPELLI VINOD,28 Years,Male,EK__[20210101153258]_THALAPELLI VINOD____[]_.pdf,71.8,Normal
1,MD JAMEELAHMED,52 Years,Male,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,36.2,Moderate
2,EDULA SRINIVAS,60 Years,Male,EK__[20210102105955]_EDULA SRINIVAS____[]_.pdf,74.9,Normal
3,YERRABELLI BAPURAO,64 Years,Male,EK__[20210102111059]_YERRABELLI BAPURAO____[]_...,76.9,Normal
4,KYATHARAJU ASHOK,35 Years,Male,EK__[20210102114455]_KYATHARAJU ASHOK____[]_.pdf,75.5,Normal
...,...,...,...,...,...,...
1442,DIDDI VAJRAMMA,85 Years,Female,EK__[20211230163917]_DIDDI VAJRAMMA____[]_.pdf,52.6,Mild
1443,ARRA RAMCHANDAR,63 Years,Male,EK__[20211231114421]_ARRA RAMCHANDAR____[]_.pdf,56.9,Normal
1444,NEERUGONGA CHANDRAKALA,38 Years,Female,EK__[20211231131117]_NEERUGONGA CHANDRAKALA___...,75.0,Normal
1445,NEERUGONDA CHANDRAKALA,38 Years,Female,EK__[20211231142128]_NEERUGONDA CHANDRAKALA___...,75.0,Normal


In [21]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG_2022.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2022echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"
    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()
    match = re.search(r"\d+", age)

    if match:
        return int(match.group())

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "echo_name": h[echo_name_col],
                "ECG age": e[ecg_age_col],
                "Echo age": h[echo_age_col],
                "ECG sex": e[ecg_sex_col],
                "Echo gender": h[echo_sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": round(score, 2),
                "age_match": True,
                "sex_match": True
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG sex": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["ECG sex"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "ECG age",
            "ECG sex",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

    result = result.rename(columns={
        "ECG age": "age",
        "ECG sex": "gender"
    })

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

print("Matched records:", len(result))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'SEX', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'STUDY_DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'MV_EA', 'LA', 'EF_TEICH_%', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EF_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'EF_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA_AO_RATIO', 'EF_CUBED_%', 'FILE_NAME', 'FILE_PATH', 'FOLDER']


ValueError: Missing required columns: ['Echo age']. Check the printed column lists above.

In [19]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("2020_Ecg1.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2020_data.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"
    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()
    match = re.search(r"\d+", age)

    if match:
        return int(match.group())

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)","EF (Teich)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "echo_name": h[echo_name_col],
                "ECG age": e[ecg_age_col],
                "Echo age": h[echo_age_col],
                "ECG sex": e[ecg_sex_col],
                "Echo gender": h[echo_sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": round(score, 2),
                "age_match": True,
                "sex_match": True
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG sex": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["ECG sex"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "ECG age",
            "ECG sex",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

    result = result.rename(columns={
        "ECG age": "age",
        "ECG sex": "gender"
    })

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

print("Matched records:", len(result))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'SEX', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'GENDER', 'DATE', 'INTERPRETATION COMMENTS', 'EF (TEICH)']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\1546805907.py:126: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(


Matched records: 272
Saved file: matched_ecg_echo_20261028_170005.xlsx


,name,age,gender,file_name,EF (Teich) %,LV
0,NADENDLA KANAKAMANIKYAM,70 Years,Female,EK__[20200820125144]_NADENDLA KANAKAMANIKYAM__...,67.7,Normal
1,BOMMAKANTI BHASKARA SWAMY,71 Years,Male,EK__[20200820160436]_BOMMAKANTI BHASKARA SWAMY...,77.6,Normal
2,BOMMAKANTI BHASKARA SWAMY,71 Years,Male,EK__[20200820160436]_BOMMAKANTI BHASKARA SWAMY...,77.6,Normal
3,BOMMAKANTI BHASKARA SWAMY,71 Years,Male,EK__[20200820160436]_BOMMAKANTI BHASKARA SWAMY...,77.6,Normal
4,AHMED PASHA,34 Years,Male,EK__[20200820115030]_AHMED PASHA____[]_.pdf,74.2,Normal
...,...,...,...,...,...,...
267,THANDA PADMA,55 Years,Female,EK__[20201230131043]_THANDA PADMA____[]_.pdf,80.5,Normal
268,SUNKARI RADHA,65 Years,Female,EK__[20201230151945]_SUNKARI RADHA____[]_.pdf,31.6,Moderate
269,SAMAKURI MAHESH,26 Years,Male,EK__[20201230154435]_SAMAKURI MAHESH____[]_.pdf,67.0,Normal
270,MANDALA HEMANTHKUMAR,48 Years,Male,EK__[20201231122734]_MANDALA HEMANTHKUMAR____[...,80.1,Normal


In [22]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("2020_Ecg1.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2020_data.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"
    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()
    match = re.search(r"\d+", age)

    if match:
        return int(match.group())

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)","EF (Teich)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "echo_name": h[echo_name_col],
                "ECG age": e[ecg_age_col],
                "Echo age": h[echo_age_col],
                "ECG sex": e[ecg_sex_col],
                "Echo gender": h[echo_sex_col],
                "file_name": e[file_col],
                "echo_study_date": h["echo_study_date"],
                "EF (Teich) %": h[ef_col],
                "match_score": round(score, 2),
                "age_match": True,
                "sex_match": True
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG sex": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["ECG sex"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "ECG age",
            "ECG sex",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

    result = result.rename(columns={
        "ECG age": "age",
        "ECG sex": "gender"
    })

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

print("Matched records:", len(result))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'SEX', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'GENDER', 'DATE', 'INTERPRETATION COMMENTS', 'EF (TEICH)']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\1546805907.py:126: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(


Matched records: 272
Saved file: matched_ecg_echo_20261028_171101.xlsx


,name,age,gender,file_name,EF (Teich) %,LV
0,NADENDLA KANAKAMANIKYAM,70 Years,Female,EK__[20200820125144]_NADENDLA KANAKAMANIKYAM__...,67.7,Normal
1,BOMMAKANTI BHASKARA SWAMY,71 Years,Male,EK__[20200820160436]_BOMMAKANTI BHASKARA SWAMY...,77.6,Normal
2,BOMMAKANTI BHASKARA SWAMY,71 Years,Male,EK__[20200820160436]_BOMMAKANTI BHASKARA SWAMY...,77.6,Normal
3,BOMMAKANTI BHASKARA SWAMY,71 Years,Male,EK__[20200820160436]_BOMMAKANTI BHASKARA SWAMY...,77.6,Normal
4,AHMED PASHA,34 Years,Male,EK__[20200820115030]_AHMED PASHA____[]_.pdf,74.2,Normal
...,...,...,...,...,...,...
267,THANDA PADMA,55 Years,Female,EK__[20201230131043]_THANDA PADMA____[]_.pdf,80.5,Normal
268,SUNKARI RADHA,65 Years,Female,EK__[20201230151945]_SUNKARI RADHA____[]_.pdf,31.6,Moderate
269,SAMAKURI MAHESH,26 Years,Male,EK__[20201230154435]_SAMAKURI MAHESH____[]_.pdf,67.0,Normal
270,MANDALA HEMANTHKUMAR,48 Years,Male,EK__[20201231122734]_MANDALA HEMANTHKUMAR____[...,80.1,Normal


In [20]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("2020_Ecg1.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2020_data.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""

    name = str(name).upper()
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    name = " ".join(name.split())

    name = re.sub(r"^SALIVA\s+", "", name)

    return name

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"

    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    age = str(age).strip().upper()

    match = re.fullmatch(r"\d+(?:\.0+)?", age)

    if match:
        return int(float(age))

    return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

echo_age_col = next(
    (c for c in ["AGE", "PATIENT AGE"] if c in echo.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY DATE", "STUDY_DATE", "DATE"] if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%"] if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "Echo age": echo_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)
echo["age_key"] = echo[echo_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "age_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        echo_name = h["name_key"]
        echo_compact = h["compact_key"]

        score = max(
            fuzz.ratio(ecg_name, echo_name),
            fuzz.ratio(ecg_compact, echo_compact),
            fuzz.token_sort_ratio(ecg_name, echo_name),
            fuzz.partial_ratio(ecg_name, echo_name),
            fuzz.partial_ratio(ecg_compact, echo_compact)
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if ecg_age is None or h["age_key"] is None:
            continue

        if ecg_age != h["age_key"]:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "age": e[ecg_age_col],
                "gender": e[ecg_sex_col],
                "file_name": e[file_col],
                "EF (Teich) %": h[ef_col],
                "gender_key": ecg_sex,
                "match_score": round(score, 2)
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if ecg_age is None or best_record["age_key"] is None:
                reason.append("Age missing")
            elif ecg_age != best_record["age_key"]:
                reason.append("Age mismatch")

            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG gender": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo age": (
                best_record[echo_age_col]
                if best_record is not None else None
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = row["gender_key"]

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"

    if ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "age",
            "gender",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("\nUnique ECG patients:", ecg["name_key"].nunique())
print("Unique Echo patients:", echo["name_key"].nunique())
print("Matched ECG-Echo records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'SEX', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'GENDER', 'DATE', 'INTERPRETATION COMMENTS', 'EF (TEICH)']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\2290736110.py:134: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(



Unique ECG patients: 1947
Unique Echo patients: 1108
Matched ECG-Echo records: 0
Unmatched ECG records: 2139
Saved file: matched_ecg_echo_20261028_170528.xlsx


""


In [26]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("2020_Ecg1.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2020_data.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"^SALIVA\s+", "", name)
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"

    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    try:
        age = float(str(age).strip())

        if age < 0:
            return None

        return age
    except (ValueError, TypeError):
        return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY_DATE", "STUDY DATE", "DATE"]
     if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF_TEICH_%", "EF_TEICH", "EF (TEICH)"]
     if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the printed column lists above."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_age = e["age_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = 0

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"]),
            fuzz.partial_ratio(ecg_name, h["name_key"]),
            fuzz.partial_ratio(ecg_compact, h["compact_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        candidates.sort(key=lambda x: x[1], reverse=True)

        for h, score in candidates:
            matches.append({
                "name": e[ecg_name_col],
                "age": e[ecg_age_col],
                "gender": e[ecg_sex_col],
                "file_name": e[file_col],
                "EF (Teich) %": h[ef_col],
                "LV": "",
                "echo_name": h[echo_name_col],
                "echo_study_date": h["echo_study_date"],
                "match_score": round(score, 2)
            })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG gender": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2),
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["gender"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "age",
            "gender",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)

ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'SEX', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'GENDER', 'DATE', 'INTERPRETATION COMMENTS', 'EF (TEICH)']


C:\Users\Admin\AppData\Local\Temp\ipykernel_2432\3908302160.py:127: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_study_date"] = pd.to_datetime(


Matched records: 4529
Unmatched ECG records: 643
Saved file: matched_ecg_echo_20261028_172634.xlsx


,name,age,gender,file_name,EF (Teich) %,LV
0,VARAYOGULA VIJAYA,47 Years,Female,EK__[20200819150355]_VARAYOGULA VIJAYA____[]_.pdf,53.3,Mild
1,VARAYOGULA VIJAYA,47 Years,Female,EK__[20200819150355]_VARAYOGULA VIJAYA____[]_.pdf,62.9,Normal
2,VARAYOGULA VIJAYA,47 Years,Female,EK__[20200819164059]_VARAYOGULA VIJAYA____[]_.pdf,53.3,Mild
3,VARAYOGULA VIJAYA,47 Years,Female,EK__[20200819164059]_VARAYOGULA VIJAYA____[]_.pdf,62.9,Normal
4,RUDROJU GANESH,43 Years,Male,EK__[20200819164943]_RUDROJU GANESH____[]_.pdf,56.5,Normal
...,...,...,...,...,...,...
4524,NARRA SRINIVAS,42 Years,Male,EK__[20201231123550]_NARRA SRINIVAS____[]_.pdf,68.3,Normal
4525,SREEPATHI PRAMEELA,52 Years,Female,EK__[20201231160726]_SREEPATHI PRAMEELA____[]_...,62.4,Normal
4526,GANDLA VEERASWAMY,45 Years,Male,EK__[20201231110110]_GANDLA VEERASWAMY____[]_.pdf,30.6,Severe
4527,SRIRAMULA SAMMAKKA,72 Years,Female,EK__[20201231124243]_SRIRAMULA SAMMAKKA____[]_...,68.8,Normal


In [ ]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize_columns(ecg)
echo = standardize_columns(echo)

print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper()
    name = re.sub(r"^SALIVA\s+", "", name)
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())

def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))

def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"
    if sex in ["F", "FEMALE"]:
        return "F"

    return sex

def clean_age(age):
    if pd.isna(age):
        return None

    try:
        age = float(str(age).strip())
        if age < 0:
            return None
        return age
    except (ValueError, TypeError):
        return None

ecg_name_col = next(
    (c for c in ["NAME"] if c in ecg.columns), None
)

echo_name_col = next(
    (c for c in ["NAME"] if c in echo.columns), None
)

ecg_age_col = next(
    (c for c in ["AGE"] if c in ecg.columns), None
)

ecg_sex_col = next(
    (c for c in ["SEX", "GENDER"] if c in ecg.columns), None
)

echo_sex_col = next(
    (c for c in ["GENDER", "SEX"] if c in echo.columns), None
)

file_col = next(
    (c for c in ["FILE NAME", "FILENAME"] if c in ecg.columns), None
)

echo_date_col = next(
    (c for c in ["STUDY_DATE", "STUDY DATE", "DATE"]
     if c in echo.columns),
    None
)

ef_col = next(
    (c for c in ["EF_TEICH_%", "EF_TEICH", "EF (TEICH)"]
     if c in echo.columns),
    None
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG age": ecg_age_col,
    "ECG sex/gender": ecg_sex_col,
    "Echo sex/gender": echo_sex_col,
    "ECG file name": file_col,
    "Echo study date": echo_date_col,
    "Echo EF (Teich)": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"ECG columns: {ecg.columns.tolist()}. "
        f"Echo columns: {echo.columns.tolist()}."
    )

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = ecg[ecg_age_col].apply(clean_age)

ecg["sex_key"] = ecg[ecg_sex_col].apply(clean_sex)
echo["sex_key"] = echo[echo_sex_col].apply(clean_sex)

echo["echo_study_date"] = pd.to_datetime(
    echo[echo_date_col], errors="coerce"
).dt.normalize()

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 <= x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy()
echo = echo[echo["name_key"] != ""].copy()

echo = echo.drop_duplicates(
    subset=["name_key", "sex_key", "echo_study_date"],
    keep="first"
)

echo_records = echo.to_dict("records")

matches = []
unmatched = []

for _, e in ecg.iterrows():
    ecg_name = e["name_key"]
    ecg_compact = e["compact_key"]
    ecg_sex = e["sex_key"]

    candidates = []
    best_record = None
    best_score = -1

    for h in echo_records:
        score = max(
            fuzz.ratio(ecg_name, h["name_key"]),
            fuzz.ratio(ecg_compact, h["compact_key"]),
            fuzz.token_sort_ratio(ecg_name, h["name_key"]),
            fuzz.partial_ratio(ecg_name, h["name_key"]),
            fuzz.partial_ratio(ecg_compact, h["compact_key"])
        )

        if score > best_score:
            best_score = score
            best_record = h

        if score < 85:
            continue

        if not ecg_sex or not h["sex_key"]:
            continue

        if ecg_sex != h["sex_key"]:
            continue

        candidates.append((h, score))

    if candidates:
        candidates.sort(key=lambda x: x[1], reverse=True)
        h, score = candidates[0]

        matches.append({
            "name": e[ecg_name_col],
            "age": e[ecg_age_col],
            "gender": e[ecg_sex_col],
            "file_name": e[file_col],
            "EF (Teich) %": h[ef_col],
            "LV": "",
            "echo_name": h[echo_name_col],
            "echo_study_date": h["echo_study_date"],
            "match_score": round(score, 2)
        })

    else:
        reason = []

        if best_record is None or best_score < 85:
            reason.append("Name mismatch")

        if best_record is not None:
            if not ecg_sex or not best_record["sex_key"]:
                reason.append("Sex/Gender missing")
            elif ecg_sex != best_record["sex_key"]:
                reason.append("Sex/Gender mismatch")

        unmatched.append({
            "ECG name": e[ecg_name_col],
            "ECG age": e[ecg_age_col],
            "ECG gender": e[ecg_sex_col],
            "Best Echo match": (
                best_record[echo_name_col]
                if best_record is not None else ""
            ),
            "Echo gender": (
                best_record[echo_sex_col]
                if best_record is not None else ""
            ),
            "Match score": round(best_score, 2) if best_score >= 0 else 0,
            "Reason": ", ".join(reason) if reason else "No valid match"
        })

result = pd.DataFrame(matches)

def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["gender"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        elif ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    elif ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "name",
            "age",
            "gender",
            "file_name",
            "EF (Teich) %",
            "LV"
        ]
    ]

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Saved file:", filename)

display(result)


In [4]:

import pandas as pd
import numpy as np
import re
from datetime import datetime
from rapidfuzz import fuzz

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")

def standardize(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df

ecg = standardize(ecg)
echo = standardize(echo)

print("ECG rows:", len(ecg))
print("Echo rows:", len(echo))
print("ECG columns:", ecg.columns.tolist())
print("Echo columns:", echo.columns.tolist())

def find_col(df, names):
    return next((c for c in names if c in df.columns), None)

def clean_name(x):
    if pd.isna(x):
        return ""
    x = str(x).upper().strip()
    x = re.sub(r"^SALIVA[\s_-]*", "", x)
    x = re.sub(r"[^A-Z0-9\s]", " ", x)
    return " ".join(x.split())

def compact_name(x):
    return re.sub(r"[^A-Z0-9]", "", clean_name(x))

def clean_sex(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    if x in ["M", "MALE", "MAN"]:
        return "M"
    if x in ["F", "FEMALE", "WOMAN"]:
        return "F"
    return x

def clean_age(x):
    if pd.isna(x):
        return None
    try:
        return float(str(x).strip())
    except (ValueError, TypeError):
        return None

en = find_col(ecg, ["NAME", "PATIENT NAME", "PATIENT_NAME"])
ea = find_col(ecg, ["AGE", "PATIENT AGE", "PATIENT_AGE"])
eg = find_col(ecg, ["SEX", "GENDER"])
ed = find_col(ecg, ["DATE", "ECG DATE", "ECG_DATE"])
ef = find_col(ecg, ["FILE NAME", "FILE_NAME", "FILENAME"])

hn = find_col(echo, ["NAME", "PATIENT NAME", "PATIENT_NAME"])
ha = find_col(echo, ["AGE", "PATIENT AGE", "PATIENT_AGE"])
hg = find_col(echo, ["GENDER", "SEX"])
hd = find_col(echo, ["STUDY_DATE", "STUDY DATE", "DATE"])
he = find_col(echo, ["EF_TEICH_%", "EF_TEICH", "EF (TEICH)"])

required = {
    "ECG name": en,
    "ECG age": ea,
    "ECG gender": eg,
    "ECG date": ed,
    "ECG file name": ef,
    "Echo name": hn,
    "Echo gender": hg,
    "Echo date": hd,
    "Echo EF": he
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )

ecg["_name"] = ecg[en].apply(clean_name)
ecg["_compact"] = ecg[en].apply(compact_name)
ecg["_age"] = ecg[ea].apply(clean_age)
ecg["_sex"] = ecg[eg].apply(clean_sex)
ecg["_date"] = pd.to_datetime(ecg[ed], errors="coerce").dt.normalize()

echo["_name"] = echo[hn].apply(clean_name)
echo["_compact"] = echo[hn].apply(compact_name)
echo["_sex"] = echo[hg].apply(clean_sex)
echo["_date"] = pd.to_datetime(echo[hd], errors="coerce").dt.normalize()

if ha is not None:
    echo["_age"] = echo[ha].apply(clean_age)

echo["_ef"] = pd.to_numeric(echo[he], errors="coerce")
echo.loc[
    echo["_ef"].between(0, 1, inclusive="both"), "_ef"
] *= 100

ecg = ecg[ecg["_name"] != ""].copy()
echo = echo[echo["_name"] != ""].copy()

print("\nUsable ECG records:", len(ecg))
print("Usable Echo records:", len(echo))
print("Echo age column:", ha)

def name_score(a, b, ac, bc):
    if not a or not b:
        return 0

    return max(
        fuzz.WRatio(a, b),
        fuzz.token_sort_ratio(a, b),
        fuzz.ratio(ac, bc)
    )

matches = []
unmatched = []

for _, e in ecg.iterrows():
    all_candidates = echo.copy()

    all_candidates["_score"] = [
        name_score(e["_name"], r["_name"], e["_compact"], r["_compact"])
        for _, r in all_candidates.iterrows()
    ]

    if all_candidates.empty:
        continue

    best_name = all_candidates.sort_values(
        "_score", ascending=False
    ).iloc[0]

    candidates = all_candidates[all_candidates["_score"] >= 80].copy()

    if candidates.empty:
        unmatched.append({
            "Name (from ECG)": e[en],
            "Age (from ECG)": e[ea],
            "Gender (from ECG)": e[eg],
            "File_name (from ECG)": e[ef],
            "Best Echo name": best_name[hn],
            "Best name score": int(best_name["_score"]),
            "Reason": "Name similarity below 80%"
        })
        continue

    if ha is not None:
        if e["_age"] is None:
            unmatched.append({
                "Name (from ECG)": e[en],
                "Age (from ECG)": e[ea],
                "Gender (from ECG)": e[eg],
                "File_name (from ECG)": e[ef],
                "Best Echo name": best_name[hn],
                "Best name score": int(best_name["_score"]),
                "Reason": "ECG age missing; cannot verify age"
            })
            continue

        candidates = candidates[
            candidates["_age"].notna() &
            (candidates["_age"] == e["_age"])
        ]

        if candidates.empty:
            unmatched.append({
                "Name (from ECG)": e[en],
                "Age (from ECG)": e[ea],
                "Gender (from ECG)": e[eg],
                "File_name (from ECG)": e[ef],
                "Best Echo name": best_name[hn],
                "Best name score": int(best_name["_score"]),
                "Reason": "Name matched, but age did not match"
            })
            continue

    if e["_sex"]:
        candidates = candidates[candidates["_sex"] == e["_sex"]]
    else:
        unmatched.append({
            "Name (from ECG)": e[en],
            "Age (from ECG)": e[ea],
            "Gender (from ECG)": e[eg],
            "File_name (from ECG)": e[ef],
            "Best Echo name": best_name[hn],
            "Best name score": int(best_name["_score"]),
            "Reason": "ECG gender missing"
        })
        continue

    if candidates.empty:
        unmatched.append({
            "Name (from ECG)": e[en],
            "Age (from ECG)": e[ea],
            "Gender (from ECG)": e[eg],
            "File_name (from ECG)": e[ef],
            "Best Echo name": best_name[hn],
            "Best name score": int(best_name["_score"]),
            "Reason": "Name matched, but gender did not match"
        })
        continue

    if pd.notna(e["_date"]):
        candidates["_date_distance"] = (
            candidates["_date"] - e["_date"]
        ).abs().dt.days

        candidates["_date_distance"] = candidates[
            "_date_distance"
        ].fillna(10**9)

        best = candidates.sort_values(
            ["_date_distance", "_score"],
            ascending=[True, False]
        ).iloc[0]
    else:
        best = candidates.sort_values(
            "_score", ascending=False
        ).iloc[0]

    matches.append({
        "Name (from ECG)": e[en],
        "Age (from ECG)": e[ea],
        "Gender (from ECG)": e[eg],
        "File_name (from ECG)": e[ef],
        "EF (Teich) % (from Echo)": best["_ef"],
        "LV (according to calculations)": ""
    })

result = pd.DataFrame(matches, columns=[
    "Name (from ECG)",
    "Age (from ECG)",
    "Gender (from ECG)",
    "File_name (from ECG)",
    "EF (Teich) % (from Echo)",
    "LV (according to calculations)"
])

def classify_lv(row):
    ef = row["EF (Teich) % (from Echo)"]
    sex = clean_sex(row["Gender (from ECG)"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        if ef >= 41:
            return "Mild"

    if sex == "F":
        if ef >= 54:
            return "Normal"
        if ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"
    if ef < 31:
        return "Severe"

    return "Unknown"

if not result.empty:
    result["LV (according to calculations)"] = result.apply(
        classify_lv, axis=1
    )

unmatched_df = pd.DataFrame(unmatched)

filename = "matched_ecg_echo_" + datetime.now().strftime(
    "%Y%m%d_%H%M%S"
) + ".xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_df.to_excel(writer, sheet_name="Unmatched ECG", index=False)

print("\nMatched records:", len(result))
print("Unmatched records:", len(unmatched_df))
print("Output file:", filename)

if not unmatched_df.empty:
    print("\nReasons for unmatched records:")
    print(unmatched_df["Reason"].value_counts())

    print("\nFirst 20 unmatched records:")
    display(unmatched_df.head(20))

display(result)


ECG rows: 5405
Echo rows: 2405
ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']

Usable ECG records: 5405
Usable Echo records: 2127
Echo age column: AGE


C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\3897766933.py:99: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["_date"] = pd.to_datetime(echo[hd], errors="coerce").dt.normalize()



Matched records: 0
Unmatched records: 5405
Output file: matched_ecg_echo_20261029_151447.xlsx

Reasons for unmatched records:
Reason
ECG age missing; cannot verify age    3484
Name similarity below 80%             1921
Name: count, dtype: int64

First 20 unmatched records:


,Name (from ECG),Age (from ECG),Gender (from ECG),File_name (from ECG),Best Echo name,Best name score,Reason
0,KALLURI VARALAXMI,70 Years,Female,EK__[20210101110036]_KALLURI VARALAXMI____[]_.pdf,KALLURI VARALAKSHMI,88,ECG age missing; cannot verify age
1,BUKYA VEERANNA,28 Years,Male,EK__[20210101121417]_BUKYA VEERANNA____[]_.pdf,VEERAMMA,67,Name similarity below 80%
2,THALAPELLI VINOD,28 Years,Male,EK__[20210101153258]_THALAPELLI VINOD____[]_.pdf,THALLAPALLI VINOD,90,ECG age missing; cannot verify age
3,CHINTHA THIRUPATHI,38 Years,Male,EK__[20210101155310]_CHINTHA THIRUPATHI____[]_...,POTHU TIRUPATHI,72,Name similarity below 80%
4,AKUNURI BHADRAIAH,43 Years,Male,EK__[20210101190531]_AKUNURI BHADRAIAH____[]_.pdf,AKUNURI VENKATAIAH,74,Name similarity below 80%
5,MD JAMEELAHMED,52 Years,Male,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,JAMEEL AHMED,91,ECG age missing; cannot verify age
6,BOINAPELLI ANJANIDEVI,60 Years,Female,EK__[20210102104925]_BOINAPELLI ANJANIDEVI____...,BONALA UMA DEVI,66,Name similarity below 80%
7,EDULA SRINIVAS,60 Years,Male,EK__[20210102105955]_EDULA SRINIVAS____[]_.pdf,SRINIVAS EDULA,100,ECG age missing; cannot verify age
8,YERRAMREDDY VIJAYA,62 Years,Female,EK__[20210102110246]_YERRAMREDDY VIJAYA____[]_...,GUDA VIJAYA,85,ECG age missing; cannot verify age
9,YERRABELLI BAPURAO,64 Years,Male,EK__[20210102111059]_YERRABELLI BAPURAO____[]_...,YERRABELLI BAPU RAO,100,ECG age missing; cannot verify age


,Name (from ECG),Age (from ECG),Gender (from ECG),File_name (from ECG),EF (Teich) % (from Echo),LV (according to calculations)


ECG columns: ['NAME', 'DATE', 'TIME', 'AGE', 'GENDER', 'RATE', 'PR', 'RR', 'QRSD', 'QT', 'QTCB', 'QTCF', 'P AXIS', 'QRS AXIS', 'T AXIS', 'ECG INTERPRETATION', 'ECG_COMMENT', 'FILE NAME', 'PDF_PATH']
Echo columns: ['NAME', 'AGE', 'STUDY DATE', 'MRN', 'GENDER', 'INTERPRETATION_COMMENTS', 'IVSD', 'IVSS', 'LVIDD', 'LVIDS', 'LVPWD', 'LVPWS', 'FS', 'EDV_TEICH', 'ESV_TEICH', 'EDV_CUBED', 'ESV_CUBED', 'LV_MASS', 'SV_TEICH', 'SV_CUBED', 'AOROOT', 'LA', 'LA_AO_RATIO', 'MV_EA', 'EF_TEICH_%', 'EF_CUBED_%', 'FILE_NAME', 'FOLDER', 'FILE_PATH']


C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\2407922455.py:177: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  echo["echo_date"] = pd.to_datetime(


KeyboardInterrupt: 

In [7]:
import pandas as pd
import re
from datetime import datetime
from itertools import combinations
from rapidfuzz import fuzz, process

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")


def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""

    name = str(name).upper().strip()
    name = re.sub(r"^SALIVA\s+", "", name)
    name = re.sub(r"[^A-Z0-9\s]", " ", name)

    return " ".join(name.split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None

    match = re.search(r"\d+", str(age))

    return int(match.group()) if match else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ["M", "MALE"]:
        return "M"

    if sex in ["F", "FEMALE"]:
        return "F"

    return ""


def parse_date(series):
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def date_from_filename(filename):
    if pd.isna(filename):
        return pd.NaT

    match = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if match:
        return pd.to_datetime(
            match.group(1),
            format="%Y%m%d",
            errors="coerce"
        )

    return pd.NaT


def get_name_score(name1, name2):
    n1 = clean_name(name1)
    n2 = clean_name(name2)

    c1 = compact_name(n1)
    c2 = compact_name(n2)

    if not c1 or not c2:
        return 0

    if n1 == n2 or c1 == c2:
        return 100

    scores = [
        fuzz.ratio(n1, n2),
        fuzz.ratio(c1, c2),
        fuzz.token_sort_ratio(n1, n2)
    ]

    t1 = n1.split()
    t2 = n2.split()

    # Handle concatenated names such as SUMAKEERTHI.
    # Try splitting the concatenated name into the same
    # number of parts as the other name.

    def split_variants(compact, count):
        if count < 2 or len(compact) < count:
            return []

        if count > 5:
            return []

        variants = []

        for cuts in combinations(
            range(1, len(compact)),
            count - 1
        ):
            points = (0,) + cuts + (len(compact),)

            parts = [
                compact[points[i]:points[i + 1]]
                for i in range(count)
            ]

            variants.append(" ".join(parts))

        return variants

    for variant in split_variants(c1, len(t2)):
        scores.append(fuzz.token_sort_ratio(variant, n2))

    for variant in split_variants(c2, len(t1)):
        scores.append(fuzz.token_sort_ratio(n1, variant))

    return max(scores)


ecg_name_col = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name_col = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age_col = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age_col = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex_col = find_col(ecg, ["SEX", "GENDER"])
echo_sex_col = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(
    ecg,
    ["FILE NAME", "FILENAME", "FILE"]
)

ecg_date_col = find_col(
    ecg,
    ["STUDY DATE", "ECG DATE", "RECORDING DATE", "DATE"]
)

echo_date_col = find_col(
    echo,
    ["STUDY DATE", "STUDY_DATE", "DATE"]
)

ef_col = find_col(
    echo,
    ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%", "EF (TEICH) %"]
)

required = {
    "ECG name": ecg_name_col,
    "Echo name": echo_name_col,
    "ECG file name": file_col,
    "Echo EF": ef_col
}

missing = [key for key, value in required.items() if value is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy()
echo = echo.copy()

ecg["original_name"] = ecg[ecg_name_col]
echo["original_name"] = echo[echo_name_col]

ecg["name_key"] = ecg[ecg_name_col].apply(clean_name)
echo["name_key"] = echo[echo_name_col].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name_col].apply(compact_name)
echo["compact_key"] = echo[echo_name_col].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age_col].apply(clean_age)
    if ecg_age_col else pd.Series(None, index=ecg.index)
)

echo["age_key"] = (
    echo[echo_age_col].apply(clean_age)
    if echo_age_col else pd.Series(None, index=echo.index)
)

ecg["sex_key"] = (
    ecg[ecg_sex_col].apply(clean_sex)
    if ecg_sex_col else ""
)

echo["sex_key"] = (
    echo[echo_sex_col].apply(clean_sex)
    if echo_sex_col else ""
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date_col])
    if ecg_date_col
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date_col])
    if echo_date_col
    else pd.Series(pd.NaT, index=echo.index)
)

# Extract the ECG date from the filename when its date is missing.
# Example: EK__[20210525105634]_PATIENT.pdf -> 2021-05-25

ecg["filename_date"] = ecg[file_col].apply(date_from_filename)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg["filename_date"]
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].copy().reset_index(drop=True)
echo = echo[echo["name_key"] != ""].copy().reset_index(drop=True)

echo_names = echo["name_key"].tolist()
echo_compact = echo["compact_key"].tolist()

pairs = []

for i, e in ecg.iterrows():

    name = e["name_key"]
    compact = e["compact_key"]

    candidates = set()

    # Fast candidate search using the original name.
    for query in [name, compact]:

        if not query:
            continue

        for choices in [echo_names, echo_compact]:

            found = process.extract(
                query,
                choices,
                scorer=fuzz.ratio,
                limit=50,
                score_cutoff=80
            )

            candidates.update(
                idx for _, _, idx in found
            )

    # Also find concatenated/reordered names.
    # For example, SUMAKEERTHI and KEERTHI SUMA.
    for j, h in echo.iterrows():

        if j in candidates:
            continue

        tokens = h["name_key"].split()

        if len(tokens) >= 2 and all(
            token in compact for token in tokens
        ):
            candidates.add(j)

    for j in candidates:

        h = echo.iloc[j]

        score = get_name_score(
            e["name_key"],
            h["name_key"]
        )

        if score < 80:
            continue

        # Reject conflicting ages when both are available.
        if (
            e["age_key"] is not None
            and h["age_key"] is not None
            and e["age_key"] != h["age_key"]
        ):
            continue

        # Reject conflicting genders when both are available.
        if (
            e["sex_key"]
            and h["sex_key"]
            and e["sex_key"] != h["sex_key"]
        ):
            continue

        ed = e["match_date"]
        hd = h["match_date"]

        # Reject different dates when both dates are available.
        if pd.notna(ed) and pd.notna(hd) and ed != hd:
            continue

        exact_date = int(
            pd.notna(ed) and pd.notna(hd) and ed == hd
        )

        age_match = int(
            e["age_key"] is not None
            and h["age_key"] is not None
            and e["age_key"] == h["age_key"]
        )

        sex_match = int(
            bool(e["sex_key"])
            and bool(h["sex_key"])
            and e["sex_key"] == h["sex_key"]
        )

        exact_name = int(score == 100)

        pairs.append(
            (
                exact_name,
                exact_date,
                score,
                age_match,
                sex_match,
                i,
                j
            )
        )


# Priority:
# 1. Exact name matches
# 2. Matching dates
# 3. Highest similarity
# 4. Matching age and gender

pairs.sort(
    key=lambda x: (
        x[0],
        x[1],
        x[2],
        x[3],
        x[4]
    ),
    reverse=True
)

used_ecg = set()
used_echo = set()
matches = []

for exact_name, exact_date, score, age_match, sex_match, i, j in pairs:

    if i in used_ecg or j in used_echo:
        continue

    e = ecg.iloc[i]
    h = echo.iloc[j]

    matches.append({
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": (
            e[ecg_age_col]
            if ecg_age_col
            else h[echo_age_col]
            if echo_age_col
            else None
        ),
        "Gender": (
            e[ecg_sex_col]
            if ecg_sex_col
            else h[echo_sex_col]
            if echo_sex_col
            else ""
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": h[ef_col],
        "Match score": round(score, 2)
    })

    used_ecg.add(i)
    used_echo.add(j)


result = pd.DataFrame(matches)


def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["Gender"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        if ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        if ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"

    if ef < 31:
        return "Severe"

    return "Unknown"


if not result.empty:

    result["LV"] = result.apply(classify_lv, axis=1)

    result = result[
        [
            "ECG name",
            "Echo name",
            "Age",
            "Gender",
            "ECG file name",
            "ECG date",
            "Echo study date",
            "EF (Teich) %",
            "LV",
            "Match score"
        ]
    ]


unmatched = []

for i, e in ecg.iterrows():

    if i in used_ecg:
        continue

    unmatched.append({
        "ECG name": e["original_name"],
        "ECG age": (
            e[ecg_age_col] if ecg_age_col else None
        ),
        "ECG gender": (
            e[ecg_sex_col] if ecg_sex_col else ""
        ),
        "ECG date": e["match_date"],
        "ECG file name": e[file_col],
        "Reason": "No valid unused Echo record matched"
    })


unmatched_df = pd.DataFrame(unmatched)

filename = (
    "matched_ecg_echo_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("ECG records:", len(ecg))
print("Echo records:", len(echo))
print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Unique Echo records used:", len(used_echo))
print("Saved file:", filename)

display(result)
display(unmatched_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\141083971.py:67: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(series, errors="coerce").dt.normalize()


KeyboardInterrupt: 

In [8]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")


def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"^SALIVA\s+", "", str(name).upper().strip())
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None
    m = re.search(r"\d+", str(age))
    return int(m.group()) if m else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()
    if sex in ("M", "MALE"):
        return "M"
    if sex in ("F", "FEMALE"):
        return "F"
    return ""


def parse_date(series):
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT
    m = re.search(r"\[(\d{8})\d{6}\]", str(filename))
    if not m:
        return pd.NaT
    return pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")


def name_score(a, b):
    a1, b1 = clean_name(a), clean_name(b)
    ac, bc = compact_name(a1), compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1)
    ]

    # Handle concatenated and reordered two-part names.
    # Example: SUMAKEERTHI versus KEERTHI SUMA.
    if not a1.count(" ") and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.token_sort_ratio(variant, b1))

    if not b1.count(" ") and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.token_sort_ratio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])
ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])
ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])
file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])

ecg_date = find_col(
    ecg, ["STUDY DATE", "ECG DATE", "RECORDING DATE", "DATE"]
)
echo_date = find_col(echo, ["STUDY DATE", "STUDY_DATE", "DATE"])
ef_col = find_col(
    echo, ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%", "EF (TEICH) %"]
)

required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )

ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

for df, col in [(ecg, ecg_name), (echo, echo_name)]:
    df["name_key"] = df[col].apply(clean_name)
    df["compact_key"] = df[col].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age) if ecg_age
    else pd.Series(None, index=ecg.index)
)
echo["age_key"] = (
    echo[echo_age].apply(clean_age) if echo_age
    else pd.Series(None, index=echo.index)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex) if ecg_sex
    else pd.Series("", index=ecg.index)
)
echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex) if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date]) if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)
echo["match_date"] = (
    parse_date(echo[echo_date]) if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")
echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

echo_names = echo["name_key"].tolist()
echo_compact = echo["compact_key"].tolist()

# Exact-name indexes avoid fuzzy searching when possible.
exact_index = {}
compact_index = {}

for j, h in echo.iterrows():
    exact_index.setdefault(h["name_key"], []).append(j)
    compact_index.setdefault(h["compact_key"], []).append(j)

pairs = []

for i, e in ecg.iterrows():
    name = e["name_key"]
    compact = e["compact_key"]
    candidates = set()

    # Retrieve exact matches first.
    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    # Retrieve likely reordered or misspelled matches.
    queries = [name]

    # Split concatenated names into two parts, not every combination.
    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            echo_names,
            scorer=fuzz.token_sort_ratio,
            limit=15,
            score_cutoff=80
        )
        candidates.update(idx for _, _, idx in found)

    # Validate only the candidates returned by the search.
    for j in candidates:
        h = echo.iloc[j]
        score = name_score(name, h["name_key"])

        if score < 80:
            continue

        # Reject conflicting ages.
        if (
            e["age_key"] is not None
            and h["age_key"] is not None
            and e["age_key"] != h["age_key"]
        ):
            continue

        # Reject conflicting genders.
        if (
            e["sex_key"]
            and h["sex_key"]
            and e["sex_key"] != h["sex_key"]
        ):
            continue

        ed, hd = e["match_date"], h["match_date"]

        # Dates must match whenever both are available.
        if pd.notna(ed) and pd.notna(hd) and ed != hd:
            continue

        exact = int(score == 100)
        date_match = int(pd.notna(ed) and pd.notna(hd) and ed == hd)
        age_match = int(
            e["age_key"] is not None
            and h["age_key"] is not None
            and e["age_key"] == h["age_key"]
        )
        sex_match = int(
            bool(e["sex_key"])
            and bool(h["sex_key"])
            and e["sex_key"] == h["sex_key"]
        )

        pairs.append(
            (exact, date_match, score, age_match, sex_match, i, j)
        )

# Exact names first, then dates, similarity, age and gender.
pairs.sort(key=lambda x: x[:5], reverse=True)

used_ecg = set()
used_echo = set()
matches = []

for exact, date_match, score, age_match, sex_match, i, j in pairs:
    if i in used_ecg or j in used_echo:
        continue

    e = ecg.iloc[i]
    h = echo.iloc[j]

    matches.append({
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": e[ecg_age] if ecg_age else (
            h[echo_age] if echo_age else None
        ),
        "Gender": e[ecg_sex] if ecg_sex else (
            h[echo_sex] if echo_sex else ""
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": h[ef_col],
        "Match score": round(score, 2)
    })

    used_ecg.add(i)
    used_echo.add(j)

result = pd.DataFrame(matches)


def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["Gender"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        if ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        if ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"

    if ef < 31:
        return "Severe"

    return "Unknown"


if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

unmatched = []

for i, e in ecg.iterrows():
    if i not in used_ecg:
        unmatched.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "No valid unused Echo match"
        })

unmatched_df = pd.DataFrame(unmatched)

filename = (
    "matched_ecg_echo_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_df.to_excel(writer, sheet_name="Unmatched ECG", index=False)

print("ECG records:", len(ecg))
print("Echo records:", len(echo))
print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_df))
print("Unique Echo records used:", len(used_echo))
print("Saved file:", filename)

display(result)
display(unmatched_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\1751441760.py:56: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(series, errors="coerce").dt.normalize()


ECG records: 5405
Echo records: 2127
Matched records: 490
Unmatched ECG records: 4915
Unique Echo records used: 490
Saved file: matched_ecg_echo_20261029_152923.xlsx


,ECG name,Echo name,Age,Gender,ECG file name,ECG date,Echo study date,EF (Teich) %,Match score,LV
0,CHINNAKKA VENKATAKRISHNAREDDY,CHINNAKKA VENKATAKRISHNA REDDY,42 Years,Male,EK__[20210128111159]_CHINNAKKA VENKATAKRISHNAR...,2021-01-28,2021-01-28,61.9,100.00,Normal
1,PALEPU ANUSHA,PALEPU ANUSHA,31 Years,Female,EK__[20210131112642]_PALEPU ANUSHA____[]_.pdf,2021-01-31,2021-01-31,73.3,100.00,Normal
2,THALLA KAVITHA,THALLA KAVITHA,32 Years,Female,EK__[20210131124414]_THALLA KAVITHA____[]_.pdf,2021-01-31,2021-01-31,68.5,100.00,Normal
3,RAYUDU SRINIVAS,RAYUDU SRINIVAS,50 Years,Male,EK__[20210225110105]_RAYUDU SRINIVAS____[]_.pdf,2021-02-25,2021-02-25,47.6,100.00,Mild
4,VOMA KOUSALYA,VOMA KOUSALYA,75 Years,Female,EK__[20210225173728]_VOMA KOUSALYA____[]_.pdf,2021-02-25,2021-02-25,62.0,100.00,Normal
...,...,...,...,...,...,...,...,...,...,...
485,PONUGOTI ASHOKRAO,ASHOK RAO PONUGOTI,65 Years,Male,EK__[20210228105120]_PONUGOTI ASHOKRAO____[]_.pdf,2021-02-28,2021-02-28,38.8,80.00,Moderate
486,BHUPELLY VIJAYA,VIJAYA BOMPELLI,43 Years,Female,EK__[20210417131659]_BHUPELLY VIJAYA____[]_.pdf,2021-04-17,2021-04-17,79.7,80.00,Normal
487,APSARABEE,AFSARI BEE.,70 Years,Female,EK__[20210529102147]_APSARABEE____[]_.pdf,2021-05-29,2021-05-29,71.2,80.00,Normal
488,A SADANANDAM,ALUGOJU SADANANDAM,66 Years,Male,EK__[20210726120623]_A SADANANDAM____[]_.pdf,2021-07-26,2021-07-26,65.1,80.00,Normal


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,KALLURI VARALAXMI,70 Years,Female,2021-01-01,EK__[20210101110036]_KALLURI VARALAXMI____[]_.pdf,No valid unused Echo match
1,BUKYA VEERANNA,28 Years,Male,2021-01-01,EK__[20210101121417]_BUKYA VEERANNA____[]_.pdf,No valid unused Echo match
2,CHINTHA THIRUPATHI,38 Years,Male,2021-01-01,EK__[20210101155310]_CHINTHA THIRUPATHI____[]_...,No valid unused Echo match
3,AKUNURI BHADRAIAH,43 Years,Male,2021-01-01,EK__[20210101190531]_AKUNURI BHADRAIAH____[]_.pdf,No valid unused Echo match
4,MD JAMEELAHMED,52 Years,Male,2021-02-01,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,No valid unused Echo match
...,...,...,...,...,...,...
4910,VARANASI PADMAJA DEVI,56 Years,Male,2021-12-31,EK__[20211231183317]_VARANASI PADMAJA DEVI____...,No valid unused Echo match
4911,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191647]_CHEVULA ANDALAMMA____[]_.pdf,No valid unused Echo match
4912,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191833]_CHEVULA ANDALAMMA____[]_.pdf,No valid unused Echo match
4913,CHEVULA LAXMINARASAIAH,75 Years,Male,2021-12-31,EK__[20211231192120]_CHEVULA LAXMINARASAIAH___...,No valid unused Echo match


In [13]:

import pandas as pd
import re
import os
from datetime import datetime
from rapidfuzz import fuzz

ecg_file = "ECG.xlsx"
echo_file = "2021echo_reports.xlsx"
output_file = "ECG_Echo_Matched.xlsx"

ecg = pd.read_excel(ecg_file, sheet_name="ECG Data")
echo = pd.read_excel(echo_file, sheet_name="Sheet1")


def clean_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = clean_columns(ecg)
echo = clean_columns(echo)


def get_col(df, names):
    for name in names:
        if name.upper() in df.columns:
            return name.upper()
    return None


ecg_name_col = get_col(ecg, ["NAME", "PATIENT NAME", "PATIENT_NAME"])
echo_name_col = get_col(echo, ["NAME", "PATIENT NAME", "PATIENT_NAME"])

ecg_age_col = get_col(ecg, ["AGE"])
echo_age_col = get_col(echo, ["AGE"])

ecg_gender_col = get_col(ecg, ["GENDER", "SEX"])
echo_gender_col = get_col(echo, ["GENDER", "SEX"])

ecg_date_col = get_col(
    ecg, ["DATE", "ECG DATE", "STUDY DATE", "TEST DATE", "EXAM DATE"]
)

echo_date_col = get_col(
    echo, ["STUDY DATE", "DATE", "ECHO DATE", "TEST DATE"]
)

ef_col = get_col(
    echo,
    [
        "EF_TEICH_%",
        "EF TEICH %",
        "EF_TEICH_%",
        "EF_TEICH",
        "EF",
        "EJECTION FRACTION"
    ]
)

filename_col = get_col(
    ecg,
    [
        "FILENAME",
        "FILE NAME",
        "FILE_NAME",
        "PDF",
        "PDF NAME",
        "FILE PATH",
        "PATH",
        "ECG FILE",
        "ECG FILENAME",
        "ECG FILE NAME"
    ]
)

if ecg_name_col is None or echo_name_col is None:
    raise ValueError(
        "Patient name column not found. Check the column names in both files."
    )

if ef_col is None:
    raise ValueError(
        "EF column not found. Check the Echo Excel column headings."
    )


def norm_name(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def name_tokens(value):
    if pd.isna(value):
        return []
    return sorted(re.sub(r"[^A-Z0-9]+", " ", str(value).upper()).split())


def name_score(a, b):
    a1 = norm_name(a)
    b1 = norm_name(b)

    if not a1 or not b1:
        return 0

    if a1 == b1:
        return 100

    ta = name_tokens(a)
    tb = name_tokens(b)

    if not ta or not tb:
        return 0

    sa = "".join(ta)
    sb = "".join(tb)

    if sa == sb:
        return 100

    return max(
        fuzz.ratio(a1, b1),
        fuzz.ratio(sa, sb),
        fuzz.token_sort_ratio(str(a), str(b))
    )


def parse_date(value):
    if pd.isna(value):
        return pd.NaT

    try:
        d = pd.to_datetime(value, errors="coerce", dayfirst=True)
        if pd.notna(d):
            return d.normalize()
    except (ValueError, TypeError):
        pass

    return pd.NaT


def extract_filename_date(value):
    if pd.isna(value):
        return pd.NaT

    text = str(value)

    for pattern in [r"\[(\d{14})\]", r"(\d{14})",
                    r"\[(\d{8})\]", r"(\d{8})"]:
        m = re.search(pattern, text)

        if not m:
            continue

        raw = m.group(1)

        try:
            fmt = "%Y%m%d%H%M%S" if len(raw) == 14 else "%Y%m%d"
            return pd.Timestamp(datetime.strptime(raw, fmt).date())
        except ValueError:
            continue

    return pd.NaT


def normalize_gender(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    if value in ["M", "MALE"]:
        return "M"

    if value in ["F", "FEMALE"]:
        return "F"

    return ""


def normalize_age(value):
    if pd.isna(value):
        return None

    m = re.search(r"\d+", str(value))

    if not m:
        return None

    try:
        return int(m.group())
    except ValueError:
        return None


def normalize_ef(value):
    if pd.isna(value):
        return None

    try:
        value = float(
            str(value).strip().replace("%", "").replace(",", "")
        )
    except (ValueError, TypeError):
        return None

    if value <= 0:
        return None

    if value <= 1:
        value *= 100

    if value > 100:
        return None

    return round(value, 2)


def classify_ef(ef, gender):
    if ef is None or pd.isna(ef):
        return "Unknown"

    gender = normalize_gender(gender)

    if gender == "M":
        limit = 52
    elif gender == "F":
        limit = 54
    else:
        return "Unknown"

    if ef > limit:
        return "Normal"
    if ef >= 41:
        return "Mild"
    if ef >= 31:
        return "Moderate"
    return "Severe"


ecg = ecg.copy()
echo = echo.copy()

ecg["_ECG_ROW"] = range(len(ecg))
echo["_ECHO_ROW"] = range(len(echo))

ecg["_NAME"] = ecg[ecg_name_col].apply(norm_name)
echo["_NAME"] = echo[echo_name_col].apply(norm_name)

ecg["_AGE"] = (
    ecg[ecg_age_col].apply(normalize_age)
    if ecg_age_col else pd.Series([None] * len(ecg), index=ecg.index)
)

echo["_AGE"] = (
    echo[echo_age_col].apply(normalize_age)
    if echo_age_col else pd.Series([None] * len(echo), index=echo.index)
)

ecg["_GENDER"] = (
    ecg[ecg_gender_col].apply(normalize_gender)
    if ecg_gender_col else ""
)

echo["_GENDER"] = (
    echo[echo_gender_col].apply(normalize_gender)
    if echo_gender_col else ""
)

ecg["_DATE"] = (
    ecg[ecg_date_col].apply(parse_date)
    if ecg_date_col else pd.Series(pd.NaT, index=ecg.index)
)

echo["_DATE"] = (
    echo[echo_date_col].apply(parse_date)
    if echo_date_col else pd.Series(pd.NaT, index=echo.index)
)

if filename_col:
    ecg["_DATE"] = ecg["_DATE"].fillna(
        ecg[filename_col].apply(extract_filename_date)
    )

echo["_EF"] = echo[ef_col].apply(normalize_ef)

candidates = []

for i in range(len(ecg)):
    er = ecg.iloc[i]

    if not er["_NAME"]:
        continue

    for j in range(len(echo)):
        hr = echo.iloc[j]

        if not hr["_NAME"] or hr["_EF"] is None:
            continue

        score = name_score(er[ecg_name_col], hr[echo_name_col])

        if score < 80:
            continue

        age1 = er["_AGE"]
        age2 = hr["_AGE"]

        if age1 is not None and age2 is not None and age1 != age2:
            continue

        gender1 = er["_GENDER"]
        gender2 = hr["_GENDER"]

        if gender1 and gender2 and gender1 != gender2:
            continue

        date1 = er["_DATE"]
        date2 = hr["_DATE"]

        date_match = (
            pd.notna(date1)
            and pd.notna(date2)
            and date1 == date2
        )

        exact_name = norm_name(er[ecg_name_col]) == norm_name(
            hr[echo_name_col]
        )

        same_reordered_name = (
            name_tokens(er[ecg_name_col]) == name_tokens(hr[echo_name_col])
        )

        candidates.append({
            "i": i,
            "j": j,
            "score": score,
            "date_match": date_match,
            "exact_name": exact_name or same_reordered_name,
            "age_known": age1 is not None and age2 is not None,
            "gender_known": bool(gender1 and gender2)
        })


candidates.sort(
    key=lambda x: (
        x["date_match"],
        x["exact_name"],
        x["score"],
        x["age_known"],
        x["gender_known"]
    ),
    reverse=True
)

used_ecg = set()
used_echo = set()
matches = []


def assign_candidate(c):
    i = c["i"]
    j = c["j"]

    if i in used_ecg or j in used_echo:
        return

    er = ecg.iloc[i]
    hr = echo.iloc[j]

    gender = hr["_GENDER"] or er["_GENDER"]

    matches.append({
        "ECG Row": int(er["_ECG_ROW"]) + 2,
        "Echo Row": int(hr["_ECHO_ROW"]) + 2,
        "ECG Name": er[ecg_name_col],
        "Echo Name": hr[echo_name_col],
        "Name Match %": round(c["score"], 2),
        "ECG Date": er["_DATE"],
        "Echo Date": hr["_DATE"],
        "Date Match": c["date_match"],
        "Age ECG": er[ecg_age_col] if ecg_age_col else None,
        "Age Echo": hr[echo_age_col] if echo_age_col else None,
        "Gender ECG": er[ecg_gender_col] if ecg_gender_col else None,
        "Gender Echo": hr[echo_gender_col] if echo_gender_col else None,
        "EF Teich (%)": hr["_EF"],
        "EF Classification": classify_ef(hr["_EF"], gender)
    })

    used_ecg.add(i)
    used_echo.add(j)


for c in candidates:
    if c["date_match"]:
        assign_candidate(c)

for c in candidates:
    if c["i"] not in used_ecg and c["j"] not in used_echo:
        assign_candidate(c)


match_df = pd.DataFrame(matches)

if not match_df.empty:
    match_df = match_df.sort_values(
        by=["ECG Row", "Date Match", "Name Match %"],
        ascending=[True, False, False]
    )

unmatched_ecg = ecg[~ecg["_ECG_ROW"].isin(used_ecg)].copy()
unmatched_echo = echo[~echo["_ECHO_ROW"].isin(used_echo)].copy()

unmatched_ecg = unmatched_ecg.drop(
    columns=["_ECG_ROW", "_NAME", "_AGE", "_GENDER", "_DATE"],
    errors="ignore"
)

unmatched_echo = unmatched_echo.drop(
    columns=["_ECHO_ROW", "_NAME", "_AGE", "_GENDER", "_DATE", "_EF"],
    errors="ignore"
)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    match_df.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_ecg.to_excel(writer, sheet_name="Unmatched ECG", index=False)
    unmatched_echo.to_excel(writer, sheet_name="Unmatched Echo", index=False)

    for sheet in writer.sheets.values():
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

        for cells in sheet.columns:
            letter = cells[0].column_letter
            width = max(
                (len(str(cell.value)) for cell in cells[:300]
                 if cell.value is not None),
                default=10
            )
            sheet.column_dimensions[letter].width = min(max(width + 2, 12), 35)

print("Matching completed.")
print("ECG records:", len(ecg))
print("Echo records:", len(echo))
print("Matched records:", len(match_df))
print("Unmatched ECG records:", len(unmatched_ecg))
print("Unmatched Echo records:", len(unmatched_echo))
print("Output file:", os.path.abspath(output_file))

KeyboardInterrupt: 

In [14]:
from rapidfuzz import process, fuzz

ecg_count = len(ecg)
echo_count = len(echo)

matched_ecg = set()
matched_echo = set()
matches = []

echo = echo.reset_index(drop=True)
ecg = ecg.reset_index(drop=True)

echo_names = echo["_NAME"].fillna("").tolist()
echo_sorted = [
    "".join(sorted(re.findall(r"[A-Z0-9]+", str(n))))
    for n in echo[echo_name_col]
]

echo_by_name = {}

for j, n in enumerate(echo_names):
    if n:
        echo_by_name.setdefault(n, []).append(j)

echo_by_sorted = {}

for j, n in enumerate(echo_sorted):
    if n:
        echo_by_sorted.setdefault(n, []).append(j)

candidates = []

for i, er in ecg.iterrows():
    n = er["_NAME"]

    if not n:
        continue

    sorted_n = "".join(sorted(re.findall(r"[A-Z0-9]+", str(er[ecg_name_col]))))

    js = set(echo_by_name.get(n, []))
    js.update(echo_by_sorted.get(sorted_n, []))

    found = process.extract(
        n,
        echo_names,
        scorer=fuzz.ratio,
        score_cutoff=80,
        limit=20
    )

    for _, score, j in found:
        js.add(j)

    found_sorted = process.extract(
        sorted_n,
        echo_sorted,
        scorer=fuzz.ratio,
        score_cutoff=80,
        limit=20
    )

    for _, score, j in found_sorted:
        js.add(j)

    for j in js:
        hr = echo.iloc[j]

        if not hr["_NAME"] or hr["_EF"] is None:
            continue

        score = name_score(er[ecg_name_col], hr[echo_name_col])

        if score < 80:
            continue

        age1 = er["_AGE"]
        age2 = hr["_AGE"]

        if age1 is not None and age2 is not None and age1 != age2:
            continue

        gender1 = er["_GENDER"]
        gender2 = hr["_GENDER"]

        if gender1 and gender2 and gender1 != gender2:
            continue

        date1 = er["_DATE"]
        date2 = hr["_DATE"]

        date_match = pd.notna(date1) and pd.notna(date2) and date1 == date2

        candidates.append((
            int(date_match),
            int(score == 100),
            score,
            i,
            j,
            date_match
        ))

candidates.sort(reverse=True)

for _, _, score, i, j, date_match in candidates:
    if i in matched_ecg or j in matched_echo:
        continue

    er = ecg.iloc[i]
    hr = echo.iloc[j]

    gender = hr["_GENDER"] or er["_GENDER"]

    matches.append({
        "ECG Row": int(er["_ECG_ROW"]),
        "Echo Row": int(hr["_ECHO_ROW"]),
        "ECG Name": er[ecg_name_col],
        "Echo Name": hr[echo_name_col],
        "Name Match %": round(score, 2),
        "ECG Date": er["_DATE"],
        "Echo Date": hr["_DATE"],
        "Age ECG": er[ecg_age_col] if ecg_age_col else None,
        "Age Echo": hr[echo_age_col] if echo_age_col else None,
        "Gender ECG": er[ecg_gender_col] if ecg_gender_col else None,
        "Gender Echo": hr[echo_gender_col] if echo_gender_col else None,
        "EF Teich (%)": hr["_EF"],
        "EF Classification": classify_ef(hr["_EF"], gender),
        "Date Match": date_match
    })

    matched_ecg.add(int(er["_ECG_ROW"]))
    matched_echo.add(int(hr["_ECHO_ROW"]))

match_df = pd.DataFrame(matches)

if not match_df.empty:
    match_df = match_df.sort_values(
        by=["Date Match", "Name Match %"],
        ascending=[False, False]
    )

unmatched = ecg[~ecg["_ECG_ROW"].isin(matched_ecg)].copy()

unmatched = unmatched.drop(
    columns=["_ECG_ROW", "_NAME", "_AGE", "_GENDER", "_DATE"],
    errors="ignore"
)

echo_unmatched = echo[~echo["_ECHO_ROW"].isin(matched_echo)].copy()

echo_unmatched = echo_unmatched.drop(
    columns=["_ECHO_ROW", "_NAME", "_AGE", "_GENDER", "_DATE", "_EF"],
    errors="ignore"
)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    match_df.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched.to_excel(writer, sheet_name="Unmatched ECG", index=False)
    echo_unmatched.to_excel(writer, sheet_name="Unmatched Echo", index=False)

print("Matching completed.")
print("ECG records:", len(ecg))
print("Echo records:", len(echo))
print("Matched records:", len(match_df))
print("Unmatched ECG records:", len(unmatched))
print("Unmatched Echo records:", len(echo_unmatched))
print("Output file:", os.path.abspath(output_file))

KeyboardInterrupt: 

In [15]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")


def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"^SALIVA\s+", "", str(name).upper().strip())
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None
    m = re.search(r"\d+", str(age))
    return int(m.group()) if m else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()
    if sex in ("M", "MALE"):
        return "M"
    if sex in ("F", "FEMALE"):
        return "F"
    return ""


def parse_date(series):
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT
    m = re.search(r"\[(\d{8})\d{6}\]", str(filename))
    if not m:
        return pd.NaT
    return pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)
    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.token_sort_ratio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.token_sort_ratio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])

ecg_date = find_col(
    ecg, ["STUDY DATE", "ECG DATE", "RECORDING DATE", "DATE"]
)

echo_date = find_col(echo, ["STUDY DATE", "STUDY_DATE", "DATE"])

ef_col = find_col(
    echo,
    ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%", "EF (TEICH) %"]
)

required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )

ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

for df, col in [(ecg, ecg_name), (echo, echo_name)]:
    df["name_key"] = df[col].apply(clean_name)
    df["compact_key"] = df[col].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index, dtype=object)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index, dtype=object)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date])
    if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date])
    if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

echo_names = echo["name_key"].tolist()

exact_index = {}
compact_index = {}

for j, h in echo.iterrows():
    exact_index.setdefault(h["name_key"], []).append(j)
    compact_index.setdefault(h["compact_key"], []).append(j)

pairs = []

for i, e in ecg.iterrows():
    name = e["name_key"]
    compact = e["compact_key"]
    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    queries = [name]

    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            echo_names,
            scorer=fuzz.token_sort_ratio,
            limit=15,
            score_cutoff=80
        )

        candidates.update(idx for _, _, idx in found)

    for j in candidates:
        h = echo.iloc[j]

        score = name_score(name, h["name_key"])

        if score < 80:
            continue

        if (
            e["age_key"] is not None
            and h["age_key"] is not None
            and e["age_key"] != h["age_key"]
        ):
            continue

        if (
            e["sex_key"]
            and h["sex_key"]
            and e["sex_key"] != h["sex_key"]
        ):
            continue

        ed = e["match_date"]
        hd = h["match_date"]

        date_match = int(
            pd.notna(ed) and pd.notna(hd) and ed == hd
        )

        exact = int(score == 100)

        age_match = int(
            e["age_key"] is not None
            and h["age_key"] is not None
            and e["age_key"] == h["age_key"]
        )

        sex_match = int(
            bool(e["sex_key"])
            and bool(h["sex_key"])
            and e["sex_key"] == h["sex_key"]
        )

        pairs.append(
            (date_match, exact, score, age_match, sex_match, i, j)
        )

pairs.sort(key=lambda x: x[:5], reverse=True)

used_ecg = set()
used_echo = set()
matches = []

for date_match, exact, score, age_match, sex_match, i, j in pairs:
    if i in used_ecg or j in used_echo:
        continue

    e = ecg.iloc[i]
    h = echo.iloc[j]

    matches.append({
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": e[ecg_age] if ecg_age else (
            h[echo_age] if echo_age else None
        ),
        "Gender": e[ecg_sex] if ecg_sex else (
            h[echo_sex] if echo_sex else ""
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": h[ef_col],
        "Match score": round(score, 2)
    })

    used_ecg.add(i)
    used_echo.add(j)

result = pd.DataFrame(matches)


def classify_lv(row):
    ef = row["EF (Teich) %"]
    sex = clean_sex(row["Gender"])

    if pd.isna(ef):
        return "Unknown"

    if sex == "M":
        if ef > 52:
            return "Normal"
        if ef >= 41:
            return "Mild"

    elif sex == "F":
        if ef >= 54:
            return "Normal"
        if ef >= 41:
            return "Mild"

    if 31 <= ef < 41:
        return "Moderate"

    if ef < 31:
        return "Severe"

    return "Unknown"


if not result.empty:
    result["LV"] = result.apply(classify_lv, axis=1)

unmatched_ecg = []

for i, e in ecg.iterrows():
    if i not in used_ecg:
        unmatched_ecg.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "No valid unused Echo match"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg)

unmatched_echo = []

for j, h in echo.iterrows():
    if j not in used_echo:
        unmatched_echo.append({
            "Echo name": h["original_name"],
            "Echo age": h[echo_age] if echo_age else None,
            "Echo gender": h[echo_sex] if echo_sex else "",
            "Echo study date": h["match_date"],
            "EF (Teich) %": h[ef_col],
            "Reason": "No valid unused ECG match"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo)

filename = (
    "matched_ecg_echo_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

print("ECG records:", len(ecg))
print("Echo records:", len(echo))
print("Matched records:", len(result))
print("Unmatched ECG records:", len(unmatched_ecg_df))
print("Unmatched Echo records:", len(unmatched_echo_df))
print("Unique Echo records used:", len(used_echo))
print("Saved file:", filename)

display(result)
display(unmatched_ecg_df)
display(unmatched_echo_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\235455644.py:56: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(series, errors="coerce").dt.normalize()


ECG records: 5405
Echo records: 2127
Matched records: 1208
Unmatched ECG records: 4197
Unmatched Echo records: 919
Unique Echo records used: 1208
Saved file: matched_ecg_echo_20261029_155649.xlsx


,ECG name,Echo name,Age,Gender,ECG file name,ECG date,Echo study date,EF (Teich) %,Match score,LV
0,CHINNAKKA VENKATAKRISHNAREDDY,CHINNAKKA VENKATAKRISHNA REDDY,42 Years,Male,EK__[20210128111159]_CHINNAKKA VENKATAKRISHNAR...,2021-01-28,2021-01-28,61.9,100.0,Normal
1,PALEPU ANUSHA,PALEPU ANUSHA,31 Years,Female,EK__[20210131112642]_PALEPU ANUSHA____[]_.pdf,2021-01-31,2021-01-31,73.3,100.0,Normal
2,THALLA KAVITHA,THALLA KAVITHA,32 Years,Female,EK__[20210131124414]_THALLA KAVITHA____[]_.pdf,2021-01-31,2021-01-31,68.5,100.0,Normal
3,RAYUDU SRINIVAS,RAYUDU SRINIVAS,50 Years,Male,EK__[20210225110105]_RAYUDU SRINIVAS____[]_.pdf,2021-02-25,2021-02-25,47.6,100.0,Mild
4,KALPALA FRANCIS,KALPALA FRANCIS GEORGE,68 Years,Male,EK__[20210225115501]_KALPALA FRANCIS____[]_.pdf,2021-02-25,2021-02-25,67.3,100.0,Normal
...,...,...,...,...,...,...,...,...,...,...
1203,MANNALA MONDAIAH,ANKAM MONDAIAH,62 Years,Male,EK__[20210712112111]_MANNALA MONDAIAH____[]_.pdf,2021-12-07,2021-12-06,66.7,80.0,Normal
1204,MEKALA SRINIVAS,SRINIVAS NAGULA,51 Years,Male,EK__[20210723174922]_MEKALA SRINIVAS____[]_.pdf,2021-07-23,2021-07-05,68.8,80.0,Normal
1205,PALLAPU YELLAMMA,DATLA YELLAMMA,48 Years,Female,EK__[20210827105535]_PALLAPU YELLAMMA____[]_.pdf,2021-08-27,2021-10-18,78.4,80.0,Normal
1206,CHANDA KUMARASWAMY,BASHABOINA KUMARASWAMY,48 Years,Male,EK__[20210903101705]_CHANDA KUMARASWAMY____[]_...,2021-03-09,2021-08-21,63.2,80.0,Normal


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,KALLURI VARALAXMI,70 Years,Female,2021-01-01,EK__[20210101110036]_KALLURI VARALAXMI____[]_.pdf,No valid unused Echo match
1,BUKYA VEERANNA,28 Years,Male,2021-01-01,EK__[20210101121417]_BUKYA VEERANNA____[]_.pdf,No valid unused Echo match
2,CHINTHA THIRUPATHI,38 Years,Male,2021-01-01,EK__[20210101155310]_CHINTHA THIRUPATHI____[]_...,No valid unused Echo match
3,AKUNURI BHADRAIAH,43 Years,Male,2021-01-01,EK__[20210101190531]_AKUNURI BHADRAIAH____[]_.pdf,No valid unused Echo match
4,MD JAMEELAHMED,52 Years,Male,2021-02-01,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,No valid unused Echo match
...,...,...,...,...,...,...
4192,VARANASI PADMAJA DEVI,56 Years,Male,2021-12-31,EK__[20211231183317]_VARANASI PADMAJA DEVI____...,No valid unused Echo match
4193,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191647]_CHEVULA ANDALAMMA____[]_.pdf,No valid unused Echo match
4194,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191833]_CHEVULA ANDALAMMA____[]_.pdf,No valid unused Echo match
4195,CHEVULA LAXMINARASAIAH,75 Years,Male,2021-12-31,EK__[20211231192120]_CHEVULA LAXMINARASAIAH___...,No valid unused Echo match


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,RAMESH KUNCHAM,44.0,Male,2021-05-03,NaN,No valid unused ECG match
1,VARSHITH,3.0,Male,2021-12-07,78.2,No valid unused ECG match
2,BODIGE SUJATHA,37.0,Female,2021-12-14,70.3,No valid unused ECG match
3,TENALI RATTHAMMA,80.0,Female,2021-03-23,74.7,No valid unused ECG match
4,YAKUB ALI,71.0,Male,2021-06-17,51.6,No valid unused ECG match
...,...,...,...,...,...,...
914,KATTU KAMESHWARI,64.0,Female,2021-04-01,NaN,No valid unused ECG match
915,KATTU KAMESHWARI,64.0,Female,2021-04-01,NaN,No valid unused ECG match
916,PASUKANTI MANOHAR,58.0,Male,2021-10-26,NaN,No valid unused ECG match
917,MD TABISH,6.0,Male,2021-11-07,NaN,No valid unused ECG match


In [16]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")


def standardize_columns(df):
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"^SALIVA\s+", "", str(name).upper().strip())
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None
    m = re.search(r"\d+", str(age))
    return int(m.group()) if m else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()

    if sex in ("M", "MALE"):
        return "M"

    if sex in ("F", "FEMALE"):
        return "F"

    return ""


def parse_date(series):
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    m = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not m:
        return pd.NaT

    return pd.to_datetime(
        m.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(
                fuzz.token_sort_ratio(variant, b1)
            )

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(
                fuzz.token_sort_ratio(a1, variant)
            )

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])

ecg_date = find_col(
    ecg,
    ["STUDY DATE", "ECG DATE", "RECORDING DATE", "DATE"]
)

echo_date = find_col(
    echo,
    ["STUDY DATE", "STUDY_DATE", "DATE"]
)

ef_col = find_col(
    echo,
    ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%", "EF (TEICH) %"]
)

required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

for df, col in [(ecg, ecg_name), (echo, echo_name)]:
    df["name_key"] = df[col].apply(clean_name)
    df["compact_key"] = df[col].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index, dtype=object)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index, dtype=object)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date])
    if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date])
    if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(
    echo[ef_col],
    errors="coerce"
)

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100
    if pd.notna(x) and 0 < x <= 1
    else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(echo_row):
    name = echo_row["name_key"]
    compact = echo_row["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    queries = [name]

    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            ecg_names,
            scorer=fuzz.token_sort_ratio,
            limit=20,
            score_cutoff=80
        )

        candidates.update(
            idx for _, _, idx in found
        )

    valid = []

    for i in candidates:
        row = ecg.iloc[i]

        score = name_score(
            echo_row["name_key"],
            row["name_key"]
        )

        if score < 80:
            continue

        if (
            echo_row["age_key"] is not None
            and row["age_key"] is not None
            and echo_row["age_key"] != row["age_key"]
        ):
            continue

        if (
            echo_row["sex_key"]
            and row["sex_key"]
            and echo_row["sex_key"] != row["sex_key"]
        ):
            continue

        ed = echo_row["match_date"]
        cd = row["match_date"]

        same_date = int(
            pd.notna(ed)
            and pd.notna(cd)
            and ed == cd
        )

        if pd.notna(ed) and pd.notna(cd):
            date_distance = abs((ed - cd).days)
        else:
            date_distance = 10**9

        exact = int(score == 100)

        valid.append(
            (same_date, exact, score, date_distance, i)
        )

    valid.sort(
        key=lambda x: (
            x[0],
            x[1],
            x[2],
            -x[3]
        ),
        reverse=True
    )

    return valid


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()

for j, h in echo.iterrows():
    candidates = find_candidates(h)

    if not candidates:
        continue

    same_date_candidates = [
        c for c in candidates if c[0] == 1
    ]

    if same_date_candidates:
        selected = same_date_candidates[0]
        match_reason = "Same patient and same date"

    else:
        selected = candidates[0]

        patient_candidates = [
            c for c in candidates
            if c[2] >= 80
        ]

        if len(patient_candidates) == 1:
            match_reason = "Same patient; only one ECG available"
        elif selected[3] < 10**9:
            match_reason = "Same patient; closest available date"
        else:
            match_reason = "Same patient; dates unavailable"

    i = selected[4]
    e = ecg.iloc[i]
    score = selected[2]

    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"
    else:
        sex = e["sex_key"] or h["sex_key"]

        if sex == "M" and ef > 52:
            lv = "Normal"
        elif sex == "F" and ef >= 54:
            lv = "Normal"
        elif ef >= 41:
            lv = "Mild"
        elif ef >= 31:
            lv = "Moderate"
        elif ef < 31:
            lv = "Severe"
        else:
            lv = "Unknown"

    matches.append({
        "Name": e["original_name"],
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": e[ecg_age] if ecg_age else (
            h[echo_age] if echo_age else None
        ),
        "Gender": e[ecg_sex] if ecg_sex else (
            h[echo_sex] if echo_sex else ""
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": ef,
        "Match score": round(score, 2),
        "Match reason": match_reason,
        "LV": lv
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


result = pd.DataFrame(matches)

unmatched_echo_rows = []

for j, h in echo.iterrows():
    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo name": h["original_name"],
            "Echo age": h[echo_age] if echo_age else None,
            "Echo gender": h[echo_sex] if echo_sex else "",
            "Echo study date": h["match_date"],
            "EF (Teich) %": h[ef_col],
            "Reason": "No valid ECG name match"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)

unmatched_ecg_rows = []

for i, e in ecg.iterrows():
    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "Not selected for any Echo report"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)

filename = (
    "echo_first_ecg_matching_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo reports:", len(result))
print("Unmatched Echo reports:", len(unmatched_echo_df))
print("Unique ECG records selected:", len(matched_ecg_indices))
print("Saved file:", filename)

display(result)
display(unmatched_echo_df)
display(unmatched_ecg_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\2370324883.py:59: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(series, errors="coerce").dt.normalize()


Echo records: 2127
ECG records: 5405
Matched Echo reports: 1226
Unmatched Echo reports: 901
Unique ECG records selected: 1195
Saved file: echo_first_ecg_matching_20261029_160354.xlsx


,Name,ECG name,Echo name,Age,Gender,ECG file name,ECG date,Echo study date,EF (Teich) %,Match score,Match reason,LV
0,KUNAMALLA SRINIVASARAO,KUNAMALLA SRINIVASARAO,KUNAMALLA SRINIVASA RAO,51 Years,Male,EK__[20210920102421]_KUNAMALLA SRINIVASARAO___...,2021-09-20,2021-09-20,75.9,100.00,Same patient and same date,Normal
1,LONKA VENKATALAXMI,LONKA VENKATALAXMI,LONKA VENKATALAKSHMI,60 Years,Female,EK__[20211206142136]_LONKA VENKATALAXMI____[]_...,2021-06-12,2021-12-10,72.7,89.47,Same patient; only one ECG available,Normal
2,MAHAMMAD NASEEM,MAHAMMAD NASEEM,MOHAMMED NASEEM,50 Years,Female,EK__[20210831171846]_MAHAMMAD NASEEM____[]_.pdf,2021-08-31,2021-08-31,63.8,86.67,Same patient and same date,Normal
3,K KAMESHARI,K KAMESHARI,KAMESHWARI,65 Years,Female,EK__[20210223160615]_K KAMESHARI____[]_.pdf,2021-02-23,2021-11-30,70.7,90.00,Same patient; only one ECG available,Normal
4,GUNDAMALLA NARAYANA,GUNDAMALLA NARAYANA,GUNDAMALLA NARAYANA,66 Years,Male,EK__[20210406105657]_GUNDAMALLA NARAYANA____[]...,2021-06-04,2021-04-06,70.8,100.00,Same patient; only one ECG available,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...
1221,ADDUNURI MALLAIAH,ADDUNURI MALLAIAH,ADUNURI MALLAIAH,51 Years,Male,EK__[20210518112536]_ADDUNURI MALLAIAH____[]_.pdf,2021-05-18,2021-06-02,68.3,96.97,Same patient; only one ECG available,Normal
1222,BODDIREDDY PULAMMA,BODDIREDDY PULAMMA,PHOOLAMMA BODDIREDDY,65 Years,Female,EK__[20210630111007]_BODDIREDDY PULAMMA____[]_...,2021-06-30,2021-06-30,63.7,89.47,Same patient and same date,Normal
1223,POUDALA LAXMAN,POUDALA LAXMAN,POUDALA LAXMAN,37 Years,Male,EK__[20210601105256]_POUDALA LAXMAN____[]_.pdf,2021-01-06,2021-06-01,68.7,100.00,Same patient; only one ECG available,Normal
1224,POLAGANI KATAMAIAH,POLAGANI KATAMAIAH,KATAMAIAH POLAGANI,62 Years,Male,EK__[20210615145415]_POLAGANI KATAMAIAH____[]_...,2021-06-15,2021-06-15,NaN,100.00,Same patient and same date,Unknown


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,RAMESH KUNCHAM,44.0,Male,2021-05-03,NaN,No valid ECG name match
1,VARSHITH,3.0,Male,2021-12-07,78.2,No valid ECG name match
2,BODIGE SUJATHA,37.0,Female,2021-12-14,70.3,No valid ECG name match
3,TENALI RATTHAMMA,80.0,Female,2021-03-23,74.7,No valid ECG name match
4,YAKUB ALI,71.0,Male,2021-06-17,51.6,No valid ECG name match
...,...,...,...,...,...,...
896,KATTU KAMESHWARI,64.0,Female,2021-04-01,NaN,No valid ECG name match
897,KATTU KAMESHWARI,64.0,Female,2021-04-01,NaN,No valid ECG name match
898,PASUKANTI MANOHAR,58.0,Male,2021-10-26,NaN,No valid ECG name match
899,MD TABISH,6.0,Male,2021-11-07,NaN,No valid ECG name match


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,KALLURI VARALAXMI,70 Years,Female,2021-01-01,EK__[20210101110036]_KALLURI VARALAXMI____[]_.pdf,Not selected for any Echo report
1,BUKYA VEERANNA,28 Years,Male,2021-01-01,EK__[20210101121417]_BUKYA VEERANNA____[]_.pdf,Not selected for any Echo report
2,CHINTHA THIRUPATHI,38 Years,Male,2021-01-01,EK__[20210101155310]_CHINTHA THIRUPATHI____[]_...,Not selected for any Echo report
3,AKUNURI BHADRAIAH,43 Years,Male,2021-01-01,EK__[20210101190531]_AKUNURI BHADRAIAH____[]_.pdf,Not selected for any Echo report
4,MD JAMEELAHMED,52 Years,Male,2021-02-01,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,Not selected for any Echo report
...,...,...,...,...,...,...
4205,VARANASI PADMAJA DEVI,56 Years,Male,2021-12-31,EK__[20211231183317]_VARANASI PADMAJA DEVI____...,Not selected for any Echo report
4206,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191647]_CHEVULA ANDALAMMA____[]_.pdf,Not selected for any Echo report
4207,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191833]_CHEVULA ANDALAMMA____[]_.pdf,Not selected for any Echo report
4208,CHEVULA LAXMINARASAIAH,75 Years,Male,2021-12-31,EK__[20211231192120]_CHEVULA LAXMINARASAIAH___...,Not selected for any Echo report


In [17]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

ecg = pd.read_excel("ECG.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2021echo_reports.xlsx", sheet_name="Sheet1")


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"^SALIVA\s+", "", str(name).upper().strip())
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None

    match = re.search(r"\d+(?:\.\d+)?", str(age))

    if not match:
        return None

    return float(match.group())


def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ("M", "MALE"):
        return "M"

    if sex in ("F", "FEMALE"):
        return "F"

    return ""


def parse_date(series):
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    match = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not match:
        return pd.NaT

    return pd.to_datetime(
        match.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.token_sort_ratio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.token_sort_ratio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])

ecg_date = find_col(
    ecg,
    ["STUDY DATE", "ECG DATE", "RECORDING DATE", "DATE"]
)

echo_date = find_col(
    echo,
    ["STUDY DATE", "STUDY_DATE", "DATE"]
)

ef_col = find_col(
    echo,
    ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%", "EF (TEICH) %"]
)


required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [key for key, value in required.items() if value is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

ecg["name_key"] = ecg[ecg_name].apply(clean_name)
echo["name_key"] = echo[echo_name].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name].apply(compact_name)
echo["compact_key"] = echo[echo_name].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index, dtype=object)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index, dtype=object)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date])
    if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date])
    if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)


ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(echo_row):
    name = echo_row["name_key"]
    compact = echo_row["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    queries = [name]

    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            ecg_names,
            scorer=fuzz.token_sort_ratio,
            limit=20,
            score_cutoff=80
        )

        candidates.update(idx for _, _, idx in found)

    valid = []

    for i in candidates:
        row = ecg.iloc[i]

        score = name_score(
            echo_row["name_key"],
            row["name_key"]
        )

        if score < 80:
            continue

        echo_age_value = echo_row["age_key"]
        ecg_age_value = row["age_key"]

        if (
            echo_age_value is not None
            and ecg_age_value is not None
            and echo_age_value != ecg_age_value
        ):
            continue

        if (
            echo_row["sex_key"]
            and row["sex_key"]
            and echo_row["sex_key"] != row["sex_key"]
        ):
            continue

        ed = echo_row["match_date"]
        cd = row["match_date"]

        same_date = int(
            pd.notna(ed)
            and pd.notna(cd)
            and ed == cd
        )

        if pd.notna(ed) and pd.notna(cd):
            date_distance = abs((ed - cd).days)
        else:
            date_distance = 10**9

        exact = int(score == 100)

        valid.append(
            (
                same_date,
                exact,
                score,
                date_distance,
                i
            )
        )

    valid.sort(
        key=lambda x: (
            x[0],
            x[1],
            x[2],
            -x[3]
        ),
        reverse=True
    )

    return valid


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()


for j, h in echo.iterrows():

    candidates = find_candidates(h)

    if not candidates:
        continue

    same_date_candidates = [
        c for c in candidates
        if c[0] == 1
    ]

    if same_date_candidates:
        selected = same_date_candidates[0]
        match_reason = "Same patient and same date"

    else:
        selected = candidates[0]

        if len(candidates) == 1:
            match_reason = "Same patient; only one ECG candidate"

        elif selected[3] < 10**9:
            match_reason = "Same patient; closest available date"

        else:
            match_reason = "Same patient; dates unavailable"

    i = selected[4]

    e = ecg.iloc[i]
    score = selected[2]
    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"

    else:
        sex = e["sex_key"] or h["sex_key"]

        if sex == "M" and ef > 52:
            lv = "Normal"

        elif sex == "F" and ef >= 54:
            lv = "Normal"

        elif ef >= 41:
            lv = "Mild"

        elif ef >= 31:
            lv = "Moderate"

        elif ef < 31:
            lv = "Severe"

        else:
            lv = "Unknown"

    matches.append({
        "Name": e["original_name"],
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": (
            e[ecg_age]
            if ecg_age
            else (h[echo_age] if echo_age else None)
        ),
        "Gender": (
            e[ecg_sex]
            if ecg_sex
            else (h[echo_sex] if echo_sex else "")
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": ef,
        "Match score": round(score, 2),
        "Match reason": match_reason,
        "LV": lv
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


result = pd.DataFrame(matches)


unmatched_echo_rows = []

for j, h in echo.iterrows():

    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo name": h["original_name"],
            "Echo age": h[echo_age] if echo_age else None,
            "Echo gender": h[echo_sex] if echo_sex else "",
            "Echo study date": h["match_date"],
            "EF (Teich) %": h[ef_col],
            "Reason": "No valid ECG name match"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)


unmatched_ecg_rows = []

for i, e in ecg.iterrows():

    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "Not selected for any Echo report"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)


filename = (
    "echo_first_ecg_matching_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )


print("========== MATCHING SUMMARY ==========")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo reports:", len(result))
print("Unmatched Echo reports:", len(unmatched_echo_df))
print("Unique ECG records selected:", len(matched_ecg_indices))
print("Saved file:", filename)


print("\n========== LV CLASSIFICATION SUMMARY ==========")

if not result.empty and "LV" in result.columns:

    counts = result["LV"].value_counts()

    print("Normal:", counts.get("Normal", 0))
    print("Mild:", counts.get("Mild", 0))
    print("Moderate:", counts.get("Moderate", 0))
    print("Severe:", counts.get("Severe", 0))
    print("Unknown:", counts.get("Unknown", 0))
    print("Total matched Echo reports:", len(result))

else:
    print("No matched Echo reports available.")
    print("Normal: 0")
    print("Mild: 0")
    print("Moderate: 0")
    print("Severe: 0")
    print("Unknown: 0")

print("===============================================")


display(result)
display(unmatched_echo_df)
display(unmatched_ecg_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\1474107432.py:67: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(series, errors="coerce").dt.normalize()


========== MATCHING SUMMARY ==========
Echo records: 2127
ECG records: 5405
Matched Echo reports: 1226
Unmatched Echo reports: 901
Unique ECG records selected: 1195
Saved file: echo_first_ecg_matching_20261029_161227.xlsx

========== LV CLASSIFICATION SUMMARY ==========
Normal: 1033
Mild: 82
Moderate: 75
Severe: 34
Unknown: 2
Total matched Echo reports: 1226


,Name,ECG name,Echo name,Age,Gender,ECG file name,ECG date,Echo study date,EF (Teich) %,Match score,Match reason,LV
0,KUNAMALLA SRINIVASARAO,KUNAMALLA SRINIVASARAO,KUNAMALLA SRINIVASA RAO,51 Years,Male,EK__[20210920102421]_KUNAMALLA SRINIVASARAO___...,2021-09-20,2021-09-20,75.9,100.00,Same patient and same date,Normal
1,LONKA VENKATALAXMI,LONKA VENKATALAXMI,LONKA VENKATALAKSHMI,60 Years,Female,EK__[20211206142136]_LONKA VENKATALAXMI____[]_...,2021-06-12,2021-12-10,72.7,89.47,Same patient; only one ECG candidate,Normal
2,MAHAMMAD NASEEM,MAHAMMAD NASEEM,MOHAMMED NASEEM,50 Years,Female,EK__[20210831171846]_MAHAMMAD NASEEM____[]_.pdf,2021-08-31,2021-08-31,63.8,86.67,Same patient and same date,Normal
3,K KAMESHARI,K KAMESHARI,KAMESHWARI,65 Years,Female,EK__[20210223160615]_K KAMESHARI____[]_.pdf,2021-02-23,2021-11-30,70.7,90.00,Same patient; only one ECG candidate,Normal
4,GUNDAMALLA NARAYANA,GUNDAMALLA NARAYANA,GUNDAMALLA NARAYANA,66 Years,Male,EK__[20210406105657]_GUNDAMALLA NARAYANA____[]...,2021-06-04,2021-04-06,70.8,100.00,Same patient; only one ECG candidate,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...
1221,ADDUNURI MALLAIAH,ADDUNURI MALLAIAH,ADUNURI MALLAIAH,51 Years,Male,EK__[20210518112536]_ADDUNURI MALLAIAH____[]_.pdf,2021-05-18,2021-06-02,68.3,96.97,Same patient; only one ECG candidate,Normal
1222,BODDIREDDY PULAMMA,BODDIREDDY PULAMMA,PHOOLAMMA BODDIREDDY,65 Years,Female,EK__[20210630111007]_BODDIREDDY PULAMMA____[]_...,2021-06-30,2021-06-30,63.7,89.47,Same patient and same date,Normal
1223,POUDALA LAXMAN,POUDALA LAXMAN,POUDALA LAXMAN,37 Years,Male,EK__[20210601105256]_POUDALA LAXMAN____[]_.pdf,2021-01-06,2021-06-01,68.7,100.00,Same patient; only one ECG candidate,Normal
1224,POLAGANI KATAMAIAH,POLAGANI KATAMAIAH,KATAMAIAH POLAGANI,62 Years,Male,EK__[20210615145415]_POLAGANI KATAMAIAH____[]_...,2021-06-15,2021-06-15,NaN,100.00,Same patient and same date,Unknown


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,RAMESH KUNCHAM,44.0,Male,2021-05-03,NaN,No valid ECG name match
1,VARSHITH,3.0,Male,2021-12-07,78.2,No valid ECG name match
2,BODIGE SUJATHA,37.0,Female,2021-12-14,70.3,No valid ECG name match
3,TENALI RATTHAMMA,80.0,Female,2021-03-23,74.7,No valid ECG name match
4,YAKUB ALI,71.0,Male,2021-06-17,51.6,No valid ECG name match
...,...,...,...,...,...,...
896,KATTU KAMESHWARI,64.0,Female,2021-04-01,NaN,No valid ECG name match
897,KATTU KAMESHWARI,64.0,Female,2021-04-01,NaN,No valid ECG name match
898,PASUKANTI MANOHAR,58.0,Male,2021-10-26,NaN,No valid ECG name match
899,MD TABISH,6.0,Male,2021-11-07,NaN,No valid ECG name match


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,KALLURI VARALAXMI,70 Years,Female,2021-01-01,EK__[20210101110036]_KALLURI VARALAXMI____[]_.pdf,Not selected for any Echo report
1,BUKYA VEERANNA,28 Years,Male,2021-01-01,EK__[20210101121417]_BUKYA VEERANNA____[]_.pdf,Not selected for any Echo report
2,CHINTHA THIRUPATHI,38 Years,Male,2021-01-01,EK__[20210101155310]_CHINTHA THIRUPATHI____[]_...,Not selected for any Echo report
3,AKUNURI BHADRAIAH,43 Years,Male,2021-01-01,EK__[20210101190531]_AKUNURI BHADRAIAH____[]_.pdf,Not selected for any Echo report
4,MD JAMEELAHMED,52 Years,Male,2021-02-01,EK__[20210102104614]_MD JAMEELAHMED____[]_.pdf,Not selected for any Echo report
...,...,...,...,...,...,...
4205,VARANASI PADMAJA DEVI,56 Years,Male,2021-12-31,EK__[20211231183317]_VARANASI PADMAJA DEVI____...,Not selected for any Echo report
4206,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191647]_CHEVULA ANDALAMMA____[]_.pdf,Not selected for any Echo report
4207,CHEVULA ANDALAMMA,66 Years,Female,2021-12-31,EK__[20211231191833]_CHEVULA ANDALAMMA____[]_.pdf,Not selected for any Echo report
4208,CHEVULA LAXMINARASAIAH,75 Years,Male,2021-12-31,EK__[20211231192120]_CHEVULA LAXMINARASAIAH___...,Not selected for any Echo report


In [18]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

ecg = pd.read_excel("2020_Ecg1.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2020_data.xlsx", sheet_name="Sheet1")


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"^SALIVA\s+", "", str(name).upper().strip())
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None

    match = re.search(r"\d+(?:\.\d+)?", str(age))

    if not match:
        return None

    return float(match.group())


def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ("M", "MALE"):
        return "M"

    if sex in ("F", "FEMALE"):
        return "F"

    return ""


def parse_date(series):
    return pd.to_datetime(series, errors="coerce").dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    match = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not match:
        return pd.NaT

    return pd.to_datetime(
        match.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.token_sort_ratio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.token_sort_ratio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])

ecg_date = find_col(
    ecg,
    ["STUDY DATE", "ECG DATE", "RECORDING DATE", "DATE"]
)

echo_date = find_col(
    echo,
    ["STUDY DATE", "STUDY_DATE", "DATE"]
)

ef_col = find_col(
    echo,
    ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%", "EF (TEICH) %"]
)


required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [key for key, value in required.items() if value is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

ecg["name_key"] = ecg[ecg_name].apply(clean_name)
echo["name_key"] = echo[echo_name].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name].apply(compact_name)
echo["compact_key"] = echo[echo_name].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index, dtype=object)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index, dtype=object)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date])
    if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date])
    if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)


ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(echo_row):
    name = echo_row["name_key"]
    compact = echo_row["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    queries = [name]

    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            ecg_names,
            scorer=fuzz.token_sort_ratio,
            limit=20,
            score_cutoff=80
        )

        candidates.update(idx for _, _, idx in found)

    valid = []

    for i in candidates:
        row = ecg.iloc[i]

        score = name_score(
            echo_row["name_key"],
            row["name_key"]
        )

        if score < 80:
            continue

        echo_age_value = echo_row["age_key"]
        ecg_age_value = row["age_key"]

        if (
            echo_age_value is not None
            and ecg_age_value is not None
            and echo_age_value != ecg_age_value
        ):
            continue

        if (
            echo_row["sex_key"]
            and row["sex_key"]
            and echo_row["sex_key"] != row["sex_key"]
        ):
            continue

        ed = echo_row["match_date"]
        cd = row["match_date"]

        same_date = int(
            pd.notna(ed)
            and pd.notna(cd)
            and ed == cd
        )

        if pd.notna(ed) and pd.notna(cd):
            date_distance = abs((ed - cd).days)
        else:
            date_distance = 10**9

        exact = int(score == 100)

        valid.append(
            (
                same_date,
                exact,
                score,
                date_distance,
                i
            )
        )

    valid.sort(
        key=lambda x: (
            x[0],
            x[1],
            x[2],
            -x[3]
        ),
        reverse=True
    )

    return valid


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()


for j, h in echo.iterrows():

    candidates = find_candidates(h)

    if not candidates:
        continue

    same_date_candidates = [
        c for c in candidates
        if c[0] == 1
    ]

    if same_date_candidates:
        selected = same_date_candidates[0]
        match_reason = "Same patient and same date"

    else:
        selected = candidates[0]

        if len(candidates) == 1:
            match_reason = "Same patient; only one ECG candidate"

        elif selected[3] < 10**9:
            match_reason = "Same patient; closest available date"

        else:
            match_reason = "Same patient; dates unavailable"

    i = selected[4]

    e = ecg.iloc[i]
    score = selected[2]
    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"

    else:
        sex = e["sex_key"] or h["sex_key"]

        if sex == "M" and ef > 52:
            lv = "Normal"

        elif sex == "F" and ef >= 54:
            lv = "Normal"

        elif ef >= 41:
            lv = "Mild"

        elif ef >= 31:
            lv = "Moderate"

        elif ef < 31:
            lv = "Severe"

        else:
            lv = "Unknown"

    matches.append({
        "Name": e["original_name"],
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": (
            e[ecg_age]
            if ecg_age
            else (h[echo_age] if echo_age else None)
        ),
        "Gender": (
            e[ecg_sex]
            if ecg_sex
            else (h[echo_sex] if echo_sex else "")
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": ef,
        "Match score": round(score, 2),
        "Match reason": match_reason,
        "LV": lv
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


result = pd.DataFrame(matches)


unmatched_echo_rows = []

for j, h in echo.iterrows():

    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo name": h["original_name"],
            "Echo age": h[echo_age] if echo_age else None,
            "Echo gender": h[echo_sex] if echo_sex else "",
            "Echo study date": h["match_date"],
            "EF (Teich) %": h[ef_col],
            "Reason": "No valid ECG name match"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)


unmatched_ecg_rows = []

for i, e in ecg.iterrows():

    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "Not selected for any Echo report"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)


filename = (
    "echo_first_ecg_matching_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )


print("========== MATCHING SUMMARY ==========")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo reports:", len(result))
print("Unmatched Echo reports:", len(unmatched_echo_df))
print("Unique ECG records selected:", len(matched_ecg_indices))
print("Saved file:", filename)


print("\n========== LV CLASSIFICATION SUMMARY ==========")

if not result.empty and "LV" in result.columns:

    counts = result["LV"].value_counts()

    print("Normal:", counts.get("Normal", 0))
    print("Mild:", counts.get("Mild", 0))
    print("Moderate:", counts.get("Moderate", 0))
    print("Severe:", counts.get("Severe", 0))
    print("Unknown:", counts.get("Unknown", 0))
    print("Total matched Echo reports:", len(result))

else:
    print("No matched Echo reports available.")
    print("Normal: 0")
    print("Mild: 0")
    print("Moderate: 0")
    print("Severe: 0")
    print("Unknown: 0")

print("===============================================")


display(result)
display(unmatched_echo_df)
display(unmatched_ecg_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\2429712181.py:67: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(series, errors="coerce").dt.normalize()


========== MATCHING SUMMARY ==========
Echo records: 1258
ECG records: 2139
Matched Echo reports: 246
Unmatched Echo reports: 1012
Unique ECG records selected: 243
Saved file: echo_first_ecg_matching_20261029_161338.xlsx

========== LV CLASSIFICATION SUMMARY ==========
Normal: 218
Mild: 12
Moderate: 9
Severe: 7
Unknown: 0
Total matched Echo reports: 246


,Name,ECG name,Echo name,Age,Gender,ECG file name,ECG date,Echo study date,EF (Teich) %,Match score,Match reason,LV
0,SALIVA DILIPKUMAR,SALIVA DILIPKUMAR,DILIP KUMAR,46 Years,Male,EK__[20201219104107]_SALIVA DILIPKUMAR____[]_.pdf,2020-12-19,2020-05-06,13.4,100.00,Same patient; only one ECG candidate,Severe
1,ARISHAM NARSAIAH,ARISHAM NARSAIAH,NARSAIAH HARSHAM,60 Years,Male,EK__[20200914112110]_ARISHAM NARSAIAH____[]_.pdf,2020-09-14,NaT,18.4,93.75,Same patient; only one ECG candidate,Severe
2,SHAGANTI VENKAYAMMA,SHAGANTI VENKAYAMMA,SHAGANTI VENKAYAMMA,65 Years,Female,EK__[20200824122232]_SHAGANTI VENKAYAMMA____[]...,2020-08-24,NaT,18.8,100.00,Same patient; dates unavailable,Severe
3,DHEEKONDA KAMALAMMA,DHEEKONDA KAMALAMMA,DEEKONDA KAMALAMMA,75 Years,Female,EK__[20201003110753]_DHEEKONDA KAMALAMMA____[]...,2020-10-03,2020-03-10,26.1,97.30,Same patient; only one ECG candidate,Severe
4,POLU JYOTHI,POLU JYOTHI,POLU JYOTHI,29 Years,Female,EK__[20201118170123]_POLU JYOTHI____[]_.pdf,2020-11-18,NaT,27.3,100.00,Same patient; only one ECG candidate,Severe
...,...,...,...,...,...,...,...,...,...,...,...,...
241,KAKERI CHANDRAKALA,KAKERI CHANDRAKALA,KAKERI CHANDRAKALA,71 Years,Female,EK__[20201204102708]_KAKERI CHANDRAKALA____[]_...,2020-12-04,2020-04-12,80.5,100.00,Same patient; only one ECG candidate,Normal
242,THANDA PADMA,THANDA PADMA,PADMA THANDA,55 Years,Female,EK__[20201230131043]_THANDA PADMA____[]_.pdf,2020-12-30,NaT,80.5,100.00,Same patient; only one ECG candidate,Normal
243,GANTA RAO MURALA,GANTA RAO MURALA,GANTA RAO MURALA,62 Years,Male,EK__[20200914110459]_GANTA RAO MURALA____[]_.pdf,2020-09-14,NaT,81.4,100.00,Same patient; only one ECG candidate,Normal
244,MARUPAKA RAMESH,MARUPAKA RAMESH,MARUPAKA RAMESH,43 Years,Male,EK__[20200821193504]_MARUPAKA RAMESH____[]_.pdf,2020-08-21,NaT,82.9,100.00,Same patient; only one ECG candidate,Normal


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,UVANI,12.00,Female,NaT,16.2,No valid ECG name match
1,NARAYANA,70.00,Male,2020-09-08,17.6,No valid ECG name match
2,SADANANDAM,43.00,Male,NaT,19.5,No valid ECG name match
3,RAJU PULUKURI,41.00,Male,NaT,20.0,No valid ECG name match
4,"VUSIRIPALLY YADAMMA.65 YRS,",NaN,Female,NaT,20.1,No valid ECG name match
...,...,...,...,...,...,...
1007,BABY OF SWATHI,1.50 months,Female,2020-06-12,87.2,No valid ECG name match
1008,BABY OF SWAPNA,10.00 days,Female,NaT,88.3,No valid ECG name match
1009,BABY OF MOUNIKA,5.00 days,Female,NaT,88.6,No valid ECG name match
1010,RAJITHA,28.00,Female,2020-11-05,91.1,No valid ECG name match


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,VEERAMALLA MURALI,54 Years,Male,2020-08-19,EK__[20200819145757]_VEERAMALLA MURALI____[]_.pdf,Not selected for any Echo report
1,VARAYOGULA VIJAYA,47 Years,Female,2020-08-19,EK__[20200819150355]_VARAYOGULA VIJAYA____[]_.pdf,Not selected for any Echo report
2,VARAYOGULA VIJAYA,47 Years,Female,2020-08-19,EK__[20200819164059]_VARAYOGULA VIJAYA____[]_.pdf,Not selected for any Echo report
3,PERLA LAXMAN,47 Years,Male,2020-08-19,EK__[20200819180913]_PERLA LAXMAN____[]_.pdf,Not selected for any Echo report
4,RUDROJU GANESH,43 Years,Male,2020-08-19,EK__[20200819164943]_RUDROJU GANESH____[]_.pdf,Not selected for any Echo report
...,...,...,...,...,...,...
1891,VENISHETTY VIJAYKUMAR,38 Years,Male,2020-12-31,EK__[20201231123330]_VENISHETTY VIJAYKUMAR____...,Not selected for any Echo report
1892,NARRA SRINIVAS,42 Years,Male,2020-12-31,EK__[20201231123550]_NARRA SRINIVAS____[]_.pdf,Not selected for any Echo report
1893,SREEPATHI PRAMEELA,52 Years,Female,2020-12-31,EK__[20201231160726]_SREEPATHI PRAMEELA____[]_...,Not selected for any Echo report
1894,GANDLA VEERASWAMY,45 Years,Male,2020-12-31,EK__[20201231110110]_GANDLA VEERASWAMY____[]_.pdf,Not selected for any Echo report


In [ ]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

ecg = pd.read_excel("2020_Ecg1.xlsx", sheet_name="ECG Data")
echo = pd.read_excel("2020_data.xlsx", sheet_name="Sheet1")


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = re.sub(r"^SALIVA\s+", "", str(name).upper().strip())
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None

    m = re.search(r"\d+(?:\.\d+)?", str(age))

    return float(m.group()) if m else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ("M", "MALE"):
        return "M"

    if sex in ("F", "FEMALE"):
        return "F"

    return ""


def parse_date(series):
    return pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=True
    ).dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    m = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not m:
        return pd.NaT

    return pd.to_datetime(
        m.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.token_sort_ratio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.token_sort_ratio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])

ecg_date = find_col(
    ecg,
    ["STUDY DATE", "ECG DATE", "RECORDING DATE", "DATE"]
)

echo_date = find_col(
    echo,
    ["STUDY DATE", "STUDY_DATE", "DATE"]
)

ef_col = find_col(
    echo,
    ["EF (TEICH)", "EF_TEICH", "EF_TEICH_%", "EF (TEICH) %"]
)


required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

ecg["name_key"] = ecg[ecg_name].apply(clean_name)
echo["name_key"] = echo[echo_name].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name].apply(compact_name)
echo["compact_key"] = echo[echo_name].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index, dtype=object)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index, dtype=object)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date])
    if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date])
    if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(echo_row):
    name = echo_row["name_key"]
    compact = echo_row["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    queries = [name]

    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            ecg_names,
            scorer=fuzz.token_sort_ratio,
            limit=20,
            score_cutoff=80
        )

        candidates.update(idx for _, _, idx in found)

    valid = []

    for i in candidates:
        row = ecg.iloc[i]

        score = name_score(
            echo_row["name_key"],
            row["name_key"]
        )

        if score < 80:
            continue

        ea = echo_row["age_key"]
        ca = row["age_key"]

        if ea is not None and ca is not None and ea != ca:
            continue

        es = echo_row["sex_key"]
        cs = row["sex_key"]

        if es and cs and es != cs:
            continue

        ed = echo_row["match_date"]
        cd = row["match_date"]

        same_date = int(
            pd.notna(ed)
            and pd.notna(cd)
            and ed == cd
        )

        if pd.notna(ed) and pd.notna(cd):
            date_distance = abs((ed - cd).days)
        else:
            date_distance = 10**9

        exact = int(score == 100)

        valid.append(
            (same_date, exact, score, date_distance, i)
        )

    valid.sort(
        key=lambda x: (
            x[0],
            x[1],
            x[2],
            -x[3]
        ),
        reverse=True
    )

    return valid


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()

for j, h in echo.iterrows():
    candidates = find_candidates(h)

    if not candidates:
        continue

    same_date_candidates = [
        c for c in candidates if c[0] == 1
    ]

    if same_date_candidates:
        selected = same_date_candidates[0]
        reason = "Same patient and same date"

    else:
        selected = candidates[0]

        if len(candidates) == 1:
            reason = "Same patient; only one ECG candidate"
        elif selected[3] < 10**9:
            reason = "Same patient; closest available date"
        else:
            reason = "Same patient; dates unavailable"

    i = selected[4]

    e = ecg.iloc[i]
    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"
    else:
        sex = e["sex_key"] or h["sex_key"]

        if sex == "M" and ef > 52:
            lv = "Normal"
        elif sex == "F" and ef >= 54:
            lv = "Normal"
        elif ef >= 41:
            lv = "Mild"
        elif ef >= 31:
            lv = "Moderate"
        elif ef < 31:
            lv = "Severe"
        else:
            lv = "Unknown"

    matches.append({
        "Name": e["original_name"],
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": (
            e[ecg_age]
            if ecg_age
            else (h[echo_age] if echo_age else None)
        ),
        "Gender": (
            e[ecg_sex]
            if ecg_sex
            else (h[echo_sex] if echo_sex else "")
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": ef,
        "Match score": round(selected[2], 2),
        "Match reason": reason,
        "LV": lv
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


result = pd.DataFrame(matches)


unmatched_echo_rows = []

for j, h in echo.iterrows():
    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo name": h["original_name"],
            "Echo age": h[echo_age] if echo_age else None,
            "Echo gender": h[echo_sex] if echo_sex else "",
            "Echo study date": h["match_date"],
            "EF (Teich) %": h[ef_col],
            "Reason": "No valid ECG name match"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)


unmatched_ecg_rows = []

for i, e in ecg.iterrows():
    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "Not selected for any Echo report"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)


filename = (
    "2020_echo_ecg_matching_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )


print("\n========== MATCHING SUMMARY ==========")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo reports:", len(result))
print("Unmatched Echo reports:", len(unmatched_echo_df))
print("Unique ECG records selected:", len(matched_ecg_indices))
print("Saved file:", filename)


print("\n========== LV CLASSIFICATION SUMMARY ==========")

if not result.empty:
    counts = result["LV"].value_counts()

    print("Normal:", counts.get("Normal", 0))
    print("Mild:", counts.get("Mild", 0))
    print("Moderate:", counts.get("Moderate", 0))
    print("Severe:", counts.get("Severe", 0))
    print("Unknown:", counts.get("Unknown", 0))
    print("Total matched Echo reports:", len(result))

else:
    print("No matched Echo reports available.")
    print("Normal: 0")
    print("Mild: 0")
    print("Moderate: 0")
    print("Severe: 0")
    print("Unknown: 0")

print("===============================================")


display(result)
display(unmatched_echo_df)
display(unmatched_ecg_df)


========== MATCHING SUMMARY ==========
Echo records: 1258
ECG records: 2139
Matched Echo reports: 246
Unmatched Echo reports: 1012
Unique ECG records selected: 243
Saved file: 2020_echo_ecg_matching_20261029_161918.xlsx

========== LV CLASSIFICATION SUMMARY ==========
Normal: 218
Mild: 12
Moderate: 9
Severe: 7
Unknown: 0
Total matched Echo reports: 246


,Name,ECG name,Echo name,Age,Gender,ECG file name,ECG date,Echo study date,EF (Teich) %,Match score,Match reason,LV
0,SALIVA DILIPKUMAR,SALIVA DILIPKUMAR,DILIP KUMAR,46 Years,Male,EK__[20201219104107]_SALIVA DILIPKUMAR____[]_.pdf,2020-12-19,2020-06-05,13.4,100.00,Same patient; only one ECG candidate,Severe
1,ARISHAM NARSAIAH,ARISHAM NARSAIAH,NARSAIAH HARSHAM,60 Years,Male,EK__[20200914112110]_ARISHAM NARSAIAH____[]_.pdf,2020-09-14,2020-09-14,18.4,93.75,Same patient and same date,Severe
2,SHAGANTI VENKAYAMMA,SHAGANTI VENKAYAMMA,SHAGANTI VENKAYAMMA,65 Years,Female,EK__[20200824122232]_SHAGANTI VENKAYAMMA____[]...,2020-08-24,2020-08-24,18.8,100.00,Same patient and same date,Severe
3,DHEEKONDA KAMALAMMA,DHEEKONDA KAMALAMMA,DEEKONDA KAMALAMMA,75 Years,Female,EK__[20201003110753]_DHEEKONDA KAMALAMMA____[]...,2020-10-03,2020-10-03,26.1,97.30,Same patient and same date,Severe
4,POLU JYOTHI,POLU JYOTHI,POLU JYOTHI,29 Years,Female,EK__[20201118170123]_POLU JYOTHI____[]_.pdf,2020-11-18,2020-09-19,27.3,100.00,Same patient; only one ECG candidate,Severe
...,...,...,...,...,...,...,...,...,...,...,...,...
241,KAKERI CHANDRAKALA,KAKERI CHANDRAKALA,KAKERI CHANDRAKALA,71 Years,Female,EK__[20201204102708]_KAKERI CHANDRAKALA____[]_...,2020-12-04,2020-12-04,80.5,100.00,Same patient and same date,Normal
242,THANDA PADMA,THANDA PADMA,PADMA THANDA,55 Years,Female,EK__[20201230131043]_THANDA PADMA____[]_.pdf,2020-12-30,2020-12-30,80.5,100.00,Same patient and same date,Normal
243,GANTA RAO MURALA,GANTA RAO MURALA,GANTA RAO MURALA,62 Years,Male,EK__[20200914110459]_GANTA RAO MURALA____[]_.pdf,2020-09-14,2020-09-14,81.4,100.00,Same patient and same date,Normal
244,MARUPAKA RAMESH,MARUPAKA RAMESH,MARUPAKA RAMESH,43 Years,Male,EK__[20200821193504]_MARUPAKA RAMESH____[]_.pdf,2020-08-21,2020-08-23,82.9,100.00,Same patient; only one ECG candidate,Normal


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,UVANI,12.00,Female,2020-04-20,16.2,No valid ECG name match
1,NARAYANA,70.00,Male,2020-08-09,17.6,No valid ECG name match
2,SADANANDAM,43.00,Male,2020-06-16,19.5,No valid ECG name match
3,RAJU PULUKURI,41.00,Male,2020-08-26,20.0,No valid ECG name match
4,"VUSIRIPALLY YADAMMA.65 YRS,",NaN,Female,2020-05-21,20.1,No valid ECG name match
...,...,...,...,...,...,...
1007,BABY OF SWATHI,1.50 months,Female,2020-12-06,87.2,No valid ECG name match
1008,BABY OF SWAPNA,10.00 days,Female,2020-06-13,88.3,No valid ECG name match
1009,BABY OF MOUNIKA,5.00 days,Female,2020-08-27,88.6,No valid ECG name match
1010,RAJITHA,28.00,Female,2020-05-11,91.1,No valid ECG name match


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,VEERAMALLA MURALI,54 Years,Male,2020-08-19,EK__[20200819145757]_VEERAMALLA MURALI____[]_.pdf,Not selected for any Echo report
1,VARAYOGULA VIJAYA,47 Years,Female,2020-08-19,EK__[20200819150355]_VARAYOGULA VIJAYA____[]_.pdf,Not selected for any Echo report
2,VARAYOGULA VIJAYA,47 Years,Female,2020-08-19,EK__[20200819164059]_VARAYOGULA VIJAYA____[]_.pdf,Not selected for any Echo report
3,PERLA LAXMAN,47 Years,Male,2020-08-19,EK__[20200819180913]_PERLA LAXMAN____[]_.pdf,Not selected for any Echo report
4,RUDROJU GANESH,43 Years,Male,2020-08-19,EK__[20200819164943]_RUDROJU GANESH____[]_.pdf,Not selected for any Echo report
...,...,...,...,...,...,...
1891,VENISHETTY VIJAYKUMAR,38 Years,Male,2020-12-31,EK__[20201231123330]_VENISHETTY VIJAYKUMAR____...,Not selected for any Echo report
1892,NARRA SRINIVAS,42 Years,Male,2020-12-31,EK__[20201231123550]_NARRA SRINIVAS____[]_.pdf,Not selected for any Echo report
1893,SREEPATHI PRAMEELA,52 Years,Female,2020-12-31,EK__[20201231160726]_SREEPATHI PRAMEELA____[]_...,Not selected for any Echo report
1894,GANDLA VEERASWAMY,45 Years,Male,2020-12-31,EK__[20201231110110]_GANDLA VEERASWAMY____[]_.pdf,Not selected for any Echo report


In [20]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

echo = pd.read_excel("2020_data.xlsx", sheet_name="Sheet1")
ecg = pd.read_excel("2020_Ecg1.xlsx", sheet_name="ECG Data")

THRESHOLD = 50


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


echo = standardize_columns(echo)
ecg = standardize_columns(ecg)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    name = re.sub(r"^SALIVA\s+", "", name)
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None
    m = re.search(r"\d+(?:\.\d+)?", str(age))
    return float(m.group()) if m else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ("M", "MALE"):
        return "M"
    if sex in ("F", "FEMALE"):
        return "F"

    return ""


def parse_date(series):
    return pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=True
    ).dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    m = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not m:
        return pd.NaT

    return pd.to_datetime(
        m.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1),
        fuzz.WRatio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.WRatio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.WRatio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

ecg_date = find_col(ecg, ["DATE", "ECG DATE", "STUDY DATE"])
echo_date = find_col(echo, ["DATE", "STUDY DATE", "STUDY_DATE"])

file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])
ef_col = find_col(echo, ["EF (TEICH)", "EF_TEICH", "EF (TEICH) %"])


required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG date": ecg_date,
    "Echo date": echo_date,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

ecg["name_key"] = ecg[ecg_name].apply(clean_name)
echo["name_key"] = echo[echo_name].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name].apply(compact_name)
echo["compact_key"] = echo[echo_name].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index, dtype=object)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index, dtype=object)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = parse_date(ecg[ecg_date])
echo["match_date"] = parse_date(echo[echo_date])

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(echo_row):
    name = echo_row["name_key"]
    compact = echo_row["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    found = process.extract(
        name,
        ecg_names,
        scorer=fuzz.WRatio,
        limit=None,
        score_cutoff=THRESHOLD
    )

    candidates.update(idx for _, _, idx in found)

    if " " not in name and len(compact) >= 6:
        for k in range(3, len(compact) - 2):
            variant = compact[:k] + " " + compact[k:]

            found = process.extract(
                variant,
                ecg_names,
                scorer=fuzz.WRatio,
                limit=None,
                score_cutoff=THRESHOLD
            )

            candidates.update(idx for _, _, idx in found)

    valid = []

    for i in candidates:
        row = ecg.iloc[i]

        score = name_score(
            echo_row["original_name"],
            row["original_name"]
        )

        if score < THRESHOLD:
            continue

        ea = echo_row["age_key"]
        ca = row["age_key"]

        if ea is not None and ca is not None and ea != ca:
            continue

        es = echo_row["sex_key"]
        cs = row["sex_key"]

        if es and cs and es != cs:
            continue

        ed = echo_row["match_date"]
        cd = row["match_date"]

        same_date = int(
            pd.notna(ed)
            and pd.notna(cd)
            and ed == cd
        )

        date_distance = (
            abs((ed - cd).days)
            if pd.notna(ed) and pd.notna(cd)
            else 10**9
        )

        valid.append(
            (same_date, score, date_distance, i)
        )

    valid.sort(
        key=lambda x: (
            x[0],
            x[1],
            -x[2]
        ),
        reverse=True
    )

    return valid


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()

for j, h in echo.iterrows():
    candidates = find_candidates(h)

    if not candidates:
        continue

    same_date_candidates = [
        c for c in candidates if c[0] == 1
    ]

    if same_date_candidates:
        selected = same_date_candidates[0]
        reason = "Same patient and same date"
    else:
        selected = candidates[0]

        if len(candidates) == 1:
            reason = "Name match; only one ECG candidate"
        elif selected[2] < 10**9:
            reason = "Name match; closest available date"
        else:
            reason = "Name match; dates unavailable"

    i = selected[3]

    e = ecg.iloc[i]
    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"
    else:
        sex = e["sex_key"] or h["sex_key"]

        if sex == "M" and ef > 52:
            lv = "Normal"
        elif sex == "F" and ef >= 54:
            lv = "Normal"
        elif ef >= 41:
            lv = "Mild"
        elif ef >= 31:
            lv = "Moderate"
        elif ef < 31:
            lv = "Severe"
        else:
            lv = "Unknown"

    matches.append({
        "Name": e["original_name"],
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": (
            e[ecg_age]
            if ecg_age
            else (h[echo_age] if echo_age else None)
        ),
        "Gender": (
            e[ecg_sex]
            if ecg_sex
            else (h[echo_sex] if echo_sex else "")
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": ef,
        "Match score": round(selected[1], 2),
        "Match reason": reason,
        "LV": lv
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


result = pd.DataFrame(matches)

unmatched_echo_rows = []

for j, h in echo.iterrows():
    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo name": h["original_name"],
            "Echo age": h[echo_age] if echo_age else None,
            "Echo gender": h[echo_sex] if echo_sex else "",
            "Echo study date": h["match_date"],
            "EF (Teich) %": h[ef_col],
            "Reason": "No valid name match at 50% threshold"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)

unmatched_ecg_rows = []

for i, e in ecg.iterrows():
    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "Not selected for any Echo report"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)

filename = (
    "2020_echo_ecg_matching_50percent_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("\n========== MATCHING SUMMARY ==========")
print("Name threshold:", THRESHOLD, "%")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo reports:", len(result))
print("Unmatched Echo reports:", len(unmatched_echo_df))
print("Unique ECG records selected:", len(matched_ecg_indices))
print("Saved file:", filename)

print("\n========== LV CLASSIFICATION SUMMARY ==========")

if not result.empty:
    counts = result["LV"].value_counts()

    print("Normal:", counts.get("Normal", 0))
    print("Mild:", counts.get("Mild", 0))
    print("Moderate:", counts.get("Moderate", 0))
    print("Severe:", counts.get("Severe", 0))
    print("Unknown:", counts.get("Unknown", 0))
    print("Total matched Echo reports:", len(result))
else:
    print("No matched Echo reports available.")
    print("Normal: 0")
    print("Mild: 0")
    print("Moderate: 0")
    print("Severe: 0")
    print("Unknown: 0")

print("======================================")

display(result)
display(unmatched_echo_df)
display(unmatched_ecg_df)


========== MATCHING SUMMARY ==========
Name threshold: 50 %
Echo records: 1258
ECG records: 2139
Matched Echo reports: 968
Unmatched Echo reports: 290
Unique ECG records selected: 738
Saved file: 2020_echo_ecg_matching_50percent_20261029_162620.xlsx

========== LV CLASSIFICATION SUMMARY ==========
Normal: 846
Mild: 50
Moderate: 42
Severe: 30
Unknown: 0
Total matched Echo reports: 968


,Name,ECG name,Echo name,Age,Gender,ECG file name,ECG date,Echo study date,EF (Teich) %,Match score,Match reason,LV
0,SALIVA DILIPKUMAR,SALIVA DILIPKUMAR,DILIP KUMAR,46 Years,Male,EK__[20201219104107]_SALIVA DILIPKUMAR____[]_.pdf,2020-12-19,2020-06-05,13.4,100.00,Name match; closest available date,Severe
1,BOLLAMPALLI JANARDHANRAO,BOLLAMPALLI JANARDHANRAO,NARAYANA,70 Years,Male,EK__[20201009155051]_BOLLAMPALLI JANARDHANRAO_...,2020-10-09,2020-08-09,17.6,60.00,Name match; closest available date,Severe
2,ARISHAM NARSAIAH,ARISHAM NARSAIAH,NARSAIAH HARSHAM,60 Years,Male,EK__[20200914112110]_ARISHAM NARSAIAH____[]_.pdf,2020-09-14,2020-09-14,18.4,93.75,Same patient and same date,Severe
3,SHAGANTI VENKAYAMMA,SHAGANTI VENKAYAMMA,SHAGANTI VENKAYAMMA,65 Years,Female,EK__[20200824122232]_SHAGANTI VENKAYAMMA____[]...,2020-08-24,2020-08-24,18.8,100.00,Same patient and same date,Severe
4,GADDAM VENUGOPAL,GADDAM VENUGOPAL,SADANANDAM,43 Years,Male,EK__[20201121201327]_GADDAM VENUGOPAL____[]_.pdf,2020-11-21,2020-06-16,19.5,56.25,Name match; closest available date,Severe
...,...,...,...,...,...,...,...,...,...,...,...,...
963,EDA GIRIDHARREDDY,EDA GIRIDHARREDDY,KISARA VIDWAN REDDY,60 Years,Male,EK__[20200914102610]_EDA GIRIDHARREDDY____[]_.pdf,2020-09-14,2020-06-26,85.0,60.61,Name match; closest available date,Normal
964,VOMA RAJYALAXMI,VOMA RAJYALAXMI,G BHAGYALAKSHMI,62 Years,Female,EK__[20200917145425]_VOMA RAJYALAXMI____[]_.pdf,2020-09-17,2020-05-28,85.7,53.33,Name match; closest available date,Normal
965,SYED AZGARALI,SYED AZGARALI,SYED ASGHAR ALI,38 Years,Male,EK__[20201203104227]_SYED AZGARALI____[]_.pdf,2020-12-03,2020-12-03,85.9,88.00,Same patient and same date,Normal
966,POTHULA RAMESH REDDY,POTHULA RAMESH REDDY,KRISHNA REDDY,61 Years,Male,EK__[20200915105826]_POTHULA RAMESH REDDY____[...,2020-09-15,2020-07-02,86.4,85.50,Name match; closest available date,Normal


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,UVANI,12.00,Female,2020-04-20,16.2,No valid name match at 50% threshold
1,"VUSIRIPALLY YADAMMA.65 YRS,",NaN,Female,2020-05-21,20.1,No valid name match at 50% threshold
2,RAMESH REDDY PALLE,62.00,Male,2020-12-29,25.3,No valid name match at 50% threshold
3,LAXMI BOSHETTI,56.00,Female,2020-09-20,26.4,No valid name match at 50% threshold
4,RUKMINI,76.00,Female,2020-06-03,29.7,No valid name match at 50% threshold
...,...,...,...,...,...,...
285,BABY OF RAJESHWARI,4.00 months,Female,2020-12-28,87.0,No valid name match at 50% threshold
286,BABY OF SWATHI,1.50 months,Female,2020-12-06,87.2,No valid name match at 50% threshold
287,BABY OF SWAPNA,10.00 days,Female,2020-06-13,88.3,No valid name match at 50% threshold
288,BABY OF MOUNIKA,5.00 days,Female,2020-08-27,88.6,No valid name match at 50% threshold


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,VEERAMALLA MURALI,54 Years,Male,2020-08-19,EK__[20200819145757]_VEERAMALLA MURALI____[]_.pdf,Not selected for any Echo report
1,VARAYOGULA VIJAYA,47 Years,Female,2020-08-19,EK__[20200819150355]_VARAYOGULA VIJAYA____[]_.pdf,Not selected for any Echo report
2,VARAYOGULA VIJAYA,47 Years,Female,2020-08-19,EK__[20200819164059]_VARAYOGULA VIJAYA____[]_.pdf,Not selected for any Echo report
3,PERLA LAXMAN,47 Years,Male,2020-08-19,EK__[20200819180913]_PERLA LAXMAN____[]_.pdf,Not selected for any Echo report
4,RUDROJU GANESH,43 Years,Male,2020-08-19,EK__[20200819164943]_RUDROJU GANESH____[]_.pdf,Not selected for any Echo report
...,...,...,...,...,...,...
1396,VENISHETTY VIJAYKUMAR,38 Years,Male,2020-12-31,EK__[20201231123330]_VENISHETTY VIJAYKUMAR____...,Not selected for any Echo report
1397,NARRA SRINIVAS,42 Years,Male,2020-12-31,EK__[20201231123550]_NARRA SRINIVAS____[]_.pdf,Not selected for any Echo report
1398,SREEPATHI PRAMEELA,52 Years,Female,2020-12-31,EK__[20201231160726]_SREEPATHI PRAMEELA____[]_...,Not selected for any Echo report
1399,GANDLA VEERASWAMY,45 Years,Male,2020-12-31,EK__[20201231110110]_GANDLA VEERASWAMY____[]_.pdf,Not selected for any Echo report


In [24]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

echo = pd.read_excel("2024echo_reports.xlsx", sheet_name="2024")
ecg = pd.read_excel("2024_Ecg1.xlsx", sheet_name="ECG Data")

THRESHOLD = 70


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


echo = standardize_columns(echo)
ecg = standardize_columns(ecg)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    name = re.sub(r"^SALIVA\s+", "", name)
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None
    m = re.search(r"\d+(?:\.\d+)?", str(age))
    return float(m.group()) if m else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""

    sex = str(sex).strip().upper()

    if sex in ("M", "MALE"):
        return "M"
    if sex in ("F", "FEMALE"):
        return "F"

    return ""


def parse_date(series):
    return pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=True
    ).dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    m = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not m:
        return pd.NaT

    return pd.to_datetime(
        m.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1),
        fuzz.WRatio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.WRatio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.WRatio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

ecg_date = find_col(ecg, ["DATE", "ECG DATE", "STUDY DATE"])
echo_date = find_col(echo, ["DATE", "STUDY DATE", "STUDY_DATE"])

file_col = find_col(ecg, ["FILE NAME", "FILENAME", "FILE"])
ef_col = find_col(echo, ["EF (TEICH)", "EF_TEICH", "EF (TEICH) %"])


required = {
    "ECG name": ecg_name,
    "Echo name": echo_name,
    "ECG date": ecg_date,
    "Echo date": echo_date,
    "ECG filename": file_col,
    "Echo EF": ef_col
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

ecg["name_key"] = ecg[ecg_name].apply(clean_name)
echo["name_key"] = echo[echo_name].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name].apply(compact_name)
echo["compact_key"] = echo[echo_name].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index, dtype=object)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index, dtype=object)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = parse_date(ecg[ecg_date])
echo["match_date"] = parse_date(echo[echo_date])

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(echo_row):
    name = echo_row["name_key"]
    compact = echo_row["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    found = process.extract(
        name,
        ecg_names,
        scorer=fuzz.WRatio,
        limit=None,
        score_cutoff=THRESHOLD
    )

    candidates.update(idx for _, _, idx in found)

    if " " not in name and len(compact) >= 6:
        for k in range(3, len(compact) - 2):
            variant = compact[:k] + " " + compact[k:]

            found = process.extract(
                variant,
                ecg_names,
                scorer=fuzz.WRatio,
                limit=None,
                score_cutoff=THRESHOLD
            )

            candidates.update(idx for _, _, idx in found)

    valid = []

    for i in candidates:
        row = ecg.iloc[i]

        score = name_score(
            echo_row["original_name"],
            row["original_name"]
        )

        if score < THRESHOLD:
            continue

        ea = echo_row["age_key"]
        ca = row["age_key"]

        if ea is not None and ca is not None and ea != ca:
            continue

        es = echo_row["sex_key"]
        cs = row["sex_key"]

        if es and cs and es != cs:
            continue

        ed = echo_row["match_date"]
        cd = row["match_date"]

        same_date = int(
            pd.notna(ed)
            and pd.notna(cd)
            and ed == cd
        )

        date_distance = (
            abs((ed - cd).days)
            if pd.notna(ed) and pd.notna(cd)
            else 10**9
        )

        valid.append(
            (same_date, score, date_distance, i)
        )

    valid.sort(
        key=lambda x: (
            x[0],
            x[1],
            -x[2]
        ),
        reverse=True
    )

    return valid


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()

for j, h in echo.iterrows():
    candidates = find_candidates(h)

    if not candidates:
        continue

    same_date_candidates = [
        c for c in candidates if c[0] == 1
    ]

    if same_date_candidates:
        selected = same_date_candidates[0]
        reason = "Same patient and same date"
    else:
        selected = candidates[0]

        if len(candidates) == 1:
            reason = "Name match; only one ECG candidate"
        elif selected[2] < 10**9:
            reason = "Name match; closest available date"
        else:
            reason = "Name match; dates unavailable"

    i = selected[3]

    e = ecg.iloc[i]
    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"
    else:
        sex = e["sex_key"] or h["sex_key"]

        if sex == "M" and ef > 52:
            lv = "Normal"
        elif sex == "F" and ef >= 54:
            lv = "Normal"
        elif ef >= 41:
            lv = "Mild"
        elif ef >= 31:
            lv = "Moderate"
        elif ef < 31:
            lv = "Severe"
        else:
            lv = "Unknown"

    matches.append({
        "Name": e["original_name"],
        "ECG name": e["original_name"],
        "Echo name": h["original_name"],
        "Age": (
            e[ecg_age]
            if ecg_age
            else (h[echo_age] if echo_age else None)
        ),
        "Gender": (
            e[ecg_sex]
            if ecg_sex
            else (h[echo_sex] if echo_sex else "")
        ),
        "ECG file name": e[file_col],
        "ECG date": e["match_date"],
        "Echo study date": h["match_date"],
        "EF (Teich) %": ef,
        "Match score": round(selected[1], 2),
        "Match reason": reason,
        "LV": lv
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


result = pd.DataFrame(matches)

unmatched_echo_rows = []

for j, h in echo.iterrows():
    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo name": h["original_name"],
            "Echo age": h[echo_age] if echo_age else None,
            "Echo gender": h[echo_sex] if echo_sex else "",
            "Echo study date": h["match_date"],
            "EF (Teich) %": h[ef_col],
            "Reason": "No valid name match at 50% threshold"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)

unmatched_ecg_rows = []

for i, e in ecg.iterrows():
    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG name": e["original_name"],
            "ECG age": e[ecg_age] if ecg_age else None,
            "ECG gender": e[ecg_sex] if ecg_sex else "",
            "ECG date": e["match_date"],
            "ECG file name": e[file_col],
            "Reason": "Not selected for any Echo report"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)

filename = (
    "2020_echo_ecg_matching_50percent_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

print("\n========== MATCHING SUMMARY ==========")
print("Name threshold:", THRESHOLD, "%")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo reports:", len(result))
print("Unmatched Echo reports:", len(unmatched_echo_df))
print("Unique ECG records selected:", len(matched_ecg_indices))
print("Saved file:", filename)

print("\n========== LV CLASSIFICATION SUMMARY ==========")

if not result.empty:
    counts = result["LV"].value_counts()

    print("Normal:", counts.get("Normal", 0))
    print("Mild:", counts.get("Mild", 0))
    print("Moderate:", counts.get("Moderate", 0))
    print("Severe:", counts.get("Severe", 0))
    print("Unknown:", counts.get("Unknown", 0))
    print("Total matched Echo reports:", len(result))
else:
    print("No matched Echo reports available.")
    print("Normal: 0")
    print("Mild: 0")
    print("Moderate: 0")
    print("Severe: 0")
    print("Unknown: 0")

print("======================================")

display(result)
display(unmatched_echo_df)
display(unmatched_ecg_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\2845748621.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(



========== MATCHING SUMMARY ==========
Name threshold: 70 %
Echo records: 1639
ECG records: 4035
Matched Echo reports: 0
Unmatched Echo reports: 1639
Unique ECG records selected: 0
Saved file: 2020_echo_ecg_matching_50percent_20261029_164413.xlsx

========== LV CLASSIFICATION SUMMARY ==========
No matched Echo reports available.
Normal: 0
Mild: 0
Moderate: 0
Severe: 0
Unknown: 0


""


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,GUDURU VINAYCHANDRA,None,Male,2024-10-07,82.2,No valid name match at 50% threshold
1,GAYATHRI,None,Female,2024-10-01,69.5,No valid name match at 50% threshold
2,METTU AADRIKA RAJ,None,Female,2024-08-31,65.7,No valid name match at 50% threshold
3,MANDALA RISHIK,None,Male,2024-10-04,75.1,No valid name match at 50% threshold
4,SHOURYATEJ,None,Male,2024-03-01,59.9,No valid name match at 50% threshold
...,...,...,...,...,...,...
1634,VINAYCHANDRA,None,Male,2024-04-12,56.1,No valid name match at 50% threshold
1635,NERELLA SIRICHANDANA,None,Female,2024-01-29,59.3,No valid name match at 50% threshold
1636,THALAGAMPA BHARATH,None,Male,2024-11-21,76.0,No valid name match at 50% threshold
1637,ARUKONDA SARITHA,None,Female,2024-01-29,62.3,No valid name match at 50% threshold


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,GANGU VANISREE,56 Years,Female,2024-01-01,EK__[20240101202919]_GANGU VANISREE____[]_.pdf,Not selected for any Echo report
1,THOTA TULASIBHAI,85 Years,Female,2024-01-01,EK__[20240101191704]_THOTA TULASIBHAI____[]_.pdf,Not selected for any Echo report
2,JANNU CHIRANJEEVI,41 Years,Male,2024-01-01,EK__[20240101134404]_JANNU CHIRANJEEVI____[]_...,Not selected for any Echo report
3,S UDAY KISHORE,38 Years,Male,2024-01-01,EK__[20240101120955]_S UDAY KISHORE____[]_.pdf,Not selected for any Echo report
4,RONDLA SRIKARREDDY,53 Years,Male,2024-01-01,EK__[20240101112512]_RONDLA SRIKARREDDY____[]...,Not selected for any Echo report
...,...,...,...,...,...,...
4030,THALLAPELLY DEVENAMMA,65 Years,Female,2024-12-31,EK__[20241231180131]_THALLAPELLY DEVENAMMA____...,Not selected for any Echo report
4031,SIRIGIRI PRINCY,34 Years,Female,2024-12-31,EK__[20241231120848]_SIRIGIRI PRINCY____[]_.pdf,Not selected for any Echo report
4032,BAIRI RAJAMALLU,85 Years,Male,2024-12-31,EK__[20241231120400]_BAIRI RAJAMALLU____[]_.pdf,Not selected for any Echo report
4033,CHITYALA SRI HARI,66 Years,Male,2024-12-31,EK__[20241231112422]_CHITYALA SRI HARI____[]_.pdf,Not selected for any Echo report


C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\1345211121.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(



========== MATCHING SUMMARY ==========
Name threshold: 50 %
Echo records: 1639
ECG records: 4035
Matched Echo reports: 0
Unmatched Echo reports: 1639
Unique ECG records selected: 0
Saved file: 2020_echo_ecg_matching_50percent_20261029_164152.xlsx

========== LV CLASSIFICATION SUMMARY ==========
No matched Echo reports available.
Normal: 0
Mild: 0
Moderate: 0
Severe: 0
Unknown: 0


""


,Echo name,Echo age,Echo gender,Echo study date,EF (Teich) %,Reason
0,GUDURU VINAYCHANDRA,None,Male,2024-10-07,82.2,No valid name match at 50% threshold
1,GAYATHRI,None,Female,2024-10-01,69.5,No valid name match at 50% threshold
2,METTU AADRIKA RAJ,None,Female,2024-08-31,65.7,No valid name match at 50% threshold
3,MANDALA RISHIK,None,Male,2024-10-04,75.1,No valid name match at 50% threshold
4,SHOURYATEJ,None,Male,2024-03-01,59.9,No valid name match at 50% threshold
...,...,...,...,...,...,...
1634,VINAYCHANDRA,None,Male,2024-04-12,56.1,No valid name match at 50% threshold
1635,NERELLA SIRICHANDANA,None,Female,2024-01-29,59.3,No valid name match at 50% threshold
1636,THALAGAMPA BHARATH,None,Male,2024-11-21,76.0,No valid name match at 50% threshold
1637,ARUKONDA SARITHA,None,Female,2024-01-29,62.3,No valid name match at 50% threshold


,ECG name,ECG age,ECG gender,ECG date,ECG file name,Reason
0,GANGU VANISREE,56 Years,Female,2024-01-01,EK__[20240101202919]_GANGU VANISREE____[]_.pdf,Not selected for any Echo report
1,THOTA TULASIBHAI,85 Years,Female,2024-01-01,EK__[20240101191704]_THOTA TULASIBHAI____[]_.pdf,Not selected for any Echo report
2,JANNU CHIRANJEEVI,41 Years,Male,2024-01-01,EK__[20240101134404]_JANNU CHIRANJEEVI____[]_...,Not selected for any Echo report
3,S UDAY KISHORE,38 Years,Male,2024-01-01,EK__[20240101120955]_S UDAY KISHORE____[]_.pdf,Not selected for any Echo report
4,RONDLA SRIKARREDDY,53 Years,Male,2024-01-01,EK__[20240101112512]_RONDLA SRIKARREDDY____[]...,Not selected for any Echo report
...,...,...,...,...,...,...
4030,THALLAPELLY DEVENAMMA,65 Years,Female,2024-12-31,EK__[20241231180131]_THALLAPELLY DEVENAMMA____...,Not selected for any Echo report
4031,SIRIGIRI PRINCY,34 Years,Female,2024-12-31,EK__[20241231120848]_SIRIGIRI PRINCY____[]_.pdf,Not selected for any Echo report
4032,BAIRI RAJAMALLU,85 Years,Male,2024-12-31,EK__[20241231120400]_BAIRI RAJAMALLU____[]_.pdf,Not selected for any Echo report
4033,CHITYALA SRI HARI,66 Years,Male,2024-12-31,EK__[20241231112422]_CHITYALA SRI HARI____[]_.pdf,Not selected for any Echo report


In [25]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

THRESHOLD = 70

echo = pd.read_excel("2024echo_reports.xlsx", sheet_name="2024")
ecg = pd.read_excel("2024_Ecg1.xlsx", sheet_name="ECG Data")


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c).replace("\u00a0", " ")).strip().upper()
        for c in df.columns
    ]
    return df


echo = standardize_columns(echo)
ecg = standardize_columns(ecg)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    name = re.sub(r"^SALIVA\s+", "", name)
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_age(age):
    if pd.isna(age):
        return None
    m = re.search(r"\d+(?:\.\d+)?", str(age))
    return float(m.group()) if m else None


def clean_sex(sex):
    if pd.isna(sex):
        return ""
    sex = str(sex).strip().upper()
    if sex in ("M", "MALE"):
        return "M"
    if sex in ("F", "FEMALE"):
        return "F"
    return ""


def parse_date(series):
    return pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=True
    ).dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    m = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not m:
        return pd.NaT

    return pd.to_datetime(
        m.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)
    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1),
        fuzz.WRatio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.WRatio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.WRatio(a1, variant))

    return max(scores)


echo_name = find_col(echo, ["NAME", "PATIENT NAME"])
ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])

echo_date = find_col(echo, ["STUDY_DATE", "STUDY DATE", "DATE"])
ecg_date = find_col(ecg, ["DATE", "ECG DATE", "STUDY DATE"])

echo_age = find_col(echo, ["AGE", "PATIENT AGE"])
ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])

echo_sex = find_col(echo, ["GENDER", "SEX"])
ecg_sex = find_col(ecg, ["SEX", "GENDER"])

ecg_file = find_col(
    ecg,
    ["FILE NAME", "FILENAME", "PDF_PATH", "FILE"]
)

echo_ef_cols = [
    c for c in ["EF_TEICH_%", "EF_TEICH"]
    if c in echo.columns
]

required = {
    "Echo name": echo_name,
    "ECG name": ecg_name,
    "Echo date": echo_date,
    "ECG date": ecg_date,
    "ECG filename": ecg_file
}

missing = [k for k, v in required.items() if v is None]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Echo columns: {echo.columns.tolist()}\n"
        f"ECG columns: {ecg.columns.tolist()}"
    )

if not echo_ef_cols:
    raise ValueError(
        "No EF_TEICH_% or EF_TEICH column found in the Echo sheet."
    )

echo = echo.copy().reset_index(drop=True)
ecg = ecg.copy().reset_index(drop=True)

echo["original_name"] = echo[echo_name]
ecg["original_name"] = ecg[ecg_name]

echo["name_key"] = echo[echo_name].apply(clean_name)
ecg["name_key"] = ecg[ecg_name].apply(clean_name)

echo["compact_key"] = echo[echo_name].apply(compact_name)
ecg["compact_key"] = ecg[ecg_name].apply(compact_name)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age else pd.Series(None, index=echo.index, dtype=object)
)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age else pd.Series(None, index=ecg.index, dtype=object)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex else pd.Series("", index=echo.index)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex else pd.Series("", index=ecg.index)
)

echo["match_date"] = parse_date(echo[echo_date])
ecg["match_date"] = parse_date(ecg[ecg_date])

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[ecg_file].apply(filename_date)
)

for col in echo_ef_cols:
    echo[col] = pd.to_numeric(echo[col], errors="coerce")

echo["EF_used"] = echo[echo_ef_cols].bfill(axis=1).iloc[:, 0]

echo["EF_used"] = echo["EF_used"].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

echo = echo[echo["name_key"] != ""].reset_index(drop=True)
ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)

ecg_names = ecg["name_key"].tolist()
ecg_compact_names = ecg["compact_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(h):
    name = h["name_key"]
    compact = h["compact_key"]

    candidates = set()
    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    found = process.extract(
        name,
        ecg_names,
        scorer=fuzz.WRatio,
        limit=None,
        score_cutoff=THRESHOLD
    )

    candidates.update(idx for _, _, idx in found)

    if " " not in name and len(compact) >= 6:
        for k in range(3, len(compact) - 2):
            variant = compact[:k] + " " + compact[k:]

            found = process.extract(
                variant,
                ecg_names,
                scorer=fuzz.WRatio,
                limit=None,
                score_cutoff=THRESHOLD
            )

            candidates.update(idx for _, _, idx in found)

    valid = []

    for i in candidates:
        e = ecg.iloc[i]

        score = name_score(
            h["original_name"],
            e["original_name"]
        )

        if score < THRESHOLD:
            continue

        ea = h["age_key"]
        ca = e["age_key"]

        if ea is not None and ca is not None and ea != ca:
            continue

        es = h["sex_key"]
        cs = e["sex_key"]

        if es and cs and es != cs:
            continue

        ed = h["match_date"]
        cd = e["match_date"]

        same_date = int(
            pd.notna(ed) and pd.notna(cd) and ed == cd
        )

        date_distance = (
            abs((ed - cd).days)
            if pd.notna(ed) and pd.notna(cd)
            else 10**9
        )

        valid.append((same_date, score, date_distance, i))

    valid.sort(
        key=lambda x: (x[0], x[1], -x[2]),
        reverse=True
    )

    return valid


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()

total_echo = len(echo)

for j, h in echo.iterrows():
    candidates = find_candidates(h)

    if candidates:
        same_date_candidates = [
            c for c in candidates if c[0] == 1
        ]

        selected = (
            same_date_candidates[0]
            if same_date_candidates
            else candidates[0]
        )

        i = selected[3]
        e = ecg.iloc[i]
        score = selected[1]
        ef = h["EF_used"]

        if selected[0]:
            reason = "Same patient and same date"
        elif selected[2] < 10**9:
            reason = "Name match; closest available date"
        else:
            reason = "Name match; dates unavailable"

        if pd.isna(ef):
            lv = "Unknown"
        else:
            sex = e["sex_key"] or h["sex_key"]

            if sex == "M" and ef > 52:
                lv = "Normal"
            elif sex == "F" and ef >= 54:
                lv = "Normal"
            elif ef >= 41:
                lv = "Mild"
            elif ef >= 31:
                lv = "Moderate"
            elif ef < 31:
                lv = "Severe"
            else:
                lv = "Unknown"

        record = {
            "Name": e["original_name"],
            "Echo Name": h["original_name"],
            "ECG Name": e["original_name"],
            "Match Score (%)": round(score, 2),
            "Match Reason": reason,
            "Echo Study Date": h["match_date"],
            "ECG Date": e["match_date"],
            "EF Used (%)": ef,
            "LV": lv
        }

        for col in echo.columns:
            if col not in [
                "original_name", "name_key", "compact_key",
                "age_key", "sex_key", "match_date", "EF_used"
            ]:
                record[f"Echo_{col}"] = h[col]

        for col in ecg.columns:
            if col not in [
                "original_name", "name_key", "compact_key",
                "age_key", "sex_key", "match_date"
            ]:
                record[f"ECG_{col}"] = e[col]

        matches.append(record)
        matched_echo_indices.add(j)
        matched_ecg_indices.add(i)

    if (j + 1) % 100 == 0 or j + 1 == total_echo:
        print(f"Processed {j + 1}/{total_echo} Echo records")


result = pd.DataFrame(matches)

unmatched_echo_rows = []

for j, h in echo.iterrows():
    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo Name": h["original_name"],
            "Echo Gender": h[echo_sex] if echo_sex else "",
            "Echo Study Date": h["match_date"],
            "EF Used (%)": h["EF_used"],
            "Reason": "No valid match at 70% threshold"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)

unmatched_ecg_rows = []

for i, e in ecg.iterrows():
    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG Name": e["original_name"],
            "ECG Age": e[ecg_age] if ecg_age else None,
            "ECG Sex": e[ecg_sex] if ecg_sex else "",
            "ECG Date": e["match_date"],
            "ECG File": e[ecg_file],
            "Reason": "Not selected for any Echo report"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)

filename = (
    "2024_echo_ecg_matching_70percent_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="Matched Records", index=False)
    unmatched_echo_df.to_excel(writer, sheet_name="Unmatched Echo", index=False)
    unmatched_ecg_df.to_excel(writer, sheet_name="Unmatched ECG", index=False)

print("\n========== MATCHING SUMMARY ==========")
print("Name threshold:", THRESHOLD, "%")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo reports:", len(result))
print("Unmatched Echo reports:", len(unmatched_echo_df))
print("Unique ECG records selected:", len(matched_ecg_indices))
print("Saved file:", filename)

print("\n========== LV CLASSIFICATION SUMMARY ==========")

if not result.empty:
    counts = result["LV"].value_counts()
    print("Normal:", counts.get("Normal", 0))
    print("Mild:", counts.get("Mild", 0))
    print("Moderate:", counts.get("Moderate", 0))
    print("Severe:", counts.get("Severe", 0))
    print("Unknown:", counts.get("Unknown", 0))
else:
    print("No matched Echo reports available.")

print("======================================")

display(result)
display(unmatched_echo_df)
display(unmatched_ecg_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\979464928.py:61: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(


Processed 100/1639 Echo records
Processed 200/1639 Echo records
Processed 300/1639 Echo records
Processed 400/1639 Echo records
Processed 500/1639 Echo records
Processed 600/1639 Echo records
Processed 700/1639 Echo records
Processed 800/1639 Echo records
Processed 900/1639 Echo records
Processed 1000/1639 Echo records
Processed 1100/1639 Echo records
Processed 1200/1639 Echo records
Processed 1300/1639 Echo records
Processed 1400/1639 Echo records
Processed 1500/1639 Echo records
Processed 1600/1639 Echo records
Processed 1639/1639 Echo records

========== MATCHING SUMMARY ==========
Name threshold: 70 %
Echo records: 1639
ECG records: 4035
Matched Echo reports: 0
Unmatched Echo reports: 1639
Unique ECG records selected: 0
Saved file: 2024_echo_ecg_matching_70percent_20261029_164812.xlsx

========== LV CLASSIFICATION SUMMARY ==========
No matched Echo reports available.


""


,Echo Name,Echo Gender,Echo Study Date,EF Used (%),Reason
0,GUDURU VINAYCHANDRA,Male,2024-10-07,82.2,No valid match at 70% threshold
1,GAYATHRI,Female,2024-10-01,69.5,No valid match at 70% threshold
2,METTU AADRIKA RAJ,Female,2024-08-31,65.7,No valid match at 70% threshold
3,MANDALA RISHIK,Male,2024-10-04,75.1,No valid match at 70% threshold
4,SHOURYATEJ,Male,2024-03-01,59.9,No valid match at 70% threshold
...,...,...,...,...,...
1634,VINAYCHANDRA,Male,2024-04-12,56.1,No valid match at 70% threshold
1635,NERELLA SIRICHANDANA,Female,2024-01-29,59.3,No valid match at 70% threshold
1636,THALAGAMPA BHARATH,Male,2024-11-21,76.0,No valid match at 70% threshold
1637,ARUKONDA SARITHA,Female,2024-01-29,62.3,No valid match at 70% threshold


,ECG Name,ECG Age,ECG Sex,ECG Date,ECG File,Reason
0,GANGU VANISREE,56 Years,Female,2024-01-01,EK__[20240101202919]_GANGU VANISREE____[]_.pdf,Not selected for any Echo report
1,THOTA TULASIBHAI,85 Years,Female,2024-01-01,EK__[20240101191704]_THOTA TULASIBHAI____[]_.pdf,Not selected for any Echo report
2,JANNU CHIRANJEEVI,41 Years,Male,2024-01-01,EK__[20240101134404]_JANNU CHIRANJEEVI____[]_...,Not selected for any Echo report
3,S UDAY KISHORE,38 Years,Male,2024-01-01,EK__[20240101120955]_S UDAY KISHORE____[]_.pdf,Not selected for any Echo report
4,RONDLA SRIKARREDDY,53 Years,Male,2024-01-01,EK__[20240101112512]_RONDLA SRIKARREDDY____[]...,Not selected for any Echo report
...,...,...,...,...,...,...
4030,THALLAPELLY DEVENAMMA,65 Years,Female,2024-12-31,EK__[20241231180131]_THALLAPELLY DEVENAMMA____...,Not selected for any Echo report
4031,SIRIGIRI PRINCY,34 Years,Female,2024-12-31,EK__[20241231120848]_SIRIGIRI PRINCY____[]_.pdf,Not selected for any Echo report
4032,BAIRI RAJAMALLU,85 Years,Male,2024-12-31,EK__[20241231120400]_BAIRI RAJAMALLU____[]_.pdf,Not selected for any Echo report
4033,CHITYALA SRI HARI,66 Years,Male,2024-12-31,EK__[20241231112422]_CHITYALA SRI HARI____[]_.pdf,Not selected for any Echo report


In [26]:
import pandas as pd
import re
import os
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

echo_file = "2024echo_reports.xlsx"
ecg_file = "2024_Ecg1.xlsx"

echo = pd.read_excel(echo_file, sheet_name="2024")
ecg = pd.read_excel(ecg_file, sheet_name="ECG Data")

THRESHOLD = 70


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


echo = standardize_columns(echo)
ecg = standardize_columns(ecg)


def find_col(df, options):
    return next((c for c in options if c in df.columns), None)


def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    name = re.sub(r"^SALIVA\s+", "", name)
    name = re.sub(r"[^A-Z0-9\s]", " ", name)
    return " ".join(name.split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)
    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.WRatio(a1, b1),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            scores.append(
                fuzz.WRatio(ac[:k] + " " + ac[k:], b1)
            )

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            scores.append(
                fuzz.WRatio(a1, bc[:k] + " " + bc[k:])
            )

    return max(scores)


def clean_sex(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    if value in ["M", "MALE"]:
        return "M"

    if value in ["F", "FEMALE"]:
        return "F"

    return ""


def parse_dates(series):
    return pd.to_datetime(
        series,
        errors="coerce",
        dayfirst=True
    ).dt.normalize()


def clean_age(value):
    if pd.isna(value):
        return None

    m = re.search(r"\d+(?:\.\d+)?", str(value))

    return float(m.group()) if m else None


ecg_name_col = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name_col = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_date_col = find_col(ecg, ["DATE", "ECG DATE", "STUDY DATE"])
echo_date_col = find_col(
    echo,
    ["STUDY_DATE", "STUDY DATE", "DATE", "ECHO DATE"]
)

ecg_age_col = find_col(ecg, ["AGE"])
echo_age_col = find_col(echo, ["AGE"])

ecg_sex_col = find_col(ecg, ["SEX", "GENDER"])
echo_sex_col = find_col(echo, ["GENDER", "SEX"])

ecg_file_col = find_col(
    ecg,
    ["FILE NAME", "FILENAME", "PDF_PATH", "FILE PATH"]
)

ef_percent_col = find_col(
    echo,
    ["EF_TEICH_%", "EF TEICH %", "EF_CUBED_%", "EF CUBED %"]
)

ef_raw_col = find_col(
    echo,
    ["EF_TEICH", "EF TEICH", "EF_CUBED", "EF CUBED", "EF"]
)

if ecg_name_col is None or echo_name_col is None:
    raise ValueError(
        f"Name column missing.\nECG columns: {list(ecg.columns)}"
        f"\nEcho columns: {list(echo.columns)}"
    )

if echo_date_col is None or ecg_date_col is None:
    raise ValueError(
        f"Date column missing.\nECG columns: {list(ecg.columns)}"
        f"\nEcho columns: {list(echo.columns)}"
    )

ecg = ecg.reset_index(drop=True).copy()
echo = echo.reset_index(drop=True).copy()

ecg["_ROW"] = range(2, len(ecg) + 2)
echo["_ROW"] = range(2, len(echo) + 2)

ecg["_NAME"] = ecg[ecg_name_col].apply(clean_name)
echo["_NAME"] = echo[echo_name_col].apply(clean_name)

ecg["_COMPACT"] = ecg[ecg_name_col].apply(compact_name)
echo["_COMPACT"] = echo[echo_name_col].apply(compact_name)

ecg["_DATE"] = parse_dates(ecg[ecg_date_col])
echo["_DATE"] = parse_dates(echo[echo_date_col])

ecg["_SEX"] = (
    ecg[ecg_sex_col].apply(clean_sex)
    if ecg_sex_col else ""
)

echo["_SEX"] = (
    echo[echo_sex_col].apply(clean_sex)
    if echo_sex_col else ""
)

ecg["_AGE"] = (
    ecg[ecg_age_col].apply(clean_age)
    if ecg_age_col else None
)

echo["_AGE"] = (
    echo[echo_age_col].apply(clean_age)
    if echo_age_col else None
)


def get_ef(row):
    value = None

    if ef_percent_col:
        value = pd.to_numeric(
            pd.Series([row[ef_percent_col]]),
            errors="coerce"
        ).iloc[0]

        if pd.notna(value):
            return float(value)

    if ef_raw_col:
        value = pd.to_numeric(
            pd.Series([row[ef_raw_col]]),
            errors="coerce"
        ).iloc[0]

        if pd.notna(value):
            value = float(value)

            if 0 < value <= 1:
                value *= 100

            return value

    return None


echo["_EF"] = echo.apply(get_ef, axis=1)

ecg_names = ecg["_NAME"].tolist()
ecg_compact_names = ecg["_COMPACT"].tolist()

compact_index = {}

for i, n in enumerate(ecg_compact_names):
    if n:
        compact_index.setdefault(n, []).append(i)


def get_candidates(echo_name, echo_compact):
    found = set()

    if echo_compact in compact_index:
        found.update(compact_index[echo_compact])

    if not echo_name:
        return []

    results = process.extract(
        echo_name,
        ecg_names,
        scorer=fuzz.WRatio,
        score_cutoff=50,
        limit=100
    )

    found.update(item[2] for item in results)

    compact_results = process.extract(
        echo_compact,
        ecg_compact_names,
        scorer=fuzz.ratio,
        score_cutoff=50,
        limit=100
    )

    found.update(item[2] for item in compact_results)

    return list(found)


matches = []
unmatched_echo = []
matched_ecg_indices = set()

for j, h in echo.iterrows():

    echo_name = h["_NAME"]
    echo_compact = h["_COMPACT"]

    if not echo_name:
        unmatched_echo.append({
            "Echo Row": h["_ROW"],
            "Echo Name": h[echo_name_col],
            "Reason": "Empty Echo name"
        })
        continue

    candidates = get_candidates(echo_name, echo_compact)
    valid = []

    for i in candidates:
        e = ecg.iloc[i]

        score = name_score(
            h[echo_name_col],
            e[ecg_name_col]
        )

        if score < THRESHOLD:
            continue

        sex1 = h["_SEX"]
        sex2 = e["_SEX"]

        if sex1 and sex2 and sex1 != sex2:
            continue

        date1 = h["_DATE"]
        date2 = e["_DATE"]

        same_date = (
            pd.notna(date1)
            and pd.notna(date2)
            and date1 == date2
        )

        age1 = h["_AGE"]
        age2 = e["_AGE"]

        age_match = (
            age1 is not None
            and age2 is not None
            and abs(age1 - age2) <= 2
        )

        date_gap = (
            abs((date1 - date2).days)
            if pd.notna(date1) and pd.notna(date2)
            else 999999
        )

        valid.append({
            "i": i,
            "score": score,
            "same_date": same_date,
            "age_match": age_match,
            "date_gap": date_gap
        })

    if not valid:
        unmatched_echo.append({
            "Echo Row": h["_ROW"],
            "Echo Name": h[echo_name_col],
            "Echo Date": h["_DATE"],
            "Reason": f"No candidate with name similarity >= {THRESHOLD}%"
        })
        continue

    valid.sort(
        key=lambda x: (
            x["same_date"],
            x["score"],
            x["age_match"],
            -x["date_gap"]
        ),
        reverse=True
    )

    selected = valid[0]
    e = ecg.iloc[selected["i"]]

    reason = (
        "Name + same date"
        if selected["same_date"]
        else "Name match; dates differ"
    )

    matches.append({
        "ECG Row": e["_ROW"],
        "Echo Row": h["_ROW"],
        "ECG Name": e[ecg_name_col],
        "Echo Name": h[echo_name_col],
        "Name Match %": round(selected["score"], 2),
        "ECG Date": e["_DATE"],
        "Echo Date": h["_DATE"],
        "Date Match": selected["same_date"],
        "Date Difference (days)": selected["date_gap"],
        "Age ECG": e[ecg_age_col] if ecg_age_col else None,
        "Age Echo": h[echo_age_col] if echo_age_col else None,
        "Gender ECG": e[ecg_sex_col] if ecg_sex_col else None,
        "Gender Echo": h[echo_sex_col] if echo_sex_col else None,
        "EF Teich (%)": h["_EF"],
        "Match Reason": reason,
        "ECG File": e[ecg_file_col] if ecg_file_col else None
    })

    matched_ecg_indices.add(selected["i"])


match_df = pd.DataFrame(matches)

unmatched_ecg = ecg[
    ~ecg.index.isin(matched_ecg_indices)
].copy()

internal_ecg = [
    "_ROW", "_NAME", "_COMPACT", "_DATE",
    "_SEX", "_AGE"
]

unmatched_ecg.drop(
    columns=internal_ecg,
    errors="ignore",
    inplace=True
)

unmatched_echo_df = pd.DataFrame(unmatched_echo)

if not match_df.empty:
    match_df = match_df.sort_values(
        by=["Date Match", "Name Match %"],
        ascending=[False, False]
    )

output_file = (
    "2024_ECG_ECHO_MATCHING_70_PERCENT_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    match_df.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

    for sheet in writer.sheets.values():
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

        for cells in sheet.columns:
            width = max(
                len(str(cell.value))
                if cell.value is not None else 0
                for cell in cells[:300]
            )

            sheet.column_dimensions[
                cells[0].column_letter
            ].width = min(max(width + 2, 12), 40)


print("\n========== MATCHING SUMMARY ==========")
print("Name threshold:", THRESHOLD, "%")
print("ECG records:", len(ecg))
print("Echo records:", len(echo))
print("Matched Echo records:", len(match_df))
print("Unmatched Echo records:", len(unmatched_echo_df))
print("Unique ECG records matched:", len(matched_ecg_indices))
print("Output file:", os.path.abspath(output_file))

if not match_df.empty:
    print("\n========== SAMPLE MATCHES ==========")
    display(
        match_df[
            [
                "ECG Name",
                "Echo Name",
                "Name Match %",
                "ECG Date",
                "Echo Date",
                "Date Match",
                "Match Reason"
            ]
        ].head(20)
    )

print("\n========== CHECK YOUR EXAMPLE ==========")

test_score = name_score(
    "PULYALA ADHITHYA",
    "PULYALA ADITYA"
)

print("Name similarity:", round(test_score, 2), "%")
print("Expected to pass threshold:", test_score >= THRESHOLD)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\179241067.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(



========== MATCHING SUMMARY ==========
Name threshold: 70 %
ECG records: 4035
Echo records: 1739
Matched Echo records: 1608
Unmatched Echo records: 131
Unique ECG records matched: 1389
Output file: c:\Users\Admin\Desktop\ECG_ECHO\2024_ECG_ECHO_MATCHING_70_PERCENT_20261029_165333.xlsx

========== SAMPLE MATCHES ==========


,ECG Name,Echo Name,Name Match %,ECG Date,Echo Date,Date Match,Match Reason
2,METTU AADRIKA RAJ,METTU AADRIKA RAJ,100.0,2024-08-31,2024-08-31,True,Name + same date
7,MANDA RAJITHA,MANDA RAJITHA,100.0,2024-01-13,2024-01-13,True,Name + same date
9,POLASA HARISH,POLASA HARISH,100.0,2024-05-12,2024-05-12,True,Name + same date
13,BARIGELA RAMADEVI,BARIGELA RAMA DEVI,100.0,2024-07-11,2024-07-11,True,Name + same date
15,SHAHEEN,SHAHEEN SULTHANA,100.0,2024-07-08,2024-07-08,True,Name + same date
16,NYALA GATTAIAH,NYALA GATTAIAH,100.0,2024-02-26,2024-02-26,True,Name + same date
17,CHENNURI DURGABHAVANI,CHENNURI DURGABHAVANI,100.0,2024-02-27,2024-02-27,True,Name + same date
22,KETHIRI RAJAIAH,KETHIRI RAJAIAH,100.0,2024-05-22,2024-05-22,True,Name + same date
31,MARABOINA KOMALA,MARABOINA KOMALA,100.0,2024-12-12,2024-12-12,True,Name + same date
32,MARABOINA KOMALA,MARABOINA KOMALA,100.0,2024-12-12,2024-12-12,True,Name + same date



========== CHECK YOUR EXAMPLE ==========
Name similarity: 93.33 %
Expected to pass threshold: True


In [27]:
def classify_ef(ef):
    if pd.isna(ef):
        return "Unknown"
    if ef >= 50:
        return "Normal"
    elif ef >= 40:
        return "Mild"
    elif ef >= 30:
        return "Moderate"
    else:
        return "Severe"


if not match_df.empty:
    match_df["LV Classification"] = match_df["EF Teich (%)"].apply(classify_ef)

    counts = match_df["LV Classification"].value_counts()

    normal = counts.get("Normal", 0)
    mild = counts.get("Mild", 0)
    moderate = counts.get("Moderate", 0)
    severe = counts.get("Severe", 0)
    unknown = counts.get("Unknown", 0)

    print("\n========== LV CLASSIFICATION SUMMARY ==========")
    print("Normal:", normal)
    print("Mild:", mild)
    print("Moderate:", moderate)
    print("Severe:", severe)
    print("Unknown / Missing EF:", unknown)
    print("Total matched records:", len(match_df))

    summary_df = pd.DataFrame({
        "LV Classification": [
            "Normal", "Mild", "Moderate", "Severe", "Unknown / Missing EF"
        ],
        "Count": [normal, mild, moderate, severe, unknown]
    })

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace"
    ) as writer:
        summary_df.to_excel(
            writer,
            sheet_name="LV Summary",
            index=False
        )

    print("\nLV Summary has been added to the Excel output.")
else:
    print("No matched records available for LV classification.")


========== LV CLASSIFICATION SUMMARY ==========
Normal: 1410
Mild: 71
Moderate: 87
Severe: 39
Unknown / Missing EF: 1
Total matched records: 1608

LV Summary has been added to the Excel output.


In [28]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

THRESHOLD = 70

ecg_file = "2024_Ecg1.xlsx"
echo_file = "2024echo_reports.xlsx"

ecg_sheet = "ECG Data"
echo_sheet = "2024"

ecg = pd.read_excel(ecg_file, sheet_name=ecg_sheet)
echo = pd.read_excel(echo_file, sheet_name=echo_sheet)


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    for option in options:
        option = re.sub(r"\s+", " ", option).strip().upper()
        if option in df.columns:
            return option
    return None


def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    name = re.sub(r"^SALIVA\s+", "", name)
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_sex(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    if value in ["M", "MALE"]:
        return "M"

    if value in ["F", "FEMALE"]:
        return "F"

    return ""


def clean_age(value):
    if pd.isna(value):
        return None

    match = re.search(r"\d+(?:\.\d+)?", str(value))

    if not match:
        return None

    return float(match.group())


def parse_date(series):
    return pd.to_datetime(series, errors="coerce", dayfirst=True).dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    match = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not match:
        return pd.NaT

    return pd.to_datetime(
        match.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.WRatio(a1, b1),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.token_sort_ratio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.token_sort_ratio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(
    ecg,
    ["FILE NAME", "FILENAME", "FILE NAME ", "FILE"]
)

ecg_date = find_col(
    ecg,
    ["DATE", "ECG DATE", "STUDY DATE", "RECORDING DATE"]
)

echo_date = find_col(
    echo,
    ["STUDY DATE", "STUDY_DATE", "DATE", "ECHO DATE"]
)

ef_col = find_col(
    echo,
    [
        "EF_TEICH_%",
        "EF TEICH %",
        "EF_TEICH (%)",
        "EF (TEICH) %",
        "EF_TEICH",
        "EF (TEICH)"
    ]
)

required = {
    "ECG patient name": ecg_name,
    "Echo patient name": echo_name,
    "ECG filename": file_col,
    "Echo EF column": ef_col
}

missing = [key for key, value in required.items() if value is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

ecg["name_key"] = ecg[ecg_name].apply(clean_name)
echo["name_key"] = echo[echo_name].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name].apply(compact_name)
echo["compact_key"] = echo[echo_name].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date])
    if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date])
    if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(h):
    name = h["name_key"]
    compact = h["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    queries = [name]

    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            ecg_names,
            scorer=fuzz.WRatio,
            limit=None,
            score_cutoff=THRESHOLD
        )

        candidates.update(index for _, _, index in found)

    valid = []

    for i in candidates:
        e = ecg.iloc[i]

        score = name_score(h["original_name"], e["original_name"])

        if score < THRESHOLD:
            continue

        if (
            h["sex_key"]
            and e["sex_key"]
            and h["sex_key"] != e["sex_key"]
        ):
            continue

        hd = h["match_date"]
        ed = e["match_date"]

        if pd.notna(hd) and pd.notna(ed):
            date_gap = abs((hd - ed).days)
            same_date = int(hd == ed)
        else:
            date_gap = 10**9
            same_date = 0

        age_match = int(
            h["age_key"] is not None
            and e["age_key"] is not None
            and h["age_key"] == e["age_key"]
        )

        valid.append(
            (same_date, score, age_match, -date_gap, i, date_gap)
        )

    valid.sort(
        key=lambda x: (x[0], x[1], x[2], x[3]),
        reverse=True
    )

    return valid


def classify_ef(ef):
    if pd.isna(ef):
        return "Unknown"

    if ef >= 50:
        return "Normal"
    elif ef >= 40:
        return "Mild"
    elif ef >= 30:
        return "Moderate"
    else:
        return "Severe"


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()

for j, h in echo.iterrows():
    candidates = find_candidates(h)

    if not candidates:
        continue

    selected = candidates[0]
    i = selected[4]
    date_gap = selected[5]

    e = ecg.iloc[i]
    score = selected[1]
    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"
    else:
        lv = classify_ef(ef)

    if selected[0] == 1:
        reason = "Name matched; same study date"
    elif date_gap < 10**9:
        reason = f"Name matched; date difference {date_gap} days"
    else:
        reason = "Name matched; dates unavailable"

    matches.append({
        "ECG Row": i + 2,
        "Echo Row": j + 2,
        "ECG Name": e["original_name"],
        "Echo Name": h["original_name"],
        "Name Match %": round(score, 2),
        "ECG Date": e["match_date"],
        "Echo Date": h["match_date"],
        "Date Difference (days)": (
            date_gap if date_gap < 10**9 else None
        ),
        "Age ECG": e[ecg_age] if ecg_age else None,
        "Age Echo": h[echo_age] if echo_age else None,
        "Gender ECG": e[ecg_sex] if ecg_sex else "",
        "Gender Echo": h[echo_sex] if echo_sex else "",
        "EF Teich (%)": ef,
        "LV Classification": lv,
        "Match Reason": reason,
        "ECG File": e[file_col]
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


match_df = pd.DataFrame(matches)

unmatched_echo_rows = []

for j, h in echo.iterrows():
    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo Row": j + 2,
            "Echo Name": h["original_name"],
            "Age": h[echo_age] if echo_age else None,
            "Gender": h[echo_sex] if echo_sex else "",
            "Echo Date": h["match_date"],
            "EF Teich (%)": h[ef_col],
            "Reason": "No ECG name match at or above threshold"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)

unmatched_ecg_rows = []

for i, e in ecg.iterrows():
    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG Row": i + 2,
            "ECG Name": e["original_name"],
            "Age": e[ecg_age] if ecg_age else None,
            "Gender": e[ecg_sex] if ecg_sex else "",
            "ECG Date": e["match_date"],
            "ECG File": e[file_col],
            "Reason": "Not matched to any Echo record"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)


categories = [
    "Normal",
    "Mild",
    "Moderate",
    "Severe",
    "Unknown"
]

if not match_df.empty:
    counts = match_df["LV Classification"].value_counts()
else:
    counts = pd.Series(dtype=int)

summary_df = pd.DataFrame({
    "LV Classification": categories,
    "Count": [int(counts.get(category, 0)) for category in categories]
})

summary_df.loc[len(summary_df)] = [
    "Total matched Echo records",
    len(match_df)
]

output_file = (
    "2024_ECG_ECHO_MATCHING_70_PERCENT_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    match_df.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="LV Summary",
        index=False
    )


print("\n========== MATCHING SUMMARY ==========")
print("Name threshold:", THRESHOLD, "%")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo records:", len(match_df))
print("Unmatched Echo records:", len(unmatched_echo_df))
print("Unmatched ECG records:", len(unmatched_ecg_df))
print("Unique ECG records matched:", len(matched_ecg_indices))

print("\n========== LV CLASSIFICATION SUMMARY ==========")

for category in categories:
    print(f"{category}: {int(counts.get(category, 0))}")

print("Total matched Echo records:", len(match_df))
print("\nExcel output saved to:", output_file)
print("Sheets: Matched Records, Unmatched Echo, Unmatched ECG, LV Summary")

display(match_df)
display(unmatched_echo_df)
display(unmatched_ecg_df)
display(summary_df)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14588\2213413429.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series, errors="coerce", dayfirst=True).dt.normalize()



========== MATCHING SUMMARY ==========
Name threshold: 70 %
Echo records: 1639
ECG records: 4035
Matched Echo records: 1596
Unmatched Echo records: 43
Unmatched ECG records: 2658
Unique ECG records matched: 1377

========== LV CLASSIFICATION SUMMARY ==========
Normal: 1398
Mild: 71
Moderate: 87
Severe: 39
Unknown: 1
Total matched Echo records: 1596

Excel output saved to: 2024_ECG_ECHO_MATCHING_70_PERCENT_20261029_165914.xlsx
Sheets: Matched Records, Unmatched Echo, Unmatched ECG, LV Summary


,ECG Row,Echo Row,ECG Name,Echo Name,Name Match %,ECG Date,Echo Date,Date Difference (days),Age ECG,Age Echo,Gender ECG,Gender Echo,EF Teich (%),LV Classification,Match Reason,ECG File
0,3429,2,GUDURU AVINASH,GUDURU VINAYCHANDRA,72.73,2024-10-04,2024-10-07,3,15 Years,None,Male,Male,82.2,Normal,Name matched; date difference 3 days,EK__[20241004135414]_GUDURU AVINASH____[]_.pdf
1,3415,3,PODISHETTY GAYATHRI,GAYATHRI,100.00,2024-10-03,2024-10-01,2,57 Years,None,Female,Female,69.5,Normal,Name matched; date difference 2 days,EK__[20241003110620]_PODISHETTY GAYATHRI____[]...
2,2957,4,METTU AADRIKA RAJ,METTU AADRIKA RAJ,100.00,2024-08-31,2024-08-31,0,3 Years,None,Female,Female,65.7,Normal,Name matched; same study date,EK__[20240831190637]_METTU AADRIKA RAJ____[]_(...
3,3066,7,HAJRA AYMAN,HAJIRA AYMAN,95.65,2024-09-09,2024-09-09,0,9 Years,None,Female,Female,74.0,Normal,Name matched; same study date,EK__[20240909111445]_HAJRA AYMAN____[]_.pdf
4,1162,8,NAREDLA MANASWI,MANASWINI,78.75,2024-03-30,2024-06-12,74,16 Years,None,Female,Female,66.8,Normal,Name matched; date difference 74 days,EK__[20240330171629]_NAREDLA MANASWI____[]_.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1591,276,1636,PADAVALAN KRISHNACHANDRA,VINAYCHANDRA,77.14,2024-01-20,2024-04-12,83,15 Years,None,Male,Male,56.1,Normal,Name matched; date difference 83 days,EK__[20240120113430]_PADAVALAN KRISHNACHANDRA_...
1592,1374,1637,NEELAM SPANDANA,NERELLA SIRICHANDANA,74.29,2024-04-19,2024-01-29,81,17 Years,None,Female,Female,59.3,Normal,Name matched; date difference 81 days,EK__[20240419115200]_NEELAM SPANDANA____[]_.pdf
1593,2223,1638,CHINTHALA BHARATHI,THALAGAMPA BHARATH,72.22,2024-07-04,2024-11-21,140,25 Years,None,Male,Male,76.0,Normal,Name matched; date difference 140 days,EK__[20240704120213]_CHINTHALA BHARATHI____[]_...
1594,1338,1639,ARUKONDA SARITHA,ARUKONDA SARITHA,100.00,2024-04-16,2024-01-29,78,30 Years,None,Female,Female,62.3,Normal,Name matched; date difference 78 days,EK__[20240416193359]_ARUKONDA SARITHA____[]_.pdf


,Echo Row,Echo Name,Age,Gender,Echo Date,EF Teich (%),Reason
0,5,MANDALA RISHIK,None,Male,2024-10-04,75.1,No ECG name match at or above threshold
1,6,SHOURYATEJ,None,Male,2024-03-01,59.9,No ECG name match at or above threshold
2,15,VIPLAVA KATHI,None,Female,2024-12-09,59.2,No ECG name match at or above threshold
3,16,THANUJA JOGULA,None,Female,2024-01-15,67.6,No ECG name match at or above threshold
4,20,PRAJWAL YADAV,None,Male,2024-05-15,73.9,No ECG name match at or above threshold
5,21,MANTHENA VINEEKSHA,None,Female,2024-12-07,73.5,No ECG name match at or above threshold
6,49,AZMERA RAVALIKA,None,Female,2024-02-27,64.9,No ECG name match at or above threshold
7,67,MARIYAMMA EMMADI,None,Male,2024-06-05,62.7,No ECG name match at or above threshold
8,98,BASAVARAJU SANDHYA,None,Female,2024-02-14,54.0,No ECG name match at or above threshold
9,130,KONKATI SAIVARDHAN,None,Male,2024-02-16,71.5,No ECG name match at or above threshold


,ECG Row,ECG Name,Age,Gender,ECG Date,ECG File,Reason
0,2,GANGU VANISREE,56 Years,Female,2024-01-01,EK__[20240101202919]_GANGU VANISREE____[]_.pdf,Not matched to any Echo record
1,6,RONDLA SRIKARREDDY,53 Years,Male,2024-01-01,EK__[20240101112512]_RONDLA SRIKARREDDY____[]...,Not matched to any Echo record
2,11,GUNDA SRINIVAS,53 Years,Male,2024-01-02,EK__[20240102141846]_GUNDA SRINIVAS____[]_.pdf,Not matched to any Echo record
3,12,GUNDA THIRUPATHAIAH,65 Years,Male,2024-01-02,EK__[20240102145654]_GUNDA THIRUPATHAIAH____[]...,Not matched to any Echo record
4,14,SK SADIQ,62 Years,Male,2024-01-02,EK__[20240102131214]_SK SADIQ____[]_.pdf,Not matched to any Echo record
...,...,...,...,...,...,...,...
2653,4032,THALLAPELLY DEVENAMMA,65 Years,Female,2024-12-31,EK__[20241231180131]_THALLAPELLY DEVENAMMA____...,Not matched to any Echo record
2654,4033,SIRIGIRI PRINCY,34 Years,Female,2024-12-31,EK__[20241231120848]_SIRIGIRI PRINCY____[]_.pdf,Not matched to any Echo record
2655,4034,BAIRI RAJAMALLU,85 Years,Male,2024-12-31,EK__[20241231120400]_BAIRI RAJAMALLU____[]_.pdf,Not matched to any Echo record
2656,4035,CHITYALA SRI HARI,66 Years,Male,2024-12-31,EK__[20241231112422]_CHITYALA SRI HARI____[]_.pdf,Not matched to any Echo record


,LV Classification,Count
0,Normal,1398
1,Mild,71
2,Moderate,87
3,Severe,39
4,Unknown,1
5,Total matched Echo records,1596


In [29]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz, process
from IPython.display import display

THRESHOLD = 70

ecg_file = "ecg_2022.xlsx"
echo_file = "2022echo_reports.xlsx"

ecg_sheet = "ECG Data"
echo_sheet = "Sheet1"

ecg = pd.read_excel(ecg_file, sheet_name=ecg_sheet)
echo = pd.read_excel(echo_file, sheet_name=echo_sheet)


def standardize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip().upper()
        for c in df.columns
    ]
    return df


ecg = standardize_columns(ecg)
echo = standardize_columns(echo)


def find_col(df, options):
    for option in options:
        option = re.sub(r"\s+", " ", option).strip().upper()
        if option in df.columns:
            return option
    return None


def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).upper().strip()
    name = re.sub(r"^SALIVA\s+", "", name)
    return " ".join(re.sub(r"[^A-Z0-9\s]", " ", name).split())


def compact_name(name):
    return re.sub(r"[^A-Z0-9]", "", clean_name(name))


def clean_sex(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    if value in ["M", "MALE"]:
        return "M"

    if value in ["F", "FEMALE"]:
        return "F"

    return ""


def clean_age(value):
    if pd.isna(value):
        return None

    match = re.search(r"\d+(?:\.\d+)?", str(value))

    if not match:
        return None

    return float(match.group())


def parse_date(series):
    return pd.to_datetime(series, errors="coerce", dayfirst=True).dt.normalize()


def filename_date(filename):
    if pd.isna(filename):
        return pd.NaT

    match = re.search(r"\[(\d{8})\d{6}\]", str(filename))

    if not match:
        return pd.NaT

    return pd.to_datetime(
        match.group(1),
        format="%Y%m%d",
        errors="coerce"
    )


def name_score(a, b):
    a1 = clean_name(a)
    b1 = clean_name(b)

    ac = compact_name(a1)
    bc = compact_name(b1)

    if not ac or not bc:
        return 0

    if a1 == b1 or ac == bc:
        return 100

    scores = [
        fuzz.ratio(a1, b1),
        fuzz.ratio(ac, bc),
        fuzz.WRatio(a1, b1),
        fuzz.token_sort_ratio(a1, b1),
        fuzz.token_set_ratio(a1, b1)
    ]

    if " " not in a1 and len(ac) >= 6:
        for k in range(3, len(ac) - 2):
            variant = ac[:k] + " " + ac[k:]
            scores.append(fuzz.token_sort_ratio(variant, b1))

    if " " not in b1 and len(bc) >= 6:
        for k in range(3, len(bc) - 2):
            variant = bc[:k] + " " + bc[k:]
            scores.append(fuzz.token_sort_ratio(a1, variant))

    return max(scores)


ecg_name = find_col(ecg, ["NAME", "PATIENT NAME"])
echo_name = find_col(echo, ["NAME", "PATIENT NAME"])

ecg_age = find_col(ecg, ["AGE", "PATIENT AGE"])
echo_age = find_col(echo, ["AGE", "PATIENT AGE"])

ecg_sex = find_col(ecg, ["SEX", "GENDER"])
echo_sex = find_col(echo, ["GENDER", "SEX"])

file_col = find_col(
    ecg,
    ["FILE NAME", "FILENAME", "FILE NAME ", "FILE"]
)

ecg_date = find_col(
    ecg,
    ["DATE", "ECG DATE", "STUDY DATE", "RECORDING DATE"]
)

echo_date = find_col(
    echo,
    ["STUDY DATE", "STUDY_DATE", "DATE", "ECHO DATE"]
)

ef_col = find_col(
    echo,
    [
        "EF_TEICH_%",
        "EF TEICH %",
        "EF_TEICH (%)",
        "EF (TEICH) %",
        "EF_TEICH",
        "EF (TEICH)"
    ]
)

required = {
    "ECG patient name": ecg_name,
    "Echo patient name": echo_name,
    "ECG filename": file_col,
    "Echo EF column": ef_col
}

missing = [key for key, value in required.items() if value is None]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"ECG columns: {ecg.columns.tolist()}\n"
        f"Echo columns: {echo.columns.tolist()}"
    )


ecg = ecg.copy().reset_index(drop=True)
echo = echo.copy().reset_index(drop=True)

ecg["original_name"] = ecg[ecg_name]
echo["original_name"] = echo[echo_name]

ecg["name_key"] = ecg[ecg_name].apply(clean_name)
echo["name_key"] = echo[echo_name].apply(clean_name)

ecg["compact_key"] = ecg[ecg_name].apply(compact_name)
echo["compact_key"] = echo[echo_name].apply(compact_name)

ecg["age_key"] = (
    ecg[ecg_age].apply(clean_age)
    if ecg_age
    else pd.Series(None, index=ecg.index)
)

echo["age_key"] = (
    echo[echo_age].apply(clean_age)
    if echo_age
    else pd.Series(None, index=echo.index)
)

ecg["sex_key"] = (
    ecg[ecg_sex].apply(clean_sex)
    if ecg_sex
    else pd.Series("", index=ecg.index)
)

echo["sex_key"] = (
    echo[echo_sex].apply(clean_sex)
    if echo_sex
    else pd.Series("", index=echo.index)
)

ecg["match_date"] = (
    parse_date(ecg[ecg_date])
    if ecg_date
    else pd.Series(pd.NaT, index=ecg.index)
)

echo["match_date"] = (
    parse_date(echo[echo_date])
    if echo_date
    else pd.Series(pd.NaT, index=echo.index)
)

ecg["match_date"] = ecg["match_date"].fillna(
    ecg[file_col].apply(filename_date)
)

echo[ef_col] = pd.to_numeric(echo[ef_col], errors="coerce")

echo[ef_col] = echo[ef_col].apply(
    lambda x: x * 100 if pd.notna(x) and 0 < x <= 1 else x
)

ecg = ecg[ecg["name_key"] != ""].reset_index(drop=True)
echo = echo[echo["name_key"] != ""].reset_index(drop=True)

ecg_names = ecg["name_key"].tolist()

exact_index = {}
compact_index = {}

for i, row in ecg.iterrows():
    exact_index.setdefault(row["name_key"], []).append(i)
    compact_index.setdefault(row["compact_key"], []).append(i)


def find_candidates(h):
    name = h["name_key"]
    compact = h["compact_key"]

    candidates = set()

    candidates.update(exact_index.get(name, []))
    candidates.update(compact_index.get(compact, []))

    queries = [name]

    if " " not in name and len(compact) >= 6:
        queries.extend(
            compact[:k] + " " + compact[k:]
            for k in range(3, len(compact) - 2)
        )

    for query in queries:
        found = process.extract(
            query,
            ecg_names,
            scorer=fuzz.WRatio,
            limit=None,
            score_cutoff=THRESHOLD
        )

        candidates.update(index for _, _, index in found)

    valid = []

    for i in candidates:
        e = ecg.iloc[i]

        score = name_score(h["original_name"], e["original_name"])

        if score < THRESHOLD:
            continue

        if (
            h["sex_key"]
            and e["sex_key"]
            and h["sex_key"] != e["sex_key"]
        ):
            continue

        hd = h["match_date"]
        ed = e["match_date"]

        if pd.notna(hd) and pd.notna(ed):
            date_gap = abs((hd - ed).days)
            same_date = int(hd == ed)
        else:
            date_gap = 10**9
            same_date = 0

        age_match = int(
            h["age_key"] is not None
            and e["age_key"] is not None
            and h["age_key"] == e["age_key"]
        )

        valid.append(
            (same_date, score, age_match, -date_gap, i, date_gap)
        )

    valid.sort(
        key=lambda x: (x[0], x[1], x[2], x[3]),
        reverse=True
    )

    return valid


def classify_ef(ef):
    if pd.isna(ef):
        return "Unknown"

    if ef >= 50:
        return "Normal"
    elif ef >= 40:
        return "Mild"
    elif ef >= 30:
        return "Moderate"
    else:
        return "Severe"


matches = []
matched_echo_indices = set()
matched_ecg_indices = set()

for j, h in echo.iterrows():
    candidates = find_candidates(h)

    if not candidates:
        continue

    selected = candidates[0]
    i = selected[4]
    date_gap = selected[5]

    e = ecg.iloc[i]
    score = selected[1]
    ef = h[ef_col]

    if pd.isna(ef):
        lv = "Unknown"
    else:
        lv = classify_ef(ef)

    if selected[0] == 1:
        reason = "Name matched; same study date"
    elif date_gap < 10**9:
        reason = f"Name matched; date difference {date_gap} days"
    else:
        reason = "Name matched; dates unavailable"

    matches.append({
        "ECG Row": i + 2,
        "Echo Row": j + 2,
        "ECG Name": e["original_name"],
        "Echo Name": h["original_name"],
        "Name Match %": round(score, 2),
        "ECG Date": e["match_date"],
        "Echo Date": h["match_date"],
        "Date Difference (days)": (
            date_gap if date_gap < 10**9 else None
        ),
        "Age ECG": e[ecg_age] if ecg_age else None,
        "Age Echo": h[echo_age] if echo_age else None,
        "Gender ECG": e[ecg_sex] if ecg_sex else "",
        "Gender Echo": h[echo_sex] if echo_sex else "",
        "EF Teich (%)": ef,
        "LV Classification": lv,
        "Match Reason": reason,
        "ECG File": e[file_col]
    })

    matched_echo_indices.add(j)
    matched_ecg_indices.add(i)


match_df = pd.DataFrame(matches)

unmatched_echo_rows = []

for j, h in echo.iterrows():
    if j not in matched_echo_indices:
        unmatched_echo_rows.append({
            "Echo Row": j + 2,
            "Echo Name": h["original_name"],
            "Age": h[echo_age] if echo_age else None,
            "Gender": h[echo_sex] if echo_sex else "",
            "Echo Date": h["match_date"],
            "EF Teich (%)": h[ef_col],
            "Reason": "No ECG name match at or above threshold"
        })

unmatched_echo_df = pd.DataFrame(unmatched_echo_rows)

unmatched_ecg_rows = []

for i, e in ecg.iterrows():
    if i not in matched_ecg_indices:
        unmatched_ecg_rows.append({
            "ECG Row": i + 2,
            "ECG Name": e["original_name"],
            "Age": e[ecg_age] if ecg_age else None,
            "Gender": e[ecg_sex] if ecg_sex else "",
            "ECG Date": e["match_date"],
            "ECG File": e[file_col],
            "Reason": "Not matched to any Echo record"
        })

unmatched_ecg_df = pd.DataFrame(unmatched_ecg_rows)


categories = [
    "Normal",
    "Mild",
    "Moderate",
    "Severe",
    "Unknown"
]

if not match_df.empty:
    counts = match_df["LV Classification"].value_counts()
else:
    counts = pd.Series(dtype=int)

summary_df = pd.DataFrame({
    "LV Classification": categories,
    "Count": [int(counts.get(category, 0)) for category in categories]
})

summary_df.loc[len(summary_df)] = [
    "Total matched Echo records",
    len(match_df)
]

output_file = (
    "2024_ECG_ECHO_MATCHING_70_PERCENT_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + ".xlsx"
)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    match_df.to_excel(
        writer,
        sheet_name="Matched Records",
        index=False
    )

    unmatched_echo_df.to_excel(
        writer,
        sheet_name="Unmatched Echo",
        index=False
    )

    unmatched_ecg_df.to_excel(
        writer,
        sheet_name="Unmatched ECG",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="LV Summary",
        index=False
    )


print("\n========== MATCHING SUMMARY ==========")
print("Name threshold:", THRESHOLD, "%")
print("Echo records:", len(echo))
print("ECG records:", len(ecg))
print("Matched Echo records:", len(match_df))
print("Unmatched Echo records:", len(unmatched_echo_df))
print("Unmatched ECG records:", len(unmatched_ecg_df))
print("Unique ECG records matched:", len(matched_ecg_indices))

print("\n========== LV CLASSIFICATION SUMMARY ==========")

for category in categories:
    print(f"{category}: {int(counts.get(category, 0))}")

print("Total matched Echo records:", len(match_df))
print("\nExcel output saved to:", output_file)
print("Sheets: Matched Records, Unmatched Echo, Unmatched ECG, LV Summary")

display(match_df)
display(unmatched_echo_df)
display(unmatched_ecg_df)
display(summary_df)


========== MATCHING SUMMARY ==========
Name threshold: 70 %
Echo records: 965
ECG records: 4828
Matched Echo records: 937
Unmatched Echo records: 28
Unmatched ECG records: 3960
Unique ECG records matched: 868

========== LV CLASSIFICATION SUMMARY ==========
Normal: 827
Mild: 43
Moderate: 45
Severe: 21
Unknown: 1
Total matched Echo records: 937

Excel output saved to: 2024_ECG_ECHO_MATCHING_70_PERCENT_20261029_170108.xlsx
Sheets: Matched Records, Unmatched Echo, Unmatched ECG, LV Summary


,ECG Row,Echo Row,ECG Name,Echo Name,Name Match %,ECG Date,Echo Date,Date Difference (days),Age ECG,Age Echo,Gender ECG,Gender Echo,EF Teich (%),LV Classification,Match Reason,ECG File
0,3358,5,GOLLAPALLY HARSHAVARDHAN,HARSHAVARDHAN GOLLAPELLI,91.67,2022-06-23,2022-06-23,0,6 Years,None,Male,Male,63.0,Normal,Name matched; same study date,EK__[20220623110321]_GOLLAPALLY HARSHAVARDHAN_...
1,1429,6,GOPARAJU BHAGYASREE,BHAVYASREE,81.00,2022-12-15,2022-07-04,164,34 Years,None,Female,Female,63.5,Normal,Name matched; date difference 164 days,EK__[20221215141118]_GOPARAJU BHAGYASREE____[]...
2,3462,8,KANDULA SATHWIK,KANDULA SATHWIK,100.00,2022-06-29,2022-06-29,0,7 Years,None,Male,Male,67.2,Normal,Name matched; same study date,EK__[20220629101417]_KANDULA SATHWIK____[]_.pdf
3,3146,9,SK ANKUSH,SAI THANUSH,70.00,2022-06-06,2022-06-30,24,29 Years,None,Male,Male,74.6,Normal,Name matched; date difference 24 days,EK__[20220606154114]_SK ANKUSH____[]_.pdf
4,3507,10,koduru venkateshwar reddy,ESHWAR,90.00,2022-07-01,2022-07-01,0,42 Years,None,Male,Male,65.6,Normal,Name matched; same study date,EK__[20220701191902]_koduru venkateshwar reddy...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
932,1388,962,CHINNALA PICHAMMA,CHINNALA PITCHAMMA,97.14,2022-12-12,2022-12-13,1,65 Years,None,Female,Female,72.2,Normal,Name matched; date difference 1 days,EK__[20221212145745]_CHINNALA PICHAMMA____[]_.pdf
933,2305,963,NADENDLLA KOTESHWARA RAO,NADENDLA KOTESHWAR RAO,95.65,2022-04-03,2022-04-03,0,81 Years,None,Male,Male,63.9,Normal,Name matched; same study date,EK__[20220403095415]_NADENDLLA KOTESHWARA RAO_...
934,1745,964,NAMENDLA KOMRAIAH,NAMINDLA KOMURAIAH,91.43,2022-02-21,2022-05-04,72,64 Years,None,Male,Male,75.8,Normal,Name matched; date difference 72 days,EK__[20220221172753]_NAMENDLA KOMRAIAH____[]_.pdf
935,1064,965,NAMANI RAVINDER,NAMANI RAVINDER,100.00,2022-11-17,2022-11-17,0,55 Years,None,Male,Male,55.6,Normal,Name matched; same study date,EK__[20221117110756]_NAMANI RAVINDER____[]_.pdf


,Echo Row,Echo Name,Age,Gender,Echo Date,EF Teich (%),Reason
0,2,APOORVA,None,Female,2022-12-03,75.4,No ECG name match at or above threshold
1,3,RITHVIKA KONDAPAKA,None,Female,2022-08-20,57.5,No ECG name match at or above threshold
2,4,SRIKRITHI,None,Female,2022-06-11,78.3,No ECG name match at or above threshold
3,7,ANIRUDDH,None,Male,2022-04-18,66.5,No ECG name match at or above threshold
4,13,SATHWIKA,None,Female,2022-08-25,64.5,No ECG name match at or above threshold
5,41,KONGA SHRAVYA,None,Female,2022-08-19,67.0,No ECG name match at or above threshold
6,47,SIDDHARTH MOURYA GANDHAM,None,Male,2022-11-25,69.9,No ECG name match at or above threshold
7,73,MARKA NARMADA,None,Female,2022-06-26,77.5,No ECG name match at or above threshold
8,107,GUBALA CHAITANYA,None,Female,2022-11-07,75.3,No ECG name match at or above threshold
9,115,VANGARI PADMINI,None,Female,2022-07-22,71.5,No ECG name match at or above threshold


,ECG Row,ECG Name,Age,Gender,ECG Date,ECG File,Reason
0,2,NEELAM KALAVATHI,50 Years,Female,2022-01-01,EK__[20220101101245]_NEELAM KALAVATHI____[]_.pdf,Not matched to any Echo record
1,3,CHINTHA SAMMAKKA,60 Years,Male,2022-01-01,EK__[20220101101550]_CHINTHA SAMMAKKA____[]_.pdf,Not matched to any Echo record
2,4,DHIDIGAM SRISHYLAM,81 Years,Male,2022-01-01,EK__[20220101103034]_DHIDIGAM SRISHYLAM____[]_...,Not matched to any Echo record
3,5,CHITTEMPALLY VENKATANARSAIAH,78 Years,Male,2022-01-01,EK__[20220101105552]_CHITTEMPALLY VENKATANARSA...,Not matched to any Echo record
4,6,CHANDANALA LINGAMURTHY,72 Years,Male,2022-01-01,EK__[20220101111505]_CHANDANALA LINGAMURTHY___...,Not matched to any Echo record
...,...,...,...,...,...,...,...
3955,4825,YELUGAM RAVALI,31 Years,Female,2022-09-30,EK__[20220930183055]_YELUGAM RAVALI____[]_.pdf,Not matched to any Echo record
3956,4826,KALIYARAPU LAKSHMI,66 Years,Female,2022-09-30,EK__[20220930185127]_KALIYARAPU LAKSHMI____[]_...,Not matched to any Echo record
3957,4827,KALIYARAPU LAKSHMI,66 Years,Female,2022-09-30,EK__[20220930185127]_KALIYARAPU LAKSHMI____[]_...,Not matched to any Echo record
3958,4828,PANJALA UMARANI,49 Years,Female,2022-09-30,EK__[20220930191341]_PANJALA UMARANI____[]_ - ...,Not matched to any Echo record


,LV Classification,Count
0,Normal,827
1,Mild,43
2,Moderate,45
3,Severe,21
4,Unknown,1
5,Total matched Echo records,937
